In [ ]:
# =========================================================
# real_data_physics_informed_additive_poly_sysid_pipeline.ipynb
# =========================================================
# PURPOSE
#   Real-data notebook for system identification using a
#   physics-consistent additive polynomial voltage surrogate:
#
#       Z_hat(x) = c0
#                  + sum_{k=1..d} a_k * dxp^k
#                  + sum_{k=1..d} b_k * dxn^k
#                  + k_ln * ln(ceR/ceL)        [optional]
#
#   where
#
#       dxp = (xp - xp_ref) / xp_scale
#       dxn = (xn - xn_ref) / xn_scale
#
#   and terminal voltage model:
#
#       V_hat = Z_hat(x) - Ns * I * R0 - v_rc   [if USE_RC=True]
#       V_hat = Z_hat(x) - Ns * I * R0          [if USE_RC=False]
#
# CORE IDEA
#   We want the voltage surrogate to stay close to battery physics.
#   Therefore, we do NOT use explicit time-dependent transient terms
#   such as c_j * exp(-t/tau_j) in Z_hat.
#
#   Instead, Z_hat is built only from state-dependent quantities:
#     - positive-electrode proxy stoichiometry xp
#     - negative-electrode proxy stoichiometry xn
#     - optional electrolyte log feature ln(ceR/ceL)
#
#   This keeps the surrogate more interpretable and more consistent
#   with the physical structure of the battery model.
#
# RELATION TO STATIC LS / STATIC LS+R
#   The static LS and static LS+R checks use the SAME polynomial feature
#   structure as the surrogate:
#
#       Phi(x) = [1, dxp, dxp^2, ..., dxp^d, dxn, dxn^2, ..., dxn^d]
#                [+ ln(ceR/ceL) if enabled]
#
#   Static LS solves:
#
#       Y ≈ Phi * theta_Z
#
#   Static LS+R solves:
#
#       Y ≈ Phi * theta_Z - I * R0
#
#   These static fits are used as transparent reference models to check
#   how expressive the chosen surrogate basis is.
#
#   Important:
#   - Static LS / LS+R and CT Stage 2 share the same functional form.
#   - But CT Stage 2 still learns parameters through the dynamic model,
#     so its fitted coefficients do not have to match the static LS
#     solution exactly.
#
# WORKFLOW
#   Cell 0  : HPC-safe imports and environment setup
#   Cell 1  : User settings
#   Cell 2  : Load raw .mpr file and inspect
#   Cell 3  : Plot/score helpers + manual least-squares helpers
#   Cell 4  : Find discharge cycles and select one
#   Cell 5  : Prepare (t,U,Y): units, discharge-only, resample, trim
#   Cell 6  : Nominal 14-state proxy model for xp/xn/ceL/ceR
#   Cell 7  : JAX helpers and parameterized A(thetaA), B(thetaB)
#   Cell 8  : Physics-consistent additive polynomial surrogate builder
#   Cell 9  : Stage 2 fit: surrogate + R0, with A/B fixed nominal
#   Cell 10 : Degree sweep summary or optional sweep code
#   Cell 11 : Stage 3a fit A/B only, with Stage-2 surrogate frozen
#   Cell 12 : Stage 3b unfreeze everything and refine full model
#   Cell 13 : Reusable cycle pipeline helpers
#   Cell 14 : Cross-cycle transfer/generalization of selected-cycle model
#   Cell 15 : Fit several/all cycles independently and summarize stability
#   Cell 16 : Visualize learned nonlinearity
#   Cell 17 : Extract monitorable parameters and tell degradation story
#   Cell 18 : Degree comparison helper (17 vs 15 vs 14)
#
# NOTES
#   - This notebook is for REAL DATA, so there is no generated-truth system.
#   - "A/B fixed" means fixed to the nominal proxy dynamics from Config.
#   - The explicit time-transient basis has been removed from Z_hat to keep
#     the surrogate closer to battery physics.
#   - If early-time transient mismatch remains, a more physical next step is
#     to add dynamic states (for example RC states), rather than adding
#     explicit time exponentials to Z_hat.
#   - Static LS and LS+R are included both as performance references and
#     as interpretable ways to understand the surrogate basis.
#   - We keep the random-cycle workflow, but we now also test the model on
#     several or all cycles to judge stability and generalization.
# =========================================================


# %% ======================================================
# CELL 0 — Imports + notebook setup (HPC-safe, threaded CPU)
# =========================================================
from __future__ import annotations

import os
import math
import warnings
from dataclasses import dataclass
from typing import Tuple, List, Dict, Any, Optional

# ---- Threading: pick up CPUs from Slurm if present, else default to 8 ----
N = int(os.environ.get("SLURM_CPUS_PER_TASK", "8"))

# IMPORTANT: set BEFORE importing jax / numpy-heavy libs
os.environ["XLA_FLAGS"] = (
    f"--xla_cpu_multi_thread_eigen=true intra_op_parallelism_threads={N}"
)
os.environ["OMP_NUM_THREADS"] = str(N)
os.environ["MKL_NUM_THREADS"] = str(N)
os.environ["OPENBLAS_NUM_THREADS"] = str(N)
os.environ["NUMEXPR_NUM_THREADS"] = str(N)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

import control as ct
from scipy.linalg import block_diag

import jax
import jax.numpy as jnp
import diffrax
from jax_sysid.models import CTModel

# Project modules
import src.file_loader as fl

# JAX settings
jax.config.update("jax_enable_x64", True)
DTYPE = jnp.float64

print("SLURM_CPUS_PER_TASK:", os.environ.get("SLURM_CPUS_PER_TASK"))
print("XLA_FLAGS:", os.environ.get("XLA_FLAGS"))
print("JAX devices:", jax.devices())
print("JAX default dtype:", DTYPE)


# %% ======================================================
# CELL 1 — User settings (EDIT THESE)
# =========================================================
print("CWD:", os.getcwd())

# -----------------------------
# File and column settings
# -----------------------------
MPR_PATH = "12to1-25%CNC-3%GQDs _C01.mpr"

TIME_COL = "time/s"
I_COL    = "I/mA"
V_COL    = "Ewe/V"

# -----------------------------
# Cycle selection
# -----------------------------
# Modes:
#   "index"  -> use CYCLE_INDEX
#   "random" -> choose one cycle randomly
CYCLE_MODE  = "random"     # "index" | "random"
CYCLE_INDEX = 0

# Random behavior:
#   RANDOM_SEED = 42   -> reproducible random choice
#   RANDOM_SEED = None -> different random choice on different runs
RANDOM_SEED = 42

# Optional filtering / cycle construction
USE_MIN_CYCLE_LEN = False
MIN_CYCLE_LEN = 50

# Prepend some points before discharge starts, similar to old src logic
INCLUDE_PREVIOUS_SEGMENT = False
N_PREV_POINTS = 10

# -----------------------------
# Data prep
# -----------------------------
FORCE_UNITS = "A"       # "auto" | "mA" | "A"
V_REF       = "none"    # "none" | "first" | "mean"
RESAMPLE    = True

ENFORCE_DISCHARGE_ONLY = True
RAW_DISCHARGE_SIGN     = "negative"

# Initial state guess (proxy stoich)
XN0 = 0.60
XP0 = 0.60
CE0_DEV = 0.0

# Optional trim after selecting cycle
TMAX = -1.0

# Optional drop of first few samples after cycle extraction
DROP_FIRST_N = 0
# DROP_FIRST_N = 1

# -----------------------------
# Stage-2 polynomial settings
# -----------------------------
POLY_DEG = 17
DEGREE_SWEEP = [14, 15, 17]

USE_LN_FEATURE = False

# Physics-consistent surrogate:
# no explicit time-transient basis in Z_hat
USE_FAST_TRANSIENT_BASIS = False
TRANSIENT_TAU_LIST = []

# -----------------------------
# RC branch settings
# -----------------------------
USE_RC = False
R1_INIT_OHM = 0.05
TAU_INIT_S  = 20.0

# -----------------------------
# Optimization settings
# -----------------------------
ADAM_EPOCHS_STAGE2  = 1200
ADAM_EPOCHS_STAGE3A = 1500
ADAM_EPOCHS_STAGE3B = 2000

USE_LBFGS = True
LBFGS_EPOCHS = 300

ADAM_ETA_STAGE2  = 5e-4
ADAM_ETA_STAGE3A = 5e-4
ADAM_ETA_STAGE3B = 2e-4

# -----------------------------
# Solver settings
# -----------------------------
SOLVER_RTOL = 1e-5
SOLVER_ATOL = 1e-8
DT0_DIV     = 5.0
MAX_STEPS   = 10_000_000

# -----------------------------
# Sanity thresholds
# -----------------------------
MIN_MED_ABS_I_A = 5e-3
MIN_XP_RANGE    = 1e-4
MIN_XN_RANGE    = 1e-4

# -----------------------------
# Stage-3 regularization
# -----------------------------
RHO_TH_STAGE2  = 1e-8
RHO_TH_STAGE3A = 1e-8
RHO_TH_STAGE3B = 1e-8

# -----------------------------
# Optional light parameter clips
# -----------------------------
CLIP_RAW_A = 4.0
CLIP_RAW_B = 2.0
CLIP_RAW_Z = 50.0

# -----------------------------
# Real-cell config assumptions
# -----------------------------
N_SERIES_REAL = 1

# -----------------------------
# Debug / gating
# -----------------------------
RUN_DEGREE_SWEEP = False
RUN_STAGE3 = True

# -----------------------------
# New pipeline / generalization settings
# -----------------------------
RUN_GENERALIZATION_TESTS = True

# How many cycles to use in transfer/generalization tests:
#   "selected_subset" | "all_cycles"
GENERALIZATION_SCOPE = "selected_subset"

# GENERALIZATION_SCOPE = "all_cycles"
# REFIT_ALL_CYCLES = True

# Candidate cycles to test:
#   If None, notebook auto-picks a spread across the available cycles.
GENERALIZATION_CYCLE_IDS = None

# Number of cycles for auto-selected subset when GENERALIZATION_SCOPE="selected_subset"
N_GENERALIZATION_CYCLES = 8

# Whether to fit every tested cycle independently (heavier but useful)
RUN_PER_CYCLE_REFITS = True

# If True and GENERALIZATION_SCOPE="all_cycles", refit all cycles
REFIT_ALL_CYCLES = False

# Plot learned nonlinearity for selected cycle and a few extra cycles
RUN_NONLINEARITY_PLOTS = True
NONLINEARITY_PLOT_CYCLE_IDS = None
N_NONLINEARITY_PLOTS = 4

# Degree-comparison test across several cycles
RUN_MULTI_DEGREE_COMPARISON = True
MULTI_DEGREE_LIST = [17, 15, 14]
MULTI_DEGREE_SCOPE = "selected_subset"   # "selected_subset" | "all_cycles"
MULTI_DEGREE_CYCLE_IDS = None
N_MULTI_DEGREE_CYCLES = 6

print("Notebook target: real_data_physics_informed_additive_poly_sysid_pipeline.ipynb")
print("Main polynomial degree:", POLY_DEG)
print("Degree sweep:", DEGREE_SWEEP)
print("USE_LN_FEATURE:", USE_LN_FEATURE)
print("USE_FAST_TRANSIENT_BASIS:", USE_FAST_TRANSIENT_BASIS)
print("TRANSIENT_TAU_LIST:", TRANSIENT_TAU_LIST)
print("USE_RC:", USE_RC)
print("CYCLE_MODE:", CYCLE_MODE)
print("CYCLE_INDEX:", CYCLE_INDEX)
print("RANDOM_SEED:", RANDOM_SEED)
print("USE_MIN_CYCLE_LEN:", USE_MIN_CYCLE_LEN)
print("MIN_CYCLE_LEN:", MIN_CYCLE_LEN)
print("INCLUDE_PREVIOUS_SEGMENT:", INCLUDE_PREVIOUS_SEGMENT)
print("N_PREV_POINTS:", N_PREV_POINTS)
print("RESAMPLE:", RESAMPLE)
print("TMAX:", TMAX)
print("DROP_FIRST_N:", DROP_FIRST_N)
print("RUN_DEGREE_SWEEP:", RUN_DEGREE_SWEEP)
print("RUN_STAGE3:", RUN_STAGE3)
print("RUN_GENERALIZATION_TESTS:", RUN_GENERALIZATION_TESTS)
print("GENERALIZATION_SCOPE:", GENERALIZATION_SCOPE)
print("RUN_PER_CYCLE_REFITS:", RUN_PER_CYCLE_REFITS)
print("RUN_NONLINEARITY_PLOTS:", RUN_NONLINEARITY_PLOTS)
print("RUN_MULTI_DEGREE_COMPARISON:", RUN_MULTI_DEGREE_COMPARISON)


# %% ======================================================
# CELL 2 — Load + plot raw signals
# =========================================================
mpr_file = fl.load_mpr(MPR_PATH)
df0 = pd.DataFrame(mpr_file.data)

print("Columns:", df0.columns.tolist())
assert TIME_COL in df0.columns, f"Missing {TIME_COL}"
assert I_COL in df0.columns, f"Missing {I_COL}"
assert V_COL in df0.columns, f"Missing {V_COL}"

df = df0.set_index(TIME_COL)[[I_COL, V_COL]].copy().sort_index()
df = df.dropna()

print(df.head())

plt.figure(figsize=(12, 6))
plt.subplot(2, 1, 1)
plt.plot(df.index.values, df[I_COL].values)
plt.grid(True)
plt.title("Raw Current")
plt.ylabel(I_COL)

plt.subplot(2, 1, 2)
plt.plot(df.index.values, df[V_COL].values)
plt.grid(True)
plt.title("Raw Voltage")
plt.ylabel(V_COL)
plt.xlabel("time [s]")
plt.tight_layout()
plt.show()


# %% ======================================================
# CELL 3 — Helpers: plots + scoring + manual LS
# =========================================================
def plot_voltage(t, y, yhat=None, title="Voltage"):
    t = np.asarray(t).reshape(-1)
    y = np.asarray(y).reshape(-1, 1)

    plt.figure(figsize=(10, 4))
    plt.plot(t, y[:, 0], label="Measured")
    if yhat is not None:
        yhat = np.asarray(yhat).reshape(-1, 1)
        plt.plot(t, yhat[:, 0], "--", label="Pred")
    plt.grid(True)
    plt.legend()
    plt.xlabel("t [s]")
    plt.ylabel("V [V]")
    plt.title(title)
    ax = plt.gca()
    ax.yaxis.set_major_formatter(mticker.ScalarFormatter(useOffset=False))
    ax.ticklabel_format(axis="y", style="plain", useOffset=False)
    plt.tight_layout()
    plt.show()


def vec_reshape(y):
    y = np.asarray(y)
    if y.ndim == 1:
        y = y.reshape(-1, 1)
    return y


def compute_scores(Y, Yhat, fit="R2"):
    Y = vec_reshape(Y)
    Yhat = vec_reshape(Yhat)
    y = Y[:, 0]
    yh = Yhat[:, 0]
    fit = fit.lower()

    if fit == "r2":
        denom = np.sum((y - np.mean(y)) ** 2) + 1e-12
        return 100.0 * (1.0 - np.sum((yh - y) ** 2) / denom)

    if fit == "rmse":
        return float(np.sqrt(np.mean((yh - y) ** 2)))

    if fit == "bfr":
        denom = np.sum((y - np.mean(y)) ** 2) + 1e-12
        return 100.0 * (1.0 - np.linalg.norm(yh - y) / np.sqrt(denom))

    raise ValueError("fit must be one of: R2 | BFR | RMSE")


def summarize_err(y_true: np.ndarray, y_pred: np.ndarray, name=""):
    y_true = vec_reshape(y_true)
    y_pred = vec_reshape(y_pred)
    err = y_pred[:, 0] - y_true[:, 0]

    rmse = float(np.sqrt(np.mean(err ** 2)))
    mae  = float(np.mean(np.abs(err)))
    p95  = float(np.percentile(np.abs(err), 95))
    p99  = float(np.percentile(np.abs(err), 99))
    mx   = float(np.max(np.abs(err)))

    print(f"\n{name} error summary:")
    print(f"  RMSE  = {rmse:.6g} V")
    print(f"  MAE   = {mae:.6g} V")
    print(f"  P95   = {p95:.6g} V")
    print(f"  P99   = {p99:.6g} V")
    print(f"  MAX   = {mx:.6g} V")
    return dict(rmse=rmse, mae=mae, p95=p95, p99=p99, max_abs=mx)


def report_fit(name, Y_true, Y_pred):
    Y_true = vec_reshape(Y_true)
    Y_pred = vec_reshape(Y_pred)
    err = Y_pred - Y_true
    max_abs = float(np.max(np.abs(err)))
    rmse = float(np.sqrt(np.mean(err ** 2)))
    r2 = float(compute_scores(Y_true, Y_pred, fit="R2"))
    bfr = float(compute_scores(Y_true, Y_pred, fit="BFR"))
    print(f"{name}: max|err|={max_abs:.6g}, rmse={rmse:.6g}, R2%={r2:.3f}, BFR%={bfr:.3f}")
    return dict(max_abs=max_abs, rmse=rmse, r2=r2, bfr=bfr)


def plot_current_and_voltage(t, u, y, title="Current and voltage"):
    t = np.asarray(t).reshape(-1)
    u = np.asarray(u).reshape(-1)
    y = np.asarray(y).reshape(-1)

    plt.figure(figsize=(11, 6))

    plt.subplot(2, 1, 1)
    plt.plot(t, u)
    plt.grid(True)
    plt.ylabel("I [A]")
    plt.title(title)

    plt.subplot(2, 1, 2)
    plt.plot(t, y)
    plt.grid(True)
    plt.ylabel("V [V]")
    plt.xlabel("t [s]")

    plt.tight_layout()
    plt.show()


def report_signal_span(name, x):
    x = np.asarray(x).reshape(-1)
    print(
        f"{name}: min={float(np.min(x)):.6g}, "
        f"max={float(np.max(x)):.6g}, "
        f"span={float(np.max(x)-np.min(x)):.6g}, "
        f"std={float(np.std(x)):.6g}"
    )


def simple_line_fit_rmse(t, y):
    t = np.asarray(t).reshape(-1)
    y = np.asarray(y).reshape(-1)
    p = np.polyfit(t, y, 1)
    ylin = np.polyval(p, t)
    rmse = float(np.sqrt(np.mean((y - ylin) ** 2)))
    return rmse, p


def plot_residuals(t, y, yhat, title="Residuals"):
    t = np.asarray(t).reshape(-1)
    y = np.asarray(y).reshape(-1)
    yhat = np.asarray(yhat).reshape(-1)
    err = yhat - y

    plt.figure(figsize=(10, 4))
    plt.plot(t, err)
    plt.grid(True)
    plt.xlabel("t [s]")
    plt.ylabel("Pred - Meas [V]")
    plt.title(title)
    plt.tight_layout()
    plt.show()


def plot_voltage_with_baselines(t, y, curves: dict, title="Voltage comparison"):
    t = np.asarray(t).reshape(-1)
    y = np.asarray(y).reshape(-1)

    plt.figure(figsize=(10, 4))
    plt.plot(t, y, label="Measured", linewidth=2)

    for name, vals in curves.items():
        vals = np.asarray(vals).reshape(-1)
        plt.plot(t, vals, "--", label=name)

    plt.grid(True)
    plt.xlabel("t [s]")
    plt.ylabel("V [V]")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()


def solve_ls_normal_eq(X: np.ndarray, y: np.ndarray, ridge: float = 0.0):
    X = np.asarray(X, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64).reshape(-1)

    XtX = X.T @ X
    Xty = X.T @ y

    if ridge > 0.0:
        XtX = XtX + ridge * np.eye(X.shape[1], dtype=np.float64)

    beta = np.linalg.solve(XtX, Xty)
    return beta


def solve_ls_qr(X: np.ndarray, y: np.ndarray):
    X = np.asarray(X, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64).reshape(-1)

    Q, R = np.linalg.qr(X, mode="reduced")
    beta = np.linalg.solve(R, Q.T @ y)
    return beta


def compare_ls_solvers(X: np.ndarray, y: np.ndarray, ridge: float = 1e-12, label="LS compare"):
    X = np.asarray(X, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64).reshape(-1)

    beta_ne = solve_ls_normal_eq(X, y, ridge=ridge)
    beta_qr = solve_ls_qr(X, y)
    beta_np, *_ = np.linalg.lstsq(X, y, rcond=None)

    diff_ne_np = float(np.linalg.norm(beta_ne - beta_np))
    diff_qr_np = float(np.linalg.norm(beta_qr - beta_np))
    diff_ne_qr = float(np.linalg.norm(beta_ne - beta_qr))

    print(f"\n[{label}] coefficient comparison")
    print(f"  ||beta_normal_eq - beta_lstsq|| = {diff_ne_np:.6e}")
    print(f"  ||beta_qr        - beta_lstsq|| = {diff_qr_np:.6e}")
    print(f"  ||beta_normal_eq - beta_qr   || = {diff_ne_qr:.6e}")

    return {
        "beta_normal_eq": beta_ne,
        "beta_qr": beta_qr,
        "beta_lstsq": beta_np,
        "diff_ne_np": diff_ne_np,
        "diff_qr_np": diff_qr_np,
        "diff_ne_qr": diff_ne_qr,
    }


def choose_spread_indices(n_total: int, n_pick: int) -> List[int]:
    if n_total <= 0:
        return []
    n_pick = max(1, min(n_pick, n_total))
    if n_pick == 1:
        return [0]
    vals = np.linspace(0, n_total - 1, n_pick)
    out = sorted(set(int(round(v)) for v in vals))
    while len(out) < n_pick:
        for k in range(n_total):
            if k not in out:
                out.append(k)
                if len(out) == n_pick:
                    break
    return sorted(out[:n_pick])


def monotonicity_fraction(x: np.ndarray) -> float:
    x = np.asarray(x).reshape(-1)
    if len(x) < 2:
        return np.nan
    dx = np.diff(x)
    return float(np.mean(dx >= 0.0))


def safe_corr(x: np.ndarray, y: np.ndarray) -> float:
    x = np.asarray(x).reshape(-1)
    y = np.asarray(y).reshape(-1)
    if len(x) < 2 or np.std(x) < 1e-15 or np.std(y) < 1e-15:
        return np.nan
    return float(np.corrcoef(x, y)[0, 1])


# %% ======================================================
# CELL 4 — Cycle finding + selection by index or random
# =========================================================
def get_previous_segment_by_iloc(df: pd.DataFrame, start_pos: int, n_prev_points: int = 10) -> pd.DataFrame:
    lo = max(0, start_pos - int(n_prev_points))
    return df.iloc[lo:start_pos].copy()


def find_discharging_cycles_with_meta(
    df: pd.DataFrame,
    i_col: str,
    tol_I: float = 1e-9,
    min_len: int | None = None,
    include_previous_segment: bool = False,
    n_prev_points: int = 10,
):
    I = df[i_col].to_numpy(dtype=float).reshape(-1)
    mask = I < -tol_I

    cycles = []
    cycle_meta = []

    start = None
    for k in range(len(mask)):
        if mask[k] and start is None:
            start = k

        end_now = (start is not None) and ((not mask[k]) or (k == len(mask) - 1))
        if end_now:
            end = k if (not mask[k]) else (k + 1)
            seg_len_core = end - start

            keep_seg = True
            if min_len is not None:
                keep_seg = seg_len_core >= int(min_len)

            if keep_seg:
                core_seg = df.iloc[start:end].copy()

                if include_previous_segment:
                    prev_seg = get_previous_segment_by_iloc(df, start, n_prev_points=n_prev_points)
                    seg = pd.concat([prev_seg, core_seg], axis=0)
                else:
                    seg = core_seg

                cycles.append(seg)

                cycle_meta.append(
                    dict(
                        start_row=int(start),
                        end_row=int(end - 1),
                        n_points_total=int(len(seg)),
                        n_points_core=int(seg_len_core),
                        t_start=float(seg.index[0]),
                        t_end=float(seg.index[-1]),
                        duration=float(seg.index[-1] - seg.index[0]),
                        i_mean=float(np.mean(core_seg[i_col].to_numpy(dtype=float))),
                        i_min=float(np.min(core_seg[i_col].to_numpy(dtype=float))),
                        i_max=float(np.max(core_seg[i_col].to_numpy(dtype=float))),
                    )
                )
            start = None

    return cycles, cycle_meta


min_len_for_search = int(MIN_CYCLE_LEN) if USE_MIN_CYCLE_LEN else None

cycles, cycle_meta = find_discharging_cycles_with_meta(
    df,
    i_col=I_COL,
    tol_I=1e-9,
    min_len=min_len_for_search,
    include_previous_segment=INCLUDE_PREVIOUS_SEGMENT,
    n_prev_points=N_PREV_POINTS,
)

if min_len_for_search is None:
    print(f"Found {len(cycles)} discharge segments with no minimum-length filter")
else:
    print(f"Found {len(cycles)} discharge segments with min_len={min_len_for_search}")

if len(cycles) == 0:
    raise RuntimeError("No discharge segments found. Check sign, current column, or cycle filters.")

cycle_lengths_total = np.array([m["n_points_total"] for m in cycle_meta], dtype=int)
cycle_lengths_core = np.array([m["n_points_core"] for m in cycle_meta], dtype=int)
cycle_durations = np.array([m["duration"] for m in cycle_meta], dtype=float)

print("[CYCLE SUMMARY]")
print("  total length min/median/max :", int(np.min(cycle_lengths_total)), int(np.median(cycle_lengths_total)), int(np.max(cycle_lengths_total)))
print("  core  length min/median/max :", int(np.min(cycle_lengths_core)), int(np.median(cycle_lengths_core)), int(np.max(cycle_lengths_core)))
print("  duration min/median/max [s] :", float(np.min(cycle_durations)), float(np.median(cycle_durations)), float(np.max(cycle_durations)))

mode = str(CYCLE_MODE).strip().lower()

if mode == "index":
    chosen = int(CYCLE_INDEX)
    chosen = max(0, min(chosen, len(cycles) - 1))
    selection_note = f"index mode -> using cycle {chosen}"

elif mode == "random":
    if RANDOM_SEED is None:
        chosen = int(np.random.randint(0, len(cycles)))
        selection_note = (
            f"random mode -> RANDOM_SEED=None, selected cycle {chosen} "
            f"(this may change from run to run)"
        )
    else:
        rng = np.random.default_rng(int(RANDOM_SEED))
        chosen = int(rng.integers(low=0, high=len(cycles)))
        selection_note = (
            f"random mode -> seed={RANDOM_SEED}, selected cycle {chosen} "
            f"(reuse with CYCLE_MODE='index', CYCLE_INDEX={chosen})"
        )
else:
    raise ValueError("CYCLE_MODE must be 'index' or 'random'.")

cycle_df = cycles[chosen].copy()
meta = cycle_meta[chosen]

print(selection_note)
print(
    f"Selected cycle index: {chosen} | "
    f"n_points_total={meta['n_points_total']} | "
    f"n_points_core={meta['n_points_core']} | "
    f"duration={meta['duration']:.6g} s | "
    f"raw time range=[{meta['t_start']:.6g}, {meta['t_end']:.6g}]"
)
print(
    f"Current summary on selected core cycle: "
    f"mean={meta['i_mean']:.6g}, min={meta['i_min']:.6g}, max={meta['i_max']:.6g}"
)

plt.figure(figsize=(12, 4))
plt.plot(cycle_df.index.values, cycle_df[I_COL].values)
plt.grid(True)
plt.title(f"Selected discharge cycle (index={chosen}, mode={mode})")
plt.ylabel(I_COL)
plt.xlabel("time [s]")
plt.tight_layout()
plt.show()


# %% ======================================================
# CELL 5 — Prepare (t,U,Y): units, discharge-only, resample, trim
# =========================================================
def _sanity_report_current_units(I_raw: np.ndarray):
    I_as_A  = I_raw
    I_as_mA = I_raw * 1e-3
    print("[UNIT CHECK] If column is actually A:")
    print("  range [A]:", float(np.min(I_as_A)), float(np.max(I_as_A)),
          " median|I|:", float(np.median(np.abs(I_as_A))))
    print("[UNIT CHECK] If column is actually mA:")
    print("  range [A]:", float(np.min(I_as_mA)), float(np.max(I_as_mA)),
          " median|I|:", float(np.median(np.abs(I_as_mA))))


def _guess_I_in_amps(I_raw: np.ndarray) -> Tuple[np.ndarray, str]:
    med = float(np.nanmedian(np.abs(I_raw)))
    if med >= 1e-3:
        return I_raw, f"auto units: treating as A (median|I|={med:.6g})"
    if med >= 1.0:
        return I_raw * 1e-3, f"auto units: mA->A (median|I|={med:.6g} mA)"
    return I_raw, f"auto units: ambiguous, using A (median|I|={med:.6g})"


def resample_uniform(t: np.ndarray, u: np.ndarray, y: np.ndarray, dt: float):
    t0, t1 = float(t[0]), float(t[-1])
    tg = np.arange(t0, t1 + dt, dt, dtype=np.float64)
    u1 = np.interp(tg, t, u[:, 0]).reshape(-1, 1)
    y1 = np.interp(tg, t, y[:, 0]).reshape(-1, 1)
    return tg, u1, y1


t = cycle_df.index.to_numpy(dtype=np.float64).reshape(-1)
I_raw = cycle_df[I_COL].to_numpy(dtype=np.float64).reshape(-1)
V_raw = cycle_df[V_COL].to_numpy(dtype=np.float64).reshape(-1)

_sanity_report_current_units(I_raw)

if str(FORCE_UNITS).lower() == "auto":
    I_A, note = _guess_I_in_amps(I_raw)
    print(note)
elif str(FORCE_UNITS).lower() == "ma":
    I_A = I_raw * 1e-3
    print("FORCE_UNITS=mA => converting to A")
elif str(FORCE_UNITS).lower() == "a":
    I_A = I_raw
    print("FORCE_UNITS=A => using as A")
else:
    raise ValueError("FORCE_UNITS must be 'auto'|'mA'|'A'")

# make time strictly increasing
keep = np.ones_like(t, dtype=bool)
keep[1:] = t[1:] > t[:-1]
t, I_A, V_raw = t[keep], I_A[keep], V_raw[keep]

# rebase time
t = t - t[0]

if ENFORCE_DISCHARGE_ONLY:
    tol = 1e-12
    if RAW_DISCHARGE_SIGN == "negative":
        mask = I_A < -tol
    elif RAW_DISCHARGE_SIGN == "positive":
        mask = I_A > +tol
    else:
        raise ValueError("RAW_DISCHARGE_SIGN must be 'negative' or 'positive'")

    before = len(t)
    t, I_A, V_raw = t[mask], I_A[mask], V_raw[mask]
    if len(t) < 5:
        raise RuntimeError("After discharge-only filter, not enough points.")
    t = t - t[0]
    print(f"Discharge-only kept {len(t)}/{before} points")

if DROP_FIRST_N > 0:
    if len(t) <= DROP_FIRST_N + 3:
        raise RuntimeError("DROP_FIRST_N is too large for this selected cycle.")
    t = t[DROP_FIRST_N:]
    I_A = I_A[DROP_FIRST_N:]
    V_raw = V_raw[DROP_FIRST_N:]
    t = t - t[0]
    print(f"Dropped first {DROP_FIRST_N} samples after cycle extraction")

if V_REF == "none":
    V = V_raw
elif V_REF == "first":
    V = V_raw - float(V_raw[0])
elif V_REF == "mean":
    V = V_raw - float(np.mean(V_raw))
else:
    raise ValueError("V_REF must be none|first|mean")

t_np = t.reshape(-1)
U_np = I_A.reshape(-1, 1)
Y_np = V.reshape(-1, 1)

print("Prepared shapes (before resample/trim):", t_np.shape, U_np.shape, Y_np.shape)
print("I range [A]:", float(U_np.min()), float(U_np.max()),
      " median|I| [A]:", float(np.median(np.abs(U_np[:, 0]))))
print("V range [V]:", float(Y_np.min()), float(Y_np.max()))

plot_voltage(t_np, Y_np, title="Prepared (pre-resample/trim)")
plot_current_and_voltage(t_np, U_np[:, 0], Y_np[:, 0], title="Prepared signals (pre-resample/trim)")

med_abs_I = float(np.median(np.abs(U_np[:, 0])))
if med_abs_I < float(MIN_MED_ABS_I_A):
    print("\n[WARN] Current magnitude is very small -> dynamics may look stiff/flat.")
    print("       Double-check FORCE_UNITS and chosen cycle.\n")

if RESAMPLE and len(t_np) > 2:
    dt_med = float(np.median(np.diff(t_np)))
    if not (np.isfinite(dt_med) and dt_med > 0):
        raise RuntimeError("Bad dt_med; cannot resample.")
    t_np, U_np, Y_np = resample_uniform(t_np, U_np, Y_np, dt=dt_med)
    print("Resampled: dt_med =", dt_med, "len =", len(t_np))
else:
    print("RESAMPLE=False -> keeping original time grid")

if TMAX > 0:
    m = t_np <= float(TMAX)
    t_np, U_np, Y_np = t_np[m], U_np[m], Y_np[m]
    print("Trimmed to TMAX:", TMAX, "len =", len(t_np))

print("Prepared shapes (final):", t_np.shape, U_np.shape, Y_np.shape)
print("Final I range [A]:", float(U_np.min()), float(U_np.max()),
      " median|I| [A]:", float(np.median(np.abs(U_np[:, 0]))))
print("Final V range [V]:", float(Y_np.min()), float(Y_np.max()))

plot_voltage(t_np, Y_np, title="Prepared (final)")
plot_current_and_voltage(t_np, U_np[:, 0], Y_np[:, 0], title="Prepared signals (final)")

print("\n[DEBUG] Final prepared signal spans")
report_signal_span("time", t_np)
report_signal_span("current [A]", U_np[:, 0])
report_signal_span("voltage [V]", Y_np[:, 0])

line_rmse, line_coeff = simple_line_fit_rmse(t_np, Y_np[:, 0])
print(f"[DEBUG] Voltage vs. straight-line RMSE = {line_rmse:.6g} V")
print(f"[DEBUG] Linear trend coeffs = slope {line_coeff[0]:.6g}, intercept {line_coeff[1]:.6g}")


# %% ======================================================
# CELL 6 — Nominal 14-state proxy model + proxy signals
# =========================================================
IDX = {
    "cn": slice(0, 4),
    "cp": slice(4, 8),
    "ce": slice(8, 14),
    "cn_surf": 3,
    "cp_surf": 7,
    "ce_left": 8,
    "ce_right": 13,
}


@dataclass
class Config:
    R: float = 8.314462618
    F: float = 96485.33212
    T: float = 298.15
    T_ref: float = 298.15

    L1: float = 25e-6
    L2: float = 20e-6
    L3: float = 25e-6
    Rn: float = 5e-6
    Rp: float = 5e-6
    A: float = 1.0

    Dn: float = 1e-14
    Dp: float = 1e-14
    De: float = 7.23e-10
    eps: float = 0.30

    kappa_n_eff: float = 1.0
    kappa_s_eff: float = 1.0
    kappa_p_eff: float = 1.0

    a_s_n: float = 1.0e6
    a_s_p: float = 1.0e6
    k_n0: float = 2.0e-11
    k_p0: float = 2.0e-11
    use_arrhenius: bool = False
    Ea_n: float = 0.0
    Ea_p: float = 0.0

    lam_n: float = 0.0
    lam_p: float = 0.0

    csn_max: float = 3.1e4
    csp_max: float = 3.1e4

    ce0: float = 1000.0
    t_plus: float = 0.38
    k_f: float = 1.0
    R_ohm: float = 0.0
    use_dynamic_film: bool = False
    Rf: float = 0.0
    L_sei: float = 0.0
    kappa_sei: float = 1.0

    ce_is_deviation: bool = True
    discharge_positive: bool = False
    ln_orientation: str = "right_over_left"
    eta_mode: str = "diff"

    I_dyn: float = 2.0
    I_for_voltage: float = 2.0

    theta_guard: float = 1e-3
    I0_floor_p: float = 1e-2
    I0_floor_n: float = 1e-2
    bv_scale: float = 0.7
    N_series: int = N_SERIES_REAL


def build_An(cfg: Config) -> np.ndarray:
    s = cfg.Dn / (cfg.Rn ** 2)
    A = np.zeros((4, 4))
    A[0, 0], A[0, 1] = -24 * s, 24 * s
    A[1, 0], A[1, 1], A[1, 2] = 16 * s, -40 * s, 24 * s
    A[2, 1], A[2, 2], A[2, 3] = 16 * s, -40 * s, 24 * s
    A[3, 2], A[3, 3] = 16 * s, -16 * s
    return A


def build_Bn(cfg: Config) -> np.ndarray:
    sign = -1.0 if cfg.discharge_positive else +1.0
    b = np.zeros((4, 1))
    b[-1, 0] = sign * (6.0 / cfg.Rn) * (1.0 / (cfg.F * cfg.a_s_n * cfg.A * cfg.L1))
    return b


def build_Ap(cfg: Config) -> np.ndarray:
    s = cfg.Dp / (cfg.Rp ** 2)
    A = np.zeros((4, 4))
    A[0, 0], A[0, 1] = -24 * s, 24 * s
    A[1, 0], A[1, 1], A[1, 2] = 16 * s, -40 * s, 24 * s
    A[2, 1], A[2, 2], A[2, 3] = 16 * s, -40 * s, 24 * s
    A[3, 2], A[3, 3] = 16 * s, -16 * s
    return A


def build_Bp(cfg: Config) -> np.ndarray:
    sign = +1.0 if cfg.discharge_positive else -1.0
    b = np.zeros((4, 1))
    b[-1, 0] = sign * (6.0 / cfg.Rp) * (1.0 / (cfg.F * cfg.a_s_p * cfg.A * cfg.L3))
    return b


def build_Ae(cfg: Config) -> np.ndarray:
    K = cfg.De / cfg.eps
    Ae = np.zeros((6, 6))

    w_in = lambda L: K * 4.0 / (L ** 2)
    w_intf = lambda La, Lb: K * 16.0 / ((La + Lb) ** 2)

    w11 = w_in(cfg.L1)
    w12 = w_intf(cfg.L1, cfg.L2)
    w23 = w_in(cfg.L2)
    w34 = w_intf(cfg.L2, cfg.L3)
    w45 = w_in(cfg.L3)

    Ae[0, 0] = -(w11)
    Ae[0, 1] = +(w11)
    Ae[1, 0] = +(w11)
    Ae[1, 1] = -(w11 + w12)
    Ae[1, 2] = +(w12)
    Ae[2, 1] = +(w12)
    Ae[2, 2] = -(w12 + w23)
    Ae[2, 3] = +(w23)
    Ae[3, 2] = +(w23)
    Ae[3, 3] = -(w23 + w34)
    Ae[3, 4] = +(w34)
    Ae[4, 3] = +(w34)
    Ae[4, 4] = -(w34 + w45)
    Ae[4, 5] = +(w45)
    Ae[5, 4] = +(w45)
    Ae[5, 5] = -(w45)

    return Ae


def build_Be(cfg: Config) -> np.ndarray:
    b = np.zeros((6, 1))
    sign_left  = -1.0 if cfg.discharge_positive else +1.0
    sign_right = +1.0 if cfg.discharge_positive else -1.0
    s1 = sign_left  * (1.0 - cfg.t_plus) / (cfg.F * cfg.A * cfg.L1 * cfg.eps)
    s3 = sign_right * (1.0 - cfg.t_plus) / (cfg.F * cfg.A * cfg.L3 * cfg.eps)
    b[0, 0] = s1
    b[1, 0] = s1
    b[4, 0] = s3
    b[5, 0] = s3
    return b


def assemble_system(cfg: Config):
    An, Ap, Ae = build_An(cfg), build_Ap(cfg), build_Ae(cfg)
    Bn, Bp, Be = build_Bn(cfg), build_Bp(cfg), build_Be(cfg)
    Aglob = block_diag(An, Ap, Ae)
    Bglob = np.vstack([Bn, Bp, Be])
    S = ct.ss(Aglob, Bglob, np.eye(14), np.zeros((14, 1)))
    return S, Aglob, Bglob


def make_x0(cfg: Config, theta_n0=0.6, theta_p0=0.6, ce0=0.0):
    x0 = np.zeros(14, dtype=np.float64)
    x0[IDX["cn"]] = float(theta_n0) * cfg.csn_max
    x0[IDX["cp"]] = float(theta_p0) * cfg.csp_max
    x0[IDX["ce"]] = float(ce0)
    return x0


CFG = Config()
CFG.discharge_positive = False

Sx, A_nom_np, B_nom_np = assemble_system(CFG)
x0_nom = make_x0(CFG, theta_n0=XN0, theta_p0=XP0, ce0=CE0_DEV)

resp = ct.forced_response(Sx, T=t_np, U=U_np[:, 0], X0=x0_nom)
X_proxy = np.asarray(resp.states).T

xp_sig = (X_proxy[:, IDX["cp_surf"]] / CFG.csp_max).astype(np.float64)
xn_sig = (X_proxy[:, IDX["cn_surf"]] / CFG.csn_max).astype(np.float64)

ceL_raw_sig = X_proxy[:, IDX["ce_left"]].astype(np.float64)
ceR_raw_sig = X_proxy[:, IDX["ce_right"]].astype(np.float64)

if CFG.ce_is_deviation:
    ceL_sig = CFG.ce0 + ceL_raw_sig
    ceR_sig = CFG.ce0 + ceR_raw_sig
else:
    ceL_sig = ceL_raw_sig
    ceR_sig = ceR_raw_sig

xp_rng = float(xp_sig.max() - xp_sig.min())
xn_rng = float(xn_sig.max() - xn_sig.min())
ceL_rng = float(ceL_sig.max() - ceL_sig.min())
ceR_rng = float(ceR_sig.max() - ceR_sig.min())

print("[PROXY] xp range:", float(xp_sig.min()), float(xp_sig.max()), " span:", xp_rng)
print("[PROXY] xn range:", float(xn_sig.min()), float(xn_sig.max()), " span:", xn_rng)
print("[PROXY] ceL min/max:", float(np.min(ceL_sig)), float(np.max(ceL_sig)), " span:", ceL_rng)
print("[PROXY] ceR min/max:", float(np.min(ceR_sig)), float(np.max(ceR_sig)), " span:", ceR_rng)

if xp_rng < MIN_XP_RANGE or xn_rng < MIN_XN_RANGE:
    print("\n[WARN] Proxies barely move -> model may look stiff.")
    print("       This usually means wrong current units, wrong cycle, or too little excitation.\n")

plt.figure(figsize=(12, 6))
plt.subplot(3, 1, 1)
plt.plot(t_np, xp_sig)
plt.grid(True)
plt.ylabel("xp proxy")

plt.subplot(3, 1, 2)
plt.plot(t_np, xn_sig)
plt.grid(True)
plt.ylabel("xn proxy")

plt.subplot(3, 1, 3)
plt.plot(t_np, ceL_sig, label="ceL")
plt.plot(t_np, ceR_sig, label="ceR")
plt.grid(True)
plt.ylabel("ce proxy")
plt.xlabel("time [s]")
plt.legend()
plt.tight_layout()
plt.show()

print("\n[DEBUG] Proxy signal spans")
report_signal_span("xp proxy", xp_sig)
report_signal_span("xn proxy", xn_sig)
report_signal_span("ceL proxy", ceL_sig)
report_signal_span("ceR proxy", ceR_sig)

if xp_rng < 1e-7 and xn_rng < 1e-7:
    print("[HARD WARNING] Stage 2 cannot learn meaningful state-dependent surrogate from this run.")
    print("               Fix excitation/units first before trusting any optimizer result.")

ln_ce_ratio_sig = np.log(np.maximum(ceR_sig / np.maximum(ceL_sig, 1e-12), 1e-12))

print("\n[DEBUG] Feature leverage check")
report_signal_span("xp proxy", xp_sig)
report_signal_span("xn proxy", xn_sig)
report_signal_span("ln(ceR/ceL)", ln_ce_ratio_sig)

plt.figure(figsize=(12, 4))
plt.plot(t_np, ln_ce_ratio_sig)
plt.grid(True)
plt.xlabel("t [s]")
plt.ylabel("ln(ceR/ceL)")
plt.title("Electrolyte log feature on proxy trajectory")
plt.tight_layout()
plt.show()

if np.max(np.abs(ln_ce_ratio_sig)) < 1e-5:
    print("[INFO] ln(ceR/ceL) feature is effectively inactive on this dataset.")


# %% ======================================================
# CELL 7 — JAX helpers + A(thetaA), B(thetaB) parameterization
# =========================================================
def softplus(x):
    return jnp.log1p(jnp.exp(-jnp.abs(x))) + jnp.maximum(x, 0.0)


def pos(x, floor=1e-12):
    return softplus(x) + floor


def softplus_inv(y):
    y = float(max(y, 1e-12))
    if y > 20:
        return y
    return np.log(np.expm1(y))


def raw_from_pos(val, floor=1e-12):
    val = float(max(val, floor + 1e-12))
    return softplus_inv(val - floor)


def thetaA_nom_from_cfg(cfg: Config) -> np.ndarray:
    th1 = cfg.Dn / (cfg.Rn ** 2)
    th2 = cfg.Dp / (cfg.Rp ** 2)
    K = cfg.De / cfg.eps
    th3 = K / (cfg.L1 ** 2)
    th4 = K / ((cfg.L1 + cfg.L2) ** 2)
    th5 = K / (cfg.L2 ** 2)
    th6 = K / ((cfg.L2 + cfg.L3) ** 2)
    th7 = K / (cfg.L3 ** 2)
    return np.array([th1, th2, th3, th4, th5, th6, th7], dtype=np.float64)


def thetaB_nom_from_cfg(cfg: Config) -> np.ndarray:
    sign_n = -1.0 if cfg.discharge_positive else +1.0
    sign_p = +1.0 if cfg.discharge_positive else -1.0

    th8 = sign_n * (1.0 / cfg.Rn) * (1.0 / (cfg.F * cfg.a_s_n * cfg.A * cfg.L1))
    th9 = sign_p * (1.0 / cfg.Rp) * (1.0 / (cfg.F * cfg.a_s_p * cfg.A * cfg.L3))

    sign_left  = -1.0 if cfg.discharge_positive else +1.0
    sign_right = +1.0 if cfg.discharge_positive else -1.0
    th10 = sign_left  * (1.0 - cfg.t_plus) / (cfg.F * cfg.A * cfg.L1 * cfg.eps)
    th11 = sign_right * (1.0 - cfg.t_plus) / (cfg.F * cfg.A * cfg.L3 * cfg.eps)

    return np.array([th8, th9, th10, th11], dtype=np.float64)


thetaA_nom_np = thetaA_nom_from_cfg(CFG)
thetaB_nom_np = thetaB_nom_from_cfg(CFG)

thetaA_nom = jnp.array(thetaA_nom_np, dtype=DTYPE)
thetaB_nom = jnp.array(thetaB_nom_np, dtype=DTYPE)


@jax.jit
def build_A_from_thetaA(thetaA: jnp.ndarray) -> jnp.ndarray:
    th1, th2, th3, th4, th5, th6, th7 = thetaA

    An = jnp.array([
        [-24 * th1,  24 * th1,    0.0,    0.0],
        [ 16 * th1, -40 * th1, 24 * th1,    0.0],
        [   0.0,    16 * th1, -40 * th1, 24 * th1],
        [   0.0,      0.0,    16 * th1, -16 * th1],
    ], dtype=DTYPE)

    Ap = jnp.array([
        [-24 * th2,  24 * th2,    0.0,    0.0],
        [ 16 * th2, -40 * th2, 24 * th2,    0.0],
        [   0.0,    16 * th2, -40 * th2, 24 * th2],
        [   0.0,      0.0,    16 * th2, -16 * th2],
    ], dtype=DTYPE)

    Ae = jnp.array([
        [-4 * th3,                     4 * th3,                    0.0,                    0.0,                    0.0,      0.0],
        [ 4 * th3, -(4 * th3 + 16 * th4),                 16 * th4,                    0.0,                    0.0,      0.0],
        [ 0.0,                    16 * th4, -(16 * th4 + 4 * th5),                  4 * th5,                    0.0,      0.0],
        [ 0.0,                      0.0,                  4 * th5, -(4 * th5 + 16 * th6),                 16 * th6,     0.0],
        [ 0.0,                      0.0,                    0.0,                  16 * th6, -(16 * th6 + 4 * th7),  4 * th7],
        [ 0.0,                      0.0,                    0.0,                    0.0,                  4 * th7, -4 * th7],
    ], dtype=DTYPE)

    A = jnp.zeros((14, 14), dtype=DTYPE)
    A = A.at[0:4, 0:4].set(An)
    A = A.at[4:8, 4:8].set(Ap)
    A = A.at[8:14, 8:14].set(Ae)
    return A


@jax.jit
def build_B_from_thetaB(thetaB: jnp.ndarray) -> jnp.ndarray:
    th8, th9, th10, th11 = thetaB
    B = jnp.zeros((14, 1), dtype=DTYPE)
    B = B.at[3, 0].set(6.0 * th8)
    B = B.at[7, 0].set(6.0 * th9)
    B = B.at[8, 0].set(th10)
    B = B.at[9, 0].set(th10)
    B = B.at[12, 0].set(th11)
    B = B.at[13, 0].set(th11)
    return B


# %% ======================================================
# CELL 8 — Additive polynomial surrogate builder
# =========================================================
def make_additive_poly_surrogate_fns(
    cfg: Config,
    deg: int,
    xp_ref: float,
    xn_ref: float,
    xp_scale: float,
    xn_scale: float,
    use_ln_feature: bool = False,
):
    xp_powers = jnp.arange(1, deg + 1, dtype=DTYPE)
    xn_powers = jnp.arange(1, deg + 1, dtype=DTYPE)

    xp_ref_j = DTYPE(float(xp_ref))
    xn_ref_j = DTYPE(float(xn_ref))
    xp_scale_j = DTYPE(float(max(xp_scale, 1e-12)))
    xn_scale_j = DTYPE(float(max(xn_scale, 1e-12)))

    @jax.jit
    def additive_features_from_x(x14: jnp.ndarray) -> jnp.ndarray:
        xp = jnp.clip(x14[IDX["cp_surf"]] / DTYPE(cfg.csp_max), 1e-9, 1.0 - 1e-9)
        xn = jnp.clip(x14[IDX["cn_surf"]] / DTYPE(cfg.csn_max), 1e-9, 1.0 - 1e-9)

        dxp = (xp - xp_ref_j) / xp_scale_j
        dxn = (xn - xn_ref_j) / xn_scale_j

        xp_feats = dxp ** xp_powers
        xn_feats = dxn ** xn_powers

        return jnp.concatenate(
            [jnp.ones((1,), dtype=DTYPE), xp_feats, xn_feats],
            axis=0
        )

    @jax.jit
    def ln_ce_ratio_from_x(x14: jnp.ndarray) -> jnp.ndarray:
        ceL_raw = x14[IDX["ce_left"]]
        ceR_raw = x14[IDX["ce_right"]]

        ceL = (DTYPE(cfg.ce0) + ceL_raw) if cfg.ce_is_deviation else ceL_raw
        ceR = (DTYPE(cfg.ce0) + ceR_raw) if cfg.ce_is_deviation else ceR_raw

        ceL = jnp.maximum(ceL, 1e-12)
        ceR = jnp.maximum(ceR, 1e-12)

        ln_arg = (ceR / ceL) if (cfg.ln_orientation == "right_over_left") else (ceL / ceR)
        return jnp.log(jnp.maximum(ln_arg, 1e-12))

    n_base = 1 + deg + deg
    n_thetaZ = n_base + (1 if use_ln_feature else 0)

    def unpack_thetaZ(thetaZ: jnp.ndarray):
        thetaZ = thetaZ.reshape(-1)
        theta_base = thetaZ[:n_base]

        if use_ln_feature:
            k_ln = thetaZ[n_base]
            return theta_base, k_ln
        return theta_base, None

    @jax.jit
    def zhat_from_thetaZ(x14: jnp.ndarray, thetaZ: jnp.ndarray) -> jnp.ndarray:
        thetaZ = jnp.clip(thetaZ.reshape(-1), -DTYPE(CLIP_RAW_Z), DTYPE(CLIP_RAW_Z))
        phi = additive_features_from_x(x14)
        theta_base, k_ln = unpack_thetaZ(thetaZ)

        z = jnp.dot(phi, theta_base)

        if use_ln_feature:
            lnratio = ln_ce_ratio_from_x(x14)
            z = z + k_ln * lnratio

        return z

    def make_thetaZ0_from_voltage(Y_np: np.ndarray) -> np.ndarray:
        thetaZ0 = np.zeros(n_thetaZ, dtype=np.float64)
        thetaZ0[0] = float(np.mean(Y_np[:, 0]))
        return thetaZ0

    meta = dict(
        deg=deg,
        n_thetaZ=n_thetaZ,
        use_ln_feature=use_ln_feature,
        xp_ref=xp_ref,
        xn_ref=xn_ref,
        xp_scale=xp_scale,
        xn_scale=xn_scale,
        structure="c0 + sum a_k*dxp^k + sum b_k*dxn^k [+ k_ln ln(ceR/ceL)]",
    )
    return make_thetaZ0_from_voltage, zhat_from_thetaZ, meta


# %% ======================================================
# CELL 9 — Stage 2: fit surrogate + R0, A/B fixed nominal
# =========================================================
print("\n==============================")
print("Stage 2: Fit surrogate + R0 with A and B fixed to nominal proxy dynamics")
print("==============================")

if float(np.median(np.abs(U_np[:, 0]))) < MIN_MED_ABS_I_A:
    raise RuntimeError(
        "Median |I| is too small for useful identification. "
        "Likely wrong current units. Try FORCE_UNITS='A'."
    )

if (xp_rng < 1e-7) and (xn_rng < 1e-7):
    raise RuntimeError(
        "xp/xn proxy motion is essentially zero. "
        "Do not fit Stage 2 yet. Fix current units or choose a better cycle."
    )

xp_ref = float(np.mean(xp_sig))
xn_ref = float(np.mean(xn_sig))

xp_scale = float(max(np.std(xp_sig), 1e-6))
xn_scale = float(max(np.std(xn_sig), 1e-6))

print("\n[DEBUG] Center-scale settings")
print("xp_ref   =", xp_ref)
print("xn_ref   =", xn_ref)
print("xp_scale =", xp_scale)
print("xn_scale =", xn_scale)

make_thetaZ0, zhat_from_thetaZ_stage2, metaZ = make_additive_poly_surrogate_fns(
    CFG,
    POLY_DEG,
    xp_ref=xp_ref,
    xn_ref=xn_ref,
    xp_scale=xp_scale,
    xn_scale=xn_scale,
    use_ln_feature=USE_LN_FEATURE,
)

dxp_dbg = (xp_sig - xp_ref) / xp_scale
dxn_dbg = (xn_sig - xn_ref) / xn_scale

Phi_cols = [np.ones_like(dxp_dbg)]
for k in range(1, POLY_DEG + 1):
    Phi_cols.append(dxp_dbg ** k)
for k in range(1, POLY_DEG + 1):
    Phi_cols.append(dxn_dbg ** k)

if USE_LN_FEATURE:
    Phi_cols.append(np.log(np.maximum(ceR_sig / np.maximum(ceL_sig, 1e-12), 1e-12)))

Phi = np.column_stack(Phi_cols)

print("\n[DEBUG] Stage-2 feature matrix diagnostics")
print("Phi shape:", Phi.shape)
col_std = np.std(Phi, axis=0)
for i, s in enumerate(col_std):
    print(f"  feature[{i}] std = {s:.6e}")

try:
    cond_phi = np.linalg.cond(Phi)
    print("Feature matrix cond(Phi):", cond_phi)
except Exception as e:
    print("Could not compute cond(Phi):", e)

thetaZ0 = make_thetaZ0(Y_np)
rawR0_0 = raw_from_pos(0.05, floor=1e-12)

params_stage2_init = [
    thetaZ0.astype(np.float64),
    np.array([rawR0_0], dtype=np.float64),
]
nx_stage2 = 14
x0_stage2 = X_proxy[0].astype(np.float64)

A_nom = jnp.array(A_nom_np, dtype=DTYPE)
B_nom = jnp.array(B_nom_np, dtype=DTYPE)

@jax.jit
def state_fcn_stage2(x, u, t, params):
    I = u[0]
    return A_nom @ x + (B_nom[:, 0] * I)

@jax.jit
def output_fcn_stage2(x, u, t, params):
    thetaZ, rawR0 = params
    I = u[0]
    Z = zhat_from_thetaZ_stage2(x, thetaZ)
    R0 = pos(rawR0[0], 1e-12)
    Vhat = Z - DTYPE(CFG.N_series) * I * R0
    return jnp.array([Vhat], dtype=DTYPE)

V_mean_baseline = np.full_like(Y_np[:, 0], np.mean(Y_np[:, 0]))
report_fit("Baseline mean-voltage", Y_np, V_mean_baseline.reshape(-1, 1))

coef_lin = np.polyfit(t_np, Y_np[:, 0], 1)
V_line_baseline = np.polyval(coef_lin, t_np)
report_fit("Baseline linear-in-time", Y_np, V_line_baseline.reshape(-1, 1))

cmp_ls = compare_ls_solvers(Phi, Y_np[:, 0], ridge=1e-12, label="Static LS")
coef_ls_manual = cmp_ls["beta_qr"]
coef_ls_np = cmp_ls["beta_lstsq"]

Y_ls_manual = Phi @ coef_ls_manual
Y_ls_np = Phi @ coef_ls_np

print("\n[DEBUG] Static LS on surrogate features (manual QR)")
report_fit("Static LS feature fit (manual QR)", Y_np, Y_ls_manual.reshape(-1, 1))

print("\n[DEBUG] Static LS on surrogate features (np.linalg.lstsq)")
report_fit("Static LS feature fit (np.linalg.lstsq)", Y_np, Y_ls_np.reshape(-1, 1))

Phi_plus_R = np.column_stack([Phi, -U_np[:, 0]])

cmp_lsr = compare_ls_solvers(Phi_plus_R, Y_np[:, 0], ridge=1e-12, label="Static LS+R")
coef_ls_r_manual = cmp_lsr["beta_qr"]
coef_ls_r_np = cmp_lsr["beta_lstsq"]

Y_ls_r_manual = Phi_plus_R @ coef_ls_r_manual
Y_ls_r_np = Phi_plus_R @ coef_ls_r_np

print("\n[DEBUG] Static LS+R (manual QR)")
report_fit("Static LS+R fit (manual QR)", Y_np, Y_ls_r_manual.reshape(-1, 1))

print("\n[DEBUG] Static LS+R (np.linalg.lstsq)")
report_fit("Static LS+R fit (np.linalg.lstsq)", Y_np, Y_ls_r_np.reshape(-1, 1))

R0_ls_manual = float(coef_ls_r_manual[-1])
R0_ls_np = float(coef_ls_r_np[-1])

print("\n[DEBUG] LS+R inferred ohmic coefficient")
print(f"  R0_manual_QR   = {R0_ls_manual:.12g}")
print(f"  R0_np_lstsq    = {R0_ls_np:.12g}")
print(f"  |difference|   = {abs(R0_ls_manual - R0_ls_np):.6e}")

plot_voltage_with_baselines(
    t_np,
    Y_np[:, 0],
    {
        "Linear baseline": V_line_baseline,
        "Static LS (manual)": Y_ls_manual,
        "Static LS+R (manual)": Y_ls_r_manual,
    },
    title="Stage-2 baseline and manual LS comparisons (physics-only)"
)

thetaZ_min = -10.0 * np.ones_like(thetaZ0, dtype=np.float64)
thetaZ_max =  10.0 * np.ones_like(thetaZ0, dtype=np.float64)

thetaZ_min[0] = float(np.min(Y_np[:, 0]) - 0.5)
thetaZ_max[0] = float(np.max(Y_np[:, 0]) + 0.5)

rawR0_min = np.array([raw_from_pos(1e-4, floor=1e-12)], dtype=np.float64)
rawR0_max = np.array([raw_from_pos(5.0,  floor=1e-12)], dtype=np.float64)

model_stage2 = CTModel(nx_stage2, 1, 1, state_fcn=state_fcn_stage2, output_fcn=output_fcn_stage2)
model_stage2.init(params=params_stage2_init, x0=x0_stage2)

try:
    model_stage2.loss(rho_x0=0.0, rho_th=RHO_TH_STAGE2, train_x0=False, xsat=1e9)
except TypeError:
    model_stage2.loss(rho_x0=0.0, rho_th=RHO_TH_STAGE2)

lbfgs_epochs = int(LBFGS_EPOCHS) if USE_LBFGS else 0

model_stage2.optimization(
    adam_epochs=int(ADAM_EPOCHS_STAGE2),
    lbfgs_epochs=lbfgs_epochs,
    adam_eta=float(ADAM_ETA_STAGE2),
    params_min=[thetaZ_min, rawR0_min],
    params_max=[thetaZ_max, rawR0_max],
)

dt = float(t_np[1] - t_np[0]) if len(t_np) > 1 else 1.0
model_stage2.integration_options(
    ode_solver=diffrax.Tsit5(),
    dt0=dt / float(DT0_DIV),
    max_steps=int(MAX_STEPS),
    stepsize_controller=diffrax.PIDController(
        rtol=float(SOLVER_RTOL),
        atol=float(SOLVER_ATOL)
    ),
)

Y0_stage2, _ = model_stage2.predict(model_stage2.x0, U_np, t_np)
report_fit("Stage 2 pre-fit", Y_np, Y0_stage2)

model_stage2.fit(Y_np, U_np, t_np)

Yhat_stage2, Xhat_stage2 = model_stage2.predict(model_stage2.x0, U_np, t_np)
report_fit("Stage 2 post-fit", Y_np, Yhat_stage2)
summarize_err(Y_np, Yhat_stage2, name=f"Stage 2 (train, deg={POLY_DEG})")

plot_voltage(t_np, Y_np, Yhat_stage2, title=f"Stage 2: physics-only surrogate fit (deg={POLY_DEG})")
plot_voltage_with_baselines(
    t_np,
    Y_np[:, 0],
    {
        "Static LS (manual)": Y_ls_manual,
        "Static LS+R (manual)": Y_ls_r_manual,
        "CT Stage 2": Yhat_stage2[:, 0],
    },
    title="Measured vs manual LS vs CT Stage 2"
)
plot_residuals(t_np, Y_np[:, 0], Yhat_stage2[:, 0], title="Stage 2 residuals")

thetaZ_hat = np.asarray(model_stage2.params[0]).reshape(-1)
rawR0_hat  = float(np.asarray(model_stage2.params[1]).reshape(-1)[0])
R0_hat     = float(np.asarray(pos(jnp.array(rawR0_hat, dtype=DTYPE), 1e-12)))

print("\nStage 2 learned:")
print("  surrogate structure:", metaZ["structure"])
print(f"  POLY_DEG={metaZ['deg']}, USE_LN_FEATURE={metaZ['use_ln_feature']}")
print("  thetaZ_hat shape:", thetaZ_hat.shape, " (n_thetaZ =", metaZ["n_thetaZ"], ")")
print("  R0_hat:", R0_hat)

print("\n[DEBUG] Stage 2 diagnostics")
report_signal_span("Measured V", Y_np[:, 0])
report_signal_span("Predicted V", Yhat_stage2[:, 0])
report_signal_span("Residual", Yhat_stage2[:, 0] - Y_np[:, 0])

corr_v = float(np.corrcoef(Y_np[:, 0], Yhat_stage2[:, 0])[0, 1]) if len(Y_np) > 2 else np.nan
print("Voltage correlation(meas,pred):", corr_v)

pred_span = float(np.max(Yhat_stage2[:, 0]) - np.min(Yhat_stage2[:, 0]))
meas_span = float(np.max(Y_np[:, 0]) - np.min(Y_np[:, 0]))

if pred_span < 0.05 * meas_span:
    print("\n[HARD WARNING] Stage 2 prediction span is too small relative to measured span.")
    print("               This fit is effectively flat and should not be used to start Stage 3.")


# %% ======================================================
# CELL 10 — Degree sweep summary or optional sweep code
# =========================================================
def run_stage2_for_degree_real(
    deg: int,
    use_ln_feature: bool = False,
    adam_epochs: int = 600,
):
    xp_ref_d = float(np.mean(xp_sig))
    xn_ref_d = float(np.mean(xn_sig))
    xp_scale_d = float(max(np.std(xp_sig), 1e-6))
    xn_scale_d = float(max(np.std(xn_sig), 1e-6))

    make_thetaZ0_d, zhat_from_thetaZ_d, meta = make_additive_poly_surrogate_fns(
        CFG,
        deg,
        xp_ref=xp_ref_d,
        xn_ref=xn_ref_d,
        xp_scale=xp_scale_d,
        xn_scale=xn_scale_d,
        use_ln_feature=use_ln_feature,
    )

    dxp_d = (xp_sig - xp_ref_d) / xp_scale_d
    dxn_d = (xn_sig - xn_ref_d) / xn_scale_d

    Phi_cols_d = [np.ones_like(dxp_d)]
    for k in range(1, deg + 1):
        Phi_cols_d.append(dxp_d ** k)
    for k in range(1, deg + 1):
        Phi_cols_d.append(dxn_d ** k)

    if use_ln_feature:
        Phi_cols_d.append(np.log(np.maximum(ceR_sig / np.maximum(ceL_sig, 1e-12), 1e-12)))

    Phi_d = np.column_stack(Phi_cols_d)

    beta_ls_d = solve_ls_qr(Phi_d, Y_np[:, 0])
    Y_ls_d = Phi_d @ beta_ls_d

    Phi_plus_R_d = np.column_stack([Phi_d, -U_np[:, 0]])
    beta_lsr_d = solve_ls_qr(Phi_plus_R_d, Y_np[:, 0])
    Y_ls_r_d = Phi_plus_R_d @ beta_lsr_d

    thetaZ0_d = make_thetaZ0_d(Y_np)
    rawR0_0_d = raw_from_pos(0.05, floor=1e-12)

    params0_d = [
        thetaZ0_d.astype(np.float64),
        np.array([rawR0_0_d], dtype=np.float64),
    ]

    x0_d = X_proxy[0].astype(np.float64)

    @jax.jit
    def state_fcn_d(x, u, t, params):
        I = u[0]
        return A_nom @ x + (B_nom[:, 0] * I)

    @jax.jit
    def output_fcn_d(x, u, t, params):
        thetaZ, rawR0 = params
        I = u[0]
        Z = zhat_from_thetaZ_d(x, thetaZ)
        R0 = pos(rawR0[0], 1e-12)
        Vhat = Z - DTYPE(CFG.N_series) * I * R0
        return jnp.array([Vhat], dtype=DTYPE)

    thetaZ_min_d = -10.0 * np.ones_like(thetaZ0_d, dtype=np.float64)
    thetaZ_max_d =  10.0 * np.ones_like(thetaZ0_d, dtype=np.float64)
    thetaZ_min_d[0] = float(np.min(Y_np[:, 0]) - 0.5)
    thetaZ_max_d[0] = float(np.max(Y_np[:, 0]) + 0.5)

    rawR0_min_d = np.array([raw_from_pos(1e-4, floor=1e-12)], dtype=np.float64)
    rawR0_max_d = np.array([raw_from_pos(5.0,  floor=1e-12)], dtype=np.float64)

    m = CTModel(14, 1, 1, state_fcn=state_fcn_d, output_fcn=output_fcn_d)
    m.init(params=params0_d, x0=x0_d)

    try:
        m.loss(rho_x0=0.0, rho_th=RHO_TH_STAGE2, train_x0=False, xsat=1e9)
    except TypeError:
        m.loss(rho_x0=0.0, rho_th=RHO_TH_STAGE2)

    lbfgs_epochs = int(LBFGS_EPOCHS) if USE_LBFGS else 0

    m.optimization(
        adam_epochs=int(adam_epochs),
        lbfgs_epochs=lbfgs_epochs,
        adam_eta=float(ADAM_ETA_STAGE2),
        params_min=[thetaZ_min_d, rawR0_min_d],
        params_max=[thetaZ_max_d, rawR0_max_d],
    )

    m.integration_options(
        ode_solver=diffrax.Tsit5(),
        dt0=dt / float(DT0_DIV),
        max_steps=int(MAX_STEPS),
        stepsize_controller=diffrax.PIDController(
            rtol=float(SOLVER_RTOL),
            atol=float(SOLVER_ATOL)
        ),
    )

    m.fit(Y_np, U_np, t_np)
    Yhat_d, _ = m.predict(m.x0, U_np, t_np)

    ct_metrics = summarize_err(Y_np, Yhat_d, name=f"CT Stage 2 train (deg={deg})")
    ls_metrics = summarize_err(Y_np, Y_ls_d.reshape(-1, 1), name=f"Static LS train (deg={deg})")
    lsr_metrics = summarize_err(Y_np, Y_ls_r_d.reshape(-1, 1), name=f"Static LS+R train (deg={deg})")

    return dict(
        deg=deg,
        n_thetaZ=meta["n_thetaZ"],
        ct=ct_metrics,
        ls=ls_metrics,
        lsr=lsr_metrics,
        structure=meta["structure"],
    )


if RUN_DEGREE_SWEEP:
    print("\n" + "=" * 90)
    print("RUNNING DEGREE SWEEP")
    print("=" * 90)

    sweep = []
    for d in DEGREE_SWEEP:
        print("\n" + "#" * 80)
        print(f"SWEEP DEGREE = {d}")
        print("#" * 80)
        sweep.append(
            run_stage2_for_degree_real(
                deg=d,
                use_ln_feature=USE_LN_FEATURE,
                adam_epochs=600,
            )
        )

    print("\n" + "=" * 90)
    print("SWEEP SUMMARY (real-data Stage 2, physics-only)")
    print("=" * 90)
    for r in sweep:
        print(
            f"deg={r['deg']:2d} | "
            f"CT RMSE={r['ct']['rmse']:.3e} | "
            f"LS RMSE={r['ls']['rmse']:.3e} | "
            f"LS+R RMSE={r['lsr']['rmse']:.3e} | "
            f"CT MAX={r['ct']['max_abs']:.3e}"
        )
else:
    print("\n[INFO] RUN_DEGREE_SWEEP=False, skipping Cell 10.")
    print("[INFO] Based on your previous sweep:")
    print("       - practical sweet-spot range: degree 14 to 17")
    print("       - chosen operating degree now:", POLY_DEG)
    print("       - now we will judge it on several/all cycles, not only one cycle")


# %% ======================================================
# CELL 11 — Stage 3a: fit A,B only, surrogate frozen from Stage 2
# =========================================================
if not RUN_STAGE3:
    print("\n[INFO] RUN_STAGE3=False, skipping Stage 3a.")
else:
    print("\n==============================")
    print("Stage 3a: Fit A,B with Stage-2 surrogate frozen")
    print("==============================")

    thetaA_nom_init = thetaA_nom_np.copy()
    thetaB_nom_init = thetaB_nom_np.copy()

    rawA_init = np.log(np.maximum(thetaA_nom_init, 1e-12))
    rawB_init = thetaB_nom_init.copy()
    rawAB_init = np.concatenate([rawA_init, rawB_init]).astype(np.float64)

    nx_stage3a = 14
    x0_stage3a = X_proxy[0].astype(np.float64)

    @jax.jit
    def unpack_ab(raw_ab: jnp.ndarray):
        raw_ab = raw_ab.reshape(-1)
        rawA = jnp.clip(raw_ab[0:7], -DTYPE(CLIP_RAW_A), DTYPE(CLIP_RAW_A))
        rawB = jnp.clip(raw_ab[7:11], -DTYPE(CLIP_RAW_B), DTYPE(CLIP_RAW_B))
        thetaA = jnp.exp(rawA)
        thetaB = rawB
        return thetaA, thetaB

    @jax.jit
    def state_fcn_stage3a(x, u, t, params):
        I = u[0]
        (raw_ab,) = params
        thetaA, thetaB = unpack_ab(raw_ab)
        A = build_A_from_thetaA(thetaA)
        B = build_B_from_thetaB(thetaB)
        return A @ x + (B[:, 0] * I)

    @jax.jit
    def output_fcn_stage3a(x, u, t, params):
        I = u[0]
        Z = zhat_from_thetaZ_stage2(x, jnp.array(thetaZ_hat, dtype=DTYPE))
        R0 = pos(jnp.array(rawR0_hat, dtype=DTYPE), 1e-12)
        Vhat = Z - DTYPE(CFG.N_series) * I * R0
        return jnp.array([Vhat], dtype=DTYPE)

    model_stage3a = CTModel(14, 1, 1, state_fcn=state_fcn_stage3a, output_fcn=output_fcn_stage3a)
    model_stage3a.init(params=[rawAB_init], x0=x0_stage3a)

    try:
        model_stage3a.loss(rho_x0=0.0, rho_th=RHO_TH_STAGE3A, train_x0=False, xsat=1e9)
    except TypeError:
        model_stage3a.loss(rho_x0=0.0, rho_th=RHO_TH_STAGE3A)

    model_stage3a.optimization(
        adam_epochs=int(ADAM_EPOCHS_STAGE3A),
        lbfgs_epochs=0,
        adam_eta=float(ADAM_ETA_STAGE3A),
    )

    model_stage3a.integration_options(
        ode_solver=diffrax.Tsit5(),
        dt0=dt / float(DT0_DIV),
        max_steps=int(MAX_STEPS),
        stepsize_controller=diffrax.PIDController(rtol=float(SOLVER_RTOL), atol=float(SOLVER_ATOL)),
    )

    Y0_stage3a, _ = model_stage3a.predict(model_stage3a.x0, U_np, t_np)
    report_fit("Stage 3a pre-fit", Y_np, Y0_stage3a)

    model_stage3a.fit(Y_np, U_np, t_np)

    Yhat_stage3a, _ = model_stage3a.predict(model_stage3a.x0, U_np, t_np)
    report_fit("Stage 3a post-fit", Y_np, Yhat_stage3a)
    summarize_err(Y_np, Yhat_stage3a, name=f"Stage 3a (train, deg={POLY_DEG})")
    plot_voltage(t_np, Y_np, Yhat_stage3a, title=f"Stage 3a fit (deg={POLY_DEG})")

    rawAB_hat = np.asarray(model_stage3a.params[0]).reshape(-1)
    rawA_hat_stage3a = rawAB_hat[0:7].copy()
    rawB_hat_stage3a = rawAB_hat[7:11].copy()


# %% ======================================================
# CELL 12 — Stage 3b: unfreeze everything and refine full model
# =========================================================
if not RUN_STAGE3:
    print("\n[INFO] RUN_STAGE3=False, skipping Stage 3b.")
else:
    print("\n==============================")
    print("Stage 3b: Unfreeze everything and refine FULL model")
    print("==============================")

    N_THETAZ = int(thetaZ_hat.shape[0])

    raw_init_full = np.concatenate([
        rawA_hat_stage3a.astype(np.float64),
        rawB_hat_stage3a.astype(np.float64),
        thetaZ_hat.astype(np.float64),
        np.array([rawR0_hat], dtype=np.float64),
    ]).astype(np.float64)

    def unpack_full(raw_theta: jnp.ndarray):
        raw_theta = raw_theta.reshape(-1)
        rawA   = jnp.clip(raw_theta[0:7], -DTYPE(CLIP_RAW_A), DTYPE(CLIP_RAW_A))
        rawB   = jnp.clip(raw_theta[7:11], -DTYPE(CLIP_RAW_B), DTYPE(CLIP_RAW_B))
        thetaZ = jnp.clip(raw_theta[11:11+N_THETAZ], -DTYPE(CLIP_RAW_Z), DTYPE(CLIP_RAW_Z))
        rawR0  = raw_theta[11+N_THETAZ]
        thetaA = jnp.exp(rawA)
        thetaB = rawB
        return thetaA, thetaB, thetaZ, rawR0

    @jax.jit
    def state_fcn_stage3b(x, u, t, params):
        I = u[0]
        (raw_theta,) = params
        thetaA, thetaB, _, _ = unpack_full(raw_theta)
        A = build_A_from_thetaA(thetaA)
        B = build_B_from_thetaB(thetaB)
        return A @ x + (B[:, 0] * I)

    @jax.jit
    def output_fcn_stage3b(x, u, t, params):
        I = u[0]
        (raw_theta,) = params
        _, _, thetaZ, rawR0 = unpack_full(raw_theta)
        Z = zhat_from_thetaZ_stage2(x, thetaZ)
        R0 = pos(rawR0, 1e-12)
        Vhat = Z - DTYPE(CFG.N_series) * I * R0
        return jnp.array([Vhat], dtype=DTYPE)

    model_stage3b = CTModel(14, 1, 1, state_fcn=state_fcn_stage3b, output_fcn=output_fcn_stage3b)
    model_stage3b.init(params=[raw_init_full], x0=X_proxy[0].astype(np.float64))

    try:
        model_stage3b.loss(rho_x0=0.0, rho_th=RHO_TH_STAGE3B, train_x0=False, xsat=1e9)
    except TypeError:
        model_stage3b.loss(rho_x0=0.0, rho_th=RHO_TH_STAGE3B)

    model_stage3b.optimization(
        adam_epochs=int(ADAM_EPOCHS_STAGE3B),
        lbfgs_epochs=0,
        adam_eta=float(ADAM_ETA_STAGE3B),
    )

    model_stage3b.integration_options(
        ode_solver=diffrax.Tsit5(),
        dt0=dt / float(DT0_DIV),
        max_steps=int(MAX_STEPS),
        stepsize_controller=diffrax.PIDController(rtol=float(SOLVER_RTOL), atol=float(SOLVER_ATOL)),
    )

    Y0_stage3b, _ = model_stage3b.predict(model_stage3b.x0, U_np, t_np)
    report_fit("Stage 3b pre-fit", Y_np, Y0_stage3b)

    model_stage3b.fit(Y_np, U_np, t_np)

    Yhat_stage3b, _ = model_stage3b.predict(model_stage3b.x0, U_np, t_np)
    report_fit("Stage 3b post-fit", Y_np, Yhat_stage3b)
    summarize_err(Y_np, Yhat_stage3b, name=f"Stage 3b (train, deg={POLY_DEG})")
    plot_voltage(t_np, Y_np, Yhat_stage3b, title=f"Stage 3b fit (deg={POLY_DEG})")
    plot_residuals(t_np, Y_np[:, 0], Yhat_stage3b[:, 0], title="Stage 3b residuals")

    raw_full_hat = np.asarray(model_stage3b.params[0]).reshape(-1)
    thetaA_hat_stage3b = np.exp(np.clip(raw_full_hat[0:7], -CLIP_RAW_A, CLIP_RAW_A))
    thetaB_hat_stage3b = raw_full_hat[7:11].copy()
    thetaZ_hat_stage3b = raw_full_hat[11:11+N_THETAZ].copy()
    rawR0_hat_stage3b = float(raw_full_hat[11+N_THETAZ])
    R0_hat_stage3b = float(np.asarray(pos(jnp.array(rawR0_hat_stage3b, dtype=DTYPE), 1e-12)))


# %% ======================================================
# CELL 13 — Reusable cycle pipeline helpers
# =========================================================
def prepare_cycle_data(
    cycle_df_local: pd.DataFrame,
    i_col: str = I_COL,
    v_col: str = V_COL,
    force_units: str = FORCE_UNITS,
    v_ref: str = V_REF,
    resample: bool = RESAMPLE,
    enforce_discharge_only: bool = ENFORCE_DISCHARGE_ONLY,
    raw_discharge_sign: str = RAW_DISCHARGE_SIGN,
    tmax: float = TMAX,
    drop_first_n: int = DROP_FIRST_N,
):
    t = cycle_df_local.index.to_numpy(dtype=np.float64).reshape(-1)
    I_raw = cycle_df_local[i_col].to_numpy(dtype=np.float64).reshape(-1)
    V_raw = cycle_df_local[v_col].to_numpy(dtype=np.float64).reshape(-1)

    if str(force_units).lower() == "auto":
        I_A, _ = _guess_I_in_amps(I_raw)
    elif str(force_units).lower() == "ma":
        I_A = I_raw * 1e-3
    elif str(force_units).lower() == "a":
        I_A = I_raw
    else:
        raise ValueError("FORCE_UNITS must be 'auto'|'mA'|'A'")

    keep = np.ones_like(t, dtype=bool)
    keep[1:] = t[1:] > t[:-1]
    t, I_A, V_raw = t[keep], I_A[keep], V_raw[keep]

    t = t - t[0]

    if enforce_discharge_only:
        tol = 1e-12
        if raw_discharge_sign == "negative":
            mask = I_A < -tol
        elif raw_discharge_sign == "positive":
            mask = I_A > +tol
        else:
            raise ValueError("RAW_DISCHARGE_SIGN must be 'negative' or 'positive'")
        t, I_A, V_raw = t[mask], I_A[mask], V_raw[mask]
        if len(t) < 5:
            raise RuntimeError("After discharge-only filter, not enough points.")
        t = t - t[0]

    if drop_first_n > 0:
        if len(t) <= drop_first_n + 3:
            raise RuntimeError("DROP_FIRST_N is too large for this selected cycle.")
        t = t[drop_first_n:]
        I_A = I_A[drop_first_n:]
        V_raw = V_raw[drop_first_n:]
        t = t - t[0]

    if v_ref == "none":
        V = V_raw
    elif v_ref == "first":
        V = V_raw - float(V_raw[0])
    elif v_ref == "mean":
        V = V_raw - float(np.mean(V_raw))
    else:
        raise ValueError("V_REF must be none|first|mean")

    t_np_local = t.reshape(-1)
    U_np_local = I_A.reshape(-1, 1)
    Y_np_local = V.reshape(-1, 1)

    if resample and len(t_np_local) > 2:
        dt_med = float(np.median(np.diff(t_np_local)))
        if not (np.isfinite(dt_med) and dt_med > 0):
            raise RuntimeError("Bad dt_med; cannot resample.")
        t_np_local, U_np_local, Y_np_local = resample_uniform(t_np_local, U_np_local, Y_np_local, dt=dt_med)

    if tmax > 0:
        m = t_np_local <= float(tmax)
        t_np_local, U_np_local, Y_np_local = t_np_local[m], U_np_local[m], Y_np_local[m]

    return t_np_local, U_np_local, Y_np_local


def build_proxy_signals(
    t_np_local: np.ndarray,
    U_np_local: np.ndarray,
    cfg: Config,
    xn0: float = XN0,
    xp0: float = XP0,
    ce0_dev: float = CE0_DEV,
):
    Sx_local, A_nom_local, B_nom_local = assemble_system(cfg)
    x0_nom_local = make_x0(cfg, theta_n0=xn0, theta_p0=xp0, ce0=ce0_dev)

    resp_local = ct.forced_response(Sx_local, T=t_np_local, U=U_np_local[:, 0], X0=x0_nom_local)
    X_proxy_local = np.asarray(resp_local.states).T

    xp_sig_local = (X_proxy_local[:, IDX["cp_surf"]] / cfg.csp_max).astype(np.float64)
    xn_sig_local = (X_proxy_local[:, IDX["cn_surf"]] / cfg.csn_max).astype(np.float64)

    ceL_raw_local = X_proxy_local[:, IDX["ce_left"]].astype(np.float64)
    ceR_raw_local = X_proxy_local[:, IDX["ce_right"]].astype(np.float64)

    if cfg.ce_is_deviation:
        ceL_sig_local = cfg.ce0 + ceL_raw_local
        ceR_sig_local = cfg.ce0 + ceR_raw_local
    else:
        ceL_sig_local = ceL_raw_local
        ceR_sig_local = ceR_raw_local

    ln_ce_ratio_local = np.log(np.maximum(ceR_sig_local / np.maximum(ceL_sig_local, 1e-12), 1e-12))

    info = dict(
        xp_rng=float(xp_sig_local.max() - xp_sig_local.min()),
        xn_rng=float(xn_sig_local.max() - xn_sig_local.min()),
        ceL_rng=float(ceL_sig_local.max() - ceL_sig_local.min()),
        ceR_rng=float(ceR_sig_local.max() - ceR_sig_local.min()),
    )

    return X_proxy_local, A_nom_local, B_nom_local, xp_sig_local, xn_sig_local, ceL_sig_local, ceR_sig_local, ln_ce_ratio_local, info


def build_phi_from_proxy(
    xp_sig_local: np.ndarray,
    xn_sig_local: np.ndarray,
    ceL_sig_local: np.ndarray,
    ceR_sig_local: np.ndarray,
    deg: int,
    xp_ref_local: float,
    xn_ref_local: float,
    xp_scale_local: float,
    xn_scale_local: float,
    use_ln_feature: bool,
):
    dxp_local = (xp_sig_local - xp_ref_local) / xp_scale_local
    dxn_local = (xn_sig_local - xn_ref_local) / xn_scale_local

    cols = [np.ones_like(dxp_local)]
    for k in range(1, deg + 1):
        cols.append(dxp_local ** k)
    for k in range(1, deg + 1):
        cols.append(dxn_local ** k)

    if use_ln_feature:
        cols.append(np.log(np.maximum(ceR_sig_local / np.maximum(ceL_sig_local, 1e-12), 1e-12)))

    Phi_local = np.column_stack(cols)
    return Phi_local, dxp_local, dxn_local


def fit_stage2_for_cycle(
    t_np_local: np.ndarray,
    U_np_local: np.ndarray,
    Y_np_local: np.ndarray,
    X_proxy_local: np.ndarray,
    A_nom_local: np.ndarray,
    B_nom_local: np.ndarray,
    xp_sig_local: np.ndarray,
    xn_sig_local: np.ndarray,
    ceL_sig_local: np.ndarray,
    ceR_sig_local: np.ndarray,
    cfg: Config,
    deg: int = POLY_DEG,
    use_ln_feature: bool = USE_LN_FEATURE,
):
    xp_ref_local = float(np.mean(xp_sig_local))
    xn_ref_local = float(np.mean(xn_sig_local))
    xp_scale_local = float(max(np.std(xp_sig_local), 1e-6))
    xn_scale_local = float(max(np.std(xn_sig_local), 1e-6))

    make_thetaZ0_local, zhat_from_thetaZ_local, meta_local = make_additive_poly_surrogate_fns(
        cfg,
        deg,
        xp_ref=xp_ref_local,
        xn_ref=xn_ref_local,
        xp_scale=xp_scale_local,
        xn_scale=xn_scale_local,
        use_ln_feature=use_ln_feature,
    )

    Phi_local, dxp_local, dxn_local = build_phi_from_proxy(
        xp_sig_local, xn_sig_local, ceL_sig_local, ceR_sig_local,
        deg=deg,
        xp_ref_local=xp_ref_local,
        xn_ref_local=xn_ref_local,
        xp_scale_local=xp_scale_local,
        xn_scale_local=xn_scale_local,
        use_ln_feature=use_ln_feature,
    )

    thetaZ0_local = make_thetaZ0_local(Y_np_local)
    rawR0_0_local = raw_from_pos(0.05, floor=1e-12)

    params_stage2_init_local = [
        thetaZ0_local.astype(np.float64),
        np.array([rawR0_0_local], dtype=np.float64),
    ]

    A_nom_j = jnp.array(A_nom_local, dtype=DTYPE)
    B_nom_j = jnp.array(B_nom_local, dtype=DTYPE)

    @jax.jit
    def state_fcn_local(x, u, t, params):
        I = u[0]
        return A_nom_j @ x + (B_nom_j[:, 0] * I)

    @jax.jit
    def output_fcn_local(x, u, t, params):
        thetaZ, rawR0 = params
        I = u[0]
        Z = zhat_from_thetaZ_local(x, thetaZ)
        R0 = pos(rawR0[0], 1e-12)
        Vhat = Z - DTYPE(cfg.N_series) * I * R0
        return jnp.array([Vhat], dtype=DTYPE)

    thetaZ_min_local = -10.0 * np.ones_like(thetaZ0_local, dtype=np.float64)
    thetaZ_max_local =  10.0 * np.ones_like(thetaZ0_local, dtype=np.float64)
    thetaZ_min_local[0] = float(np.min(Y_np_local[:, 0]) - 0.5)
    thetaZ_max_local[0] = float(np.max(Y_np_local[:, 0]) + 0.5)

    rawR0_min_local = np.array([raw_from_pos(1e-4, floor=1e-12)], dtype=np.float64)
    rawR0_max_local = np.array([raw_from_pos(5.0,  floor=1e-12)], dtype=np.float64)

    model_local = CTModel(14, 1, 1, state_fcn=state_fcn_local, output_fcn=output_fcn_local)
    model_local.init(params=params_stage2_init_local, x0=X_proxy_local[0].astype(np.float64))

    try:
        model_local.loss(rho_x0=0.0, rho_th=RHO_TH_STAGE2, train_x0=False, xsat=1e9)
    except TypeError:
        model_local.loss(rho_x0=0.0, rho_th=RHO_TH_STAGE2)

    lbfgs_epochs_local = int(LBFGS_EPOCHS) if USE_LBFGS else 0

    model_local.optimization(
        adam_epochs=int(ADAM_EPOCHS_STAGE2),
        lbfgs_epochs=lbfgs_epochs_local,
        adam_eta=float(ADAM_ETA_STAGE2),
        params_min=[thetaZ_min_local, rawR0_min_local],
        params_max=[thetaZ_max_local, rawR0_max_local],
    )

    dt_local = float(t_np_local[1] - t_np_local[0]) if len(t_np_local) > 1 else 1.0
    model_local.integration_options(
        ode_solver=diffrax.Tsit5(),
        dt0=dt_local / float(DT0_DIV),
        max_steps=int(MAX_STEPS),
        stepsize_controller=diffrax.PIDController(
            rtol=float(SOLVER_RTOL),
            atol=float(SOLVER_ATOL)
        ),
    )

    model_local.fit(Y_np_local, U_np_local, t_np_local)
    Yhat_local, Xhat_local = model_local.predict(model_local.x0, U_np_local, t_np_local)

    thetaZ_hat_local = np.asarray(model_local.params[0]).reshape(-1)
    rawR0_hat_local  = float(np.asarray(model_local.params[1]).reshape(-1)[0])
    R0_hat_local     = float(np.asarray(pos(jnp.array(rawR0_hat_local, dtype=DTYPE), 1e-12)))

    ls_beta_local = solve_ls_qr(Phi_local, Y_np_local[:, 0])
    Y_ls_local = Phi_local @ ls_beta_local

    Phi_plus_R_local = np.column_stack([Phi_local, -U_np_local[:, 0]])
    lsr_beta_local = solve_ls_qr(Phi_plus_R_local, Y_np_local[:, 0])
    Y_lsr_local = Phi_plus_R_local @ lsr_beta_local

    return dict(
        model=model_local,
        Yhat=Yhat_local,
        Xhat=Xhat_local,
        thetaZ_hat=thetaZ_hat_local,
        rawR0_hat=rawR0_hat_local,
        R0_hat=R0_hat_local,
        Y_ls=Y_ls_local.reshape(-1, 1),
        Y_lsr=Y_lsr_local.reshape(-1, 1),
        metaZ=meta_local,
        xp_ref=xp_ref_local,
        xn_ref=xn_ref_local,
        xp_scale=xp_scale_local,
        xn_scale=xn_scale_local,
        Phi=Phi_local,
        dxp=dxp_local,
        dxn=dxn_local,
        zhat_from_thetaZ=zhat_from_thetaZ_local,
        fit=report_fit("Stage 2 reusable fit", Y_np_local, Yhat_local),
    )


def fit_stage3_for_cycle(
    t_np_local: np.ndarray,
    U_np_local: np.ndarray,
    Y_np_local: np.ndarray,
    X_proxy_local: np.ndarray,
    stage2_out: dict,
    cfg: Config,
    run_stage3: bool = RUN_STAGE3,
):
    out = dict(stage3a=None, stage3b=None)

    if not run_stage3:
        return out

    thetaZ_hat_local = stage2_out["thetaZ_hat"]
    rawR0_hat_local = stage2_out["rawR0_hat"]
    zhat_from_thetaZ_local = stage2_out["zhat_from_thetaZ"]

    thetaA_nom_init = thetaA_nom_np.copy()
    thetaB_nom_init = thetaB_nom_np.copy()

    rawA_init = np.log(np.maximum(thetaA_nom_init, 1e-12))
    rawB_init = thetaB_nom_init.copy()
    rawAB_init = np.concatenate([rawA_init, rawB_init]).astype(np.float64)

    @jax.jit
    def unpack_ab(raw_ab: jnp.ndarray):
        raw_ab = raw_ab.reshape(-1)
        rawA = jnp.clip(raw_ab[0:7], -DTYPE(CLIP_RAW_A), DTYPE(CLIP_RAW_A))
        rawB = jnp.clip(raw_ab[7:11], -DTYPE(CLIP_RAW_B), DTYPE(CLIP_RAW_B))
        thetaA = jnp.exp(rawA)
        thetaB = rawB
        return thetaA, thetaB

    @jax.jit
    def state_fcn_stage3a_local(x, u, t, params):
        I = u[0]
        (raw_ab,) = params
        thetaA, thetaB = unpack_ab(raw_ab)
        A = build_A_from_thetaA(thetaA)
        B = build_B_from_thetaB(thetaB)
        return A @ x + (B[:, 0] * I)

    @jax.jit
    def output_fcn_stage3a_local(x, u, t, params):
        I = u[0]
        Z = zhat_from_thetaZ_local(x, jnp.array(thetaZ_hat_local, dtype=DTYPE))
        R0 = pos(jnp.array(rawR0_hat_local, dtype=DTYPE), 1e-12)
        Vhat = Z - DTYPE(cfg.N_series) * I * R0
        return jnp.array([Vhat], dtype=DTYPE)

    model_stage3a_local = CTModel(14, 1, 1, state_fcn=state_fcn_stage3a_local, output_fcn=output_fcn_stage3a_local)
    model_stage3a_local.init(params=[rawAB_init], x0=X_proxy_local[0].astype(np.float64))

    try:
        model_stage3a_local.loss(rho_x0=0.0, rho_th=RHO_TH_STAGE3A, train_x0=False, xsat=1e9)
    except TypeError:
        model_stage3a_local.loss(rho_x0=0.0, rho_th=RHO_TH_STAGE3A)

    model_stage3a_local.optimization(
        adam_epochs=int(ADAM_EPOCHS_STAGE3A),
        lbfgs_epochs=0,
        adam_eta=float(ADAM_ETA_STAGE3A),
    )

    dt_local = float(t_np_local[1] - t_np_local[0]) if len(t_np_local) > 1 else 1.0
    model_stage3a_local.integration_options(
        ode_solver=diffrax.Tsit5(),
        dt0=dt_local / float(DT0_DIV),
        max_steps=int(MAX_STEPS),
        stepsize_controller=diffrax.PIDController(rtol=float(SOLVER_RTOL), atol=float(SOLVER_ATOL)),
    )

    model_stage3a_local.fit(Y_np_local, U_np_local, t_np_local)
    Yhat_stage3a_local, _ = model_stage3a_local.predict(model_stage3a_local.x0, U_np_local, t_np_local)

    rawAB_hat_local = np.asarray(model_stage3a_local.params[0]).reshape(-1)
    rawA_hat_local = rawAB_hat_local[0:7].copy()
    rawB_hat_local = rawAB_hat_local[7:11].copy()

    out["stage3a"] = dict(
        model=model_stage3a_local,
        Yhat=Yhat_stage3a_local,
        rawAB_hat=rawAB_hat_local,
        rawA_hat=rawA_hat_local,
        rawB_hat=rawB_hat_local,
        fit=report_fit("Stage 3a reusable fit", Y_np_local, Yhat_stage3a_local),
    )

    N_THETAZ_LOCAL = int(thetaZ_hat_local.shape[0])

    raw_init_full_local = np.concatenate([
        rawA_hat_local.astype(np.float64),
        rawB_hat_local.astype(np.float64),
        thetaZ_hat_local.astype(np.float64),
        np.array([rawR0_hat_local], dtype=np.float64),
    ]).astype(np.float64)

    def unpack_full(raw_theta: jnp.ndarray):
        raw_theta = raw_theta.reshape(-1)
        rawA   = jnp.clip(raw_theta[0:7], -DTYPE(CLIP_RAW_A), DTYPE(CLIP_RAW_A))
        rawB   = jnp.clip(raw_theta[7:11], -DTYPE(CLIP_RAW_B), DTYPE(CLIP_RAW_B))
        thetaZ = jnp.clip(raw_theta[11:11+N_THETAZ_LOCAL], -DTYPE(CLIP_RAW_Z), DTYPE(CLIP_RAW_Z))
        rawR0  = raw_theta[11+N_THETAZ_LOCAL]
        thetaA = jnp.exp(rawA)
        thetaB = rawB
        return thetaA, thetaB, thetaZ, rawR0

    @jax.jit
    def state_fcn_stage3b_local(x, u, t, params):
        I = u[0]
        (raw_theta,) = params
        thetaA, thetaB, _, _ = unpack_full(raw_theta)
        A = build_A_from_thetaA(thetaA)
        B = build_B_from_thetaB(thetaB)
        return A @ x + (B[:, 0] * I)

    @jax.jit
    def output_fcn_stage3b_local(x, u, t, params):
        I = u[0]
        (raw_theta,) = params
        _, _, thetaZ, rawR0 = unpack_full(raw_theta)
        Z = zhat_from_thetaZ_local(x, thetaZ)
        R0 = pos(rawR0, 1e-12)
        Vhat = Z - DTYPE(cfg.N_series) * I * R0
        return jnp.array([Vhat], dtype=DTYPE)

    model_stage3b_local = CTModel(14, 1, 1, state_fcn=state_fcn_stage3b_local, output_fcn=output_fcn_stage3b_local)
    model_stage3b_local.init(params=[raw_init_full_local], x0=X_proxy_local[0].astype(np.float64))

    try:
        model_stage3b_local.loss(rho_x0=0.0, rho_th=RHO_TH_STAGE3B, train_x0=False, xsat=1e9)
    except TypeError:
        model_stage3b_local.loss(rho_x0=0.0, rho_th=RHO_TH_STAGE3B)

    model_stage3b_local.optimization(
        adam_epochs=int(ADAM_EPOCHS_STAGE3B),
        lbfgs_epochs=0,
        adam_eta=float(ADAM_ETA_STAGE3B),
    )

    model_stage3b_local.integration_options(
        ode_solver=diffrax.Tsit5(),
        dt0=dt_local / float(DT0_DIV),
        max_steps=int(MAX_STEPS),
        stepsize_controller=diffrax.PIDController(rtol=float(SOLVER_RTOL), atol=float(SOLVER_ATOL)),
    )

    model_stage3b_local.fit(Y_np_local, U_np_local, t_np_local)
    Yhat_stage3b_local, _ = model_stage3b_local.predict(model_stage3b_local.x0, U_np_local, t_np_local)

    raw_full_hat_local = np.asarray(model_stage3b_local.params[0]).reshape(-1)
    thetaA_hat_local = np.exp(np.clip(raw_full_hat_local[0:7], -CLIP_RAW_A, CLIP_RAW_A))
    thetaB_hat_local = raw_full_hat_local[7:11].copy()
    thetaZ_hat_stage3b_local = raw_full_hat_local[11:11+N_THETAZ_LOCAL].copy()
    rawR0_hat_stage3b_local = float(raw_full_hat_local[11+N_THETAZ_LOCAL])
    R0_hat_stage3b_local = float(np.asarray(pos(jnp.array(rawR0_hat_stage3b_local, dtype=DTYPE), 1e-12)))

    out["stage3b"] = dict(
        model=model_stage3b_local,
        Yhat=Yhat_stage3b_local,
        raw_full_hat=raw_full_hat_local,
        thetaA_hat=thetaA_hat_local,
        thetaB_hat=thetaB_hat_local,
        thetaZ_hat=thetaZ_hat_stage3b_local,
        rawR0_hat=rawR0_hat_stage3b_local,
        R0_hat=R0_hat_stage3b_local,
        fit=report_fit("Stage 3b reusable fit", Y_np_local, Yhat_stage3b_local),
    )

    return out


def run_full_cycle_pipeline(
    cycle_df_local: pd.DataFrame,
    cycle_idx: int,
    deg: int = POLY_DEG,
    run_stage3: bool = RUN_STAGE3,
    verbose: bool = False,
):
    try:
        t_local, U_local, Y_local = prepare_cycle_data(cycle_df_local)

        med_abs_I_local = float(np.median(np.abs(U_local[:, 0])))
        if med_abs_I_local < float(MIN_MED_ABS_I_A):
            return dict(
                ok=False,
                cycle_idx=cycle_idx,
                reason=f"median current too small: {med_abs_I_local:.6g} A"
            )

        proxy = build_proxy_signals(
            t_local, U_local, cfg=CFG,
            xn0=XN0, xp0=XP0, ce0_dev=CE0_DEV
        )
        (
            X_proxy_local, A_nom_local, B_nom_local,
            xp_sig_local, xn_sig_local,
            ceL_sig_local, ceR_sig_local,
            ln_ce_ratio_local, proxy_info_local
        ) = proxy

        if (proxy_info_local["xp_rng"] < 1e-7) and (proxy_info_local["xn_rng"] < 1e-7):
            return dict(
                ok=False,
                cycle_idx=cycle_idx,
                reason="xp/xn proxy motion essentially zero"
            )

        stage2_out = fit_stage2_for_cycle(
            t_local, U_local, Y_local,
            X_proxy_local, A_nom_local, B_nom_local,
            xp_sig_local, xn_sig_local, ceL_sig_local, ceR_sig_local,
            cfg=CFG, deg=deg, use_ln_feature=USE_LN_FEATURE,
        )

        stage3_out = fit_stage3_for_cycle(
            t_local, U_local, Y_local,
            X_proxy_local,
            stage2_out,
            cfg=CFG,
            run_stage3=run_stage3,
        )

        res = dict(
            ok=True,
            cycle_idx=cycle_idx,
            deg=deg,
            t=t_local,
            U=U_local,
            Y=Y_local,
            X_proxy=X_proxy_local,
            xp_sig=xp_sig_local,
            xn_sig=xn_sig_local,
            ceL_sig=ceL_sig_local,
            ceR_sig=ceR_sig_local,
            ln_ce_ratio=ln_ce_ratio_local,
            proxy_info=proxy_info_local,
            stage2=stage2_out,
            stage3=stage3_out,
        )

        if verbose:
            print(f"[cycle {cycle_idx}] completed successfully")

        return res

    except Exception as e:
        return dict(
            ok=False,
            cycle_idx=cycle_idx,
            deg=deg,
            reason=str(e),
        )


def evaluate_stage2_transfer(
    target_cycle_df: pd.DataFrame,
    fitted_stage2: dict,
    source_deg: int,
    source_cycle_idx: int,
):
    t_local, U_local, Y_local = prepare_cycle_data(target_cycle_df)

    proxy = build_proxy_signals(
        t_local, U_local, cfg=CFG,
        xn0=XN0, xp0=XP0, ce0_dev=CE0_DEV
    )
    (
        X_proxy_local, A_nom_local, B_nom_local,
        xp_sig_local, xn_sig_local,
        ceL_sig_local, ceR_sig_local,
        ln_ce_ratio_local, proxy_info_local
    ) = proxy

    xp_ref_local = fitted_stage2["xp_ref"]
    xn_ref_local = fitted_stage2["xn_ref"]
    xp_scale_local = fitted_stage2["xp_scale"]
    xn_scale_local = fitted_stage2["xn_scale"]

    _, zhat_from_thetaZ_local, _ = make_additive_poly_surrogate_fns(
        CFG,
        source_deg,
        xp_ref=xp_ref_local,
        xn_ref=xn_ref_local,
        xp_scale=xp_scale_local,
        xn_scale=xn_scale_local,
        use_ln_feature=USE_LN_FEATURE,
    )

    thetaZ_hat_local = fitted_stage2["thetaZ_hat"]
    rawR0_hat_local = fitted_stage2["rawR0_hat"]

    A_nom_j = jnp.array(A_nom_local, dtype=DTYPE)
    B_nom_j = jnp.array(B_nom_local, dtype=DTYPE)

    @jax.jit
    def state_fcn_transfer(x, u, t, params):
        I = u[0]
        return A_nom_j @ x + (B_nom_j[:, 0] * I)

    @jax.jit
    def output_fcn_transfer(x, u, t, params):
        I = u[0]
        Z = zhat_from_thetaZ_local(x, jnp.array(thetaZ_hat_local, dtype=DTYPE))
        R0 = pos(jnp.array(rawR0_hat_local, dtype=DTYPE), 1e-12)
        Vhat = Z - DTYPE(CFG.N_series) * I * R0
        return jnp.array([Vhat], dtype=DTYPE)

    m = CTModel(14, 1, 1, state_fcn=state_fcn_transfer, output_fcn=output_fcn_transfer)
    m.init(params=[np.array([0.0])], x0=X_proxy_local[0].astype(np.float64))  # dummy params
    dt_local = float(t_local[1] - t_local[0]) if len(t_local) > 1 else 1.0
    m.integration_options(
        ode_solver=diffrax.Tsit5(),
        dt0=dt_local / float(DT0_DIV),
        max_steps=int(MAX_STEPS),
        stepsize_controller=diffrax.PIDController(
            rtol=float(SOLVER_RTOL),
            atol=float(SOLVER_ATOL)
        ),
    )

    Yhat_local, _ = m.predict(m.x0, U_local, t_local)
    fit_local = report_fit(
        f"Stage 2 transfer | source cycle={source_cycle_idx} -> target cycle",
        Y_local, Yhat_local
    )

    return dict(
        t=t_local,
        U=U_local,
        Y=Y_local,
        Yhat=Yhat_local,
        fit=fit_local,
        proxy_info=proxy_info_local,
    )


def evaluate_stage3b_transfer(
    target_cycle_df: pd.DataFrame,
    fitted_stage3b: dict,
    fitted_stage2_meta: dict,
    source_deg: int,
    source_cycle_idx: int,
):
    t_local, U_local, Y_local = prepare_cycle_data(target_cycle_df)

    proxy = build_proxy_signals(
        t_local, U_local, cfg=CFG,
        xn0=XN0, xp0=XP0, ce0_dev=CE0_DEV
    )
    (
        X_proxy_local, _, _,
        _, _, _, _, _, proxy_info_local
    ) = proxy

    xp_ref_local = fitted_stage2_meta["xp_ref"]
    xn_ref_local = fitted_stage2_meta["xn_ref"]
    xp_scale_local = fitted_stage2_meta["xp_scale"]
    xn_scale_local = fitted_stage2_meta["xn_scale"]

    _, zhat_from_thetaZ_local, _ = make_additive_poly_surrogate_fns(
        CFG,
        source_deg,
        xp_ref=xp_ref_local,
        xn_ref=xn_ref_local,
        xp_scale=xp_scale_local,
        xn_scale=xn_scale_local,
        use_ln_feature=USE_LN_FEATURE,
    )

    thetaA_hat_local = fitted_stage3b["thetaA_hat"]
    thetaB_hat_local = fitted_stage3b["thetaB_hat"]
    thetaZ_hat_local = fitted_stage3b["thetaZ_hat"]
    rawR0_hat_local = fitted_stage3b["rawR0_hat"]

    @jax.jit
    def state_fcn_transfer(x, u, t, params):
        I = u[0]
        A = build_A_from_thetaA(jnp.array(thetaA_hat_local, dtype=DTYPE))
        B = build_B_from_thetaB(jnp.array(thetaB_hat_local, dtype=DTYPE))
        return A @ x + (B[:, 0] * I)

    @jax.jit
    def output_fcn_transfer(x, u, t, params):
        I = u[0]
        Z = zhat_from_thetaZ_local(x, jnp.array(thetaZ_hat_local, dtype=DTYPE))
        R0 = pos(jnp.array(rawR0_hat_local, dtype=DTYPE), 1e-12)
        Vhat = Z - DTYPE(CFG.N_series) * I * R0
        return jnp.array([Vhat], dtype=DTYPE)

    m = CTModel(14, 1, 1, state_fcn=state_fcn_transfer, output_fcn=output_fcn_transfer)
    m.init(params=[np.array([0.0])], x0=X_proxy_local[0].astype(np.float64))
    dt_local = float(t_local[1] - t_local[0]) if len(t_local) > 1 else 1.0
    m.integration_options(
        ode_solver=diffrax.Tsit5(),
        dt0=dt_local / float(DT0_DIV),
        max_steps=int(MAX_STEPS),
        stepsize_controller=diffrax.PIDController(
            rtol=float(SOLVER_RTOL),
            atol=float(SOLVER_ATOL)
        ),
    )

    Yhat_local, _ = m.predict(m.x0, U_local, t_local)
    fit_local = report_fit(
        f"Stage 3b transfer | source cycle={source_cycle_idx} -> target cycle",
        Y_local, Yhat_local
    )

    return dict(
        t=t_local,
        U=U_local,
        Y=Y_local,
        Yhat=Yhat_local,
        fit=fit_local,
        proxy_info=proxy_info_local,
    )


def summarize_monitorable_parameters(result: dict):
    if not result.get("ok", False):
        return dict(ok=False, cycle_idx=result.get("cycle_idx", -1))

    s2 = result["stage2"]
    s3 = result["stage3"]

    out = dict(
        ok=True,
        cycle_idx=result["cycle_idx"],
        deg=result["deg"],
        stage2_rmse=float(s2["fit"]["rmse"]),
        stage2_r2=float(s2["fit"]["r2"]),
        stage2_bfr=float(s2["fit"]["bfr"]),
        R0_stage2=float(s2["R0_hat"]),
        xp_rng=float(result["proxy_info"]["xp_rng"]),
        xn_rng=float(result["proxy_info"]["xn_rng"]),
        ceL_rng=float(result["proxy_info"]["ceL_rng"]),
        ceR_rng=float(result["proxy_info"]["ceR_rng"]),
        ln_ratio_rng=float(np.max(result["ln_ce_ratio"]) - np.min(result["ln_ce_ratio"])),
        z_theta_l2=float(np.linalg.norm(s2["thetaZ_hat"])),
        z_theta_maxabs=float(np.max(np.abs(s2["thetaZ_hat"]))),
    )

    if s3["stage3a"] is not None:
        out["stage3a_rmse"] = float(s3["stage3a"]["fit"]["rmse"])
        out["stage3a_r2"]   = float(s3["stage3a"]["fit"]["r2"])
    else:
        out["stage3a_rmse"] = np.nan
        out["stage3a_r2"]   = np.nan

    if s3["stage3b"] is not None:
        thA = np.asarray(s3["stage3b"]["thetaA_hat"]).reshape(-1)
        thB = np.asarray(s3["stage3b"]["thetaB_hat"]).reshape(-1)

        out["stage3b_rmse"] = float(s3["stage3b"]["fit"]["rmse"])
        out["stage3b_r2"]   = float(s3["stage3b"]["fit"]["r2"])
        out["R0_stage3b"]   = float(s3["stage3b"]["R0_hat"])

        for i, val in enumerate(thA, start=1):
            out[f"thetaA_{i}"] = float(val)
        for i, val in enumerate(thB, start=8):
            out[f"thetaB_{i}"] = float(val)

        out["thetaA_norm"] = float(np.linalg.norm(thA))
        out["thetaB_norm"] = float(np.linalg.norm(thB))
    else:
        out["stage3b_rmse"] = np.nan
        out["stage3b_r2"]   = np.nan
        out["R0_stage3b"]   = np.nan

    return out


def plot_learned_nonlinearity_from_result(result: dict, title_prefix=""):
    if not result.get("ok", False):
        print("Cannot plot nonlinearity for failed result.")
        return

    s2 = result["stage2"]
    xp = np.asarray(result["xp_sig"]).reshape(-1)
    xn = np.asarray(result["xn_sig"]).reshape(-1)
    X_proxy_local = np.asarray(result["X_proxy"])
    thetaZ_local = np.asarray(s2["thetaZ_hat"]).reshape(-1)
    zhat_local = s2["zhat_from_thetaZ"]

    Zvals = []
    for k in range(X_proxy_local.shape[0]):
        z = zhat_local(jnp.array(X_proxy_local[k], dtype=DTYPE), jnp.array(thetaZ_local, dtype=DTYPE))
        Zvals.append(float(np.asarray(z)))
    Zvals = np.asarray(Zvals)

    dxp = np.asarray(s2["dxp"]).reshape(-1)
    dxn = np.asarray(s2["dxn"]).reshape(-1)

    plt.figure(figsize=(12, 4))
    plt.subplot(1, 3, 1)
    plt.plot(xp, Zvals, "o-")
    plt.grid(True)
    plt.xlabel("xp")
    plt.ylabel("Z_hat")
    plt.title(f"{title_prefix} Z_hat vs xp")

    plt.subplot(1, 3, 2)
    plt.plot(xn, Zvals, "o-")
    plt.grid(True)
    plt.xlabel("xn")
    plt.ylabel("Z_hat")
    plt.title(f"{title_prefix} Z_hat vs xn")

    plt.subplot(1, 3, 3)
    plt.scatter(dxp, dxn, c=Zvals)
    plt.grid(True)
    plt.xlabel("dxp")
    plt.ylabel("dxn")
    plt.title(f"{title_prefix} operating path colored by Z_hat")
    plt.tight_layout()
    plt.show()

    print(f"{title_prefix} nonlinearity summaries")
    print("  monotonicity fraction vs time on Z_hat:", monotonicity_fraction(Zvals))
    print("  corr(Z_hat, xp):", safe_corr(Zvals, xp))
    print("  corr(Z_hat, xn):", safe_corr(Zvals, xn))


# %% ======================================================
# CELL 14 — Cross-cycle transfer/generalization of selected-cycle model
# =========================================================
if not RUN_GENERALIZATION_TESTS:
    print("\n[INFO] RUN_GENERALIZATION_TESTS=False, skipping Cell 14.")
else:
    print("\n" + "=" * 90)
    print("CROSS-CYCLE TRANSFER / GENERALIZATION TESTS")
    print("=" * 90)

    if GENERALIZATION_SCOPE == "all_cycles":
        test_cycle_ids = list(range(len(cycles)))
    else:
        if GENERALIZATION_CYCLE_IDS is None:
            test_cycle_ids = choose_spread_indices(len(cycles), N_GENERALIZATION_CYCLES)
            if chosen not in test_cycle_ids:
                test_cycle_ids = sorted(set(test_cycle_ids + [chosen]))
        else:
            test_cycle_ids = sorted(set(int(i) for i in GENERALIZATION_CYCLE_IDS if 0 <= int(i) < len(cycles)))

    print("Transfer/generalization test cycle IDs:", test_cycle_ids)
    print("Selected/fitted source cycle:", chosen)

    transfer_rows = []

    for cid in test_cycle_ids:
        cyc_df = cycles[cid]

        print("\n" + "-" * 80)
        print(f"TRANSFER TEST ON TARGET CYCLE {cid}")
        print("-" * 80)

        row = dict(target_cycle=cid, source_cycle=chosen, deg=POLY_DEG)

        try:
            s2_transfer = evaluate_stage2_transfer(
                cyc_df,
                fitted_stage2=dict(
                    thetaZ_hat=thetaZ_hat,
                    rawR0_hat=rawR0_hat,
                    xp_ref=xp_ref,
                    xn_ref=xn_ref,
                    xp_scale=xp_scale,
                    xn_scale=xn_scale,
                ),
                source_deg=POLY_DEG,
                source_cycle_idx=chosen,
            )
            row["stage2_transfer_rmse"] = float(s2_transfer["fit"]["rmse"])
            row["stage2_transfer_r2"] = float(s2_transfer["fit"]["r2"])
            row["stage2_transfer_bfr"] = float(s2_transfer["fit"]["bfr"])
        except Exception as e:
            row["stage2_transfer_rmse"] = np.nan
            row["stage2_transfer_r2"] = np.nan
            row["stage2_transfer_bfr"] = np.nan
            row["stage2_transfer_error"] = str(e)

        if RUN_STAGE3:
            try:
                s3_transfer = evaluate_stage3b_transfer(
                    cyc_df,
                    fitted_stage3b=dict(
                        thetaA_hat=thetaA_hat_stage3b,
                        thetaB_hat=thetaB_hat_stage3b,
                        thetaZ_hat=thetaZ_hat_stage3b,
                        rawR0_hat=rawR0_hat_stage3b,
                        R0_hat=R0_hat_stage3b,
                    ),
                    fitted_stage2_meta=dict(
                        xp_ref=xp_ref,
                        xn_ref=xn_ref,
                        xp_scale=xp_scale,
                        xn_scale=xn_scale,
                    ),
                    source_deg=POLY_DEG,
                    source_cycle_idx=chosen,
                )
                row["stage3b_transfer_rmse"] = float(s3_transfer["fit"]["rmse"])
                row["stage3b_transfer_r2"] = float(s3_transfer["fit"]["r2"])
                row["stage3b_transfer_bfr"] = float(s3_transfer["fit"]["bfr"])
            except Exception as e:
                row["stage3b_transfer_rmse"] = np.nan
                row["stage3b_transfer_r2"] = np.nan
                row["stage3b_transfer_bfr"] = np.nan
                row["stage3b_transfer_error"] = str(e)

        transfer_rows.append(row)

    transfer_df = pd.DataFrame(transfer_rows)
    print("\nTRANSFER SUMMARY")
    print(transfer_df)

    if not transfer_df.empty:
        plt.figure(figsize=(12, 4))
        plt.plot(transfer_df["target_cycle"], transfer_df["stage2_transfer_rmse"], "o-", label="Stage 2 transfer RMSE")
        if "stage3b_transfer_rmse" in transfer_df.columns:
            plt.plot(transfer_df["target_cycle"], transfer_df["stage3b_transfer_rmse"], "o-", label="Stage 3b transfer RMSE")
        plt.grid(True)
        plt.xlabel("Target cycle index")
        plt.ylabel("RMSE [V]")
        plt.title("Cross-cycle transfer RMSE from selected-cycle fitted model")
        plt.legend()
        plt.tight_layout()
        plt.show()

        plt.figure(figsize=(12, 4))
        plt.plot(transfer_df["target_cycle"], transfer_df["stage2_transfer_r2"], "o-", label="Stage 2 transfer R2%")
        if "stage3b_transfer_r2" in transfer_df.columns:
            plt.plot(transfer_df["target_cycle"], transfer_df["stage3b_transfer_r2"], "o-", label="Stage 3b transfer R2%")
        plt.grid(True)
        plt.xlabel("Target cycle index")
        plt.ylabel("R2 [%]")
        plt.title("Cross-cycle transfer R2 from selected-cycle fitted model")
        plt.legend()
        plt.tight_layout()
        plt.show()


# %% ======================================================
# CELL 15 — Fit several/all cycles independently and summarize stability
# =========================================================
if not RUN_GENERALIZATION_TESTS or not RUN_PER_CYCLE_REFITS:
    print("\n[INFO] Skipping per-cycle refits in Cell 15.")
else:
    print("\n" + "=" * 90)
    print("PER-CYCLE REFITS ACROSS SEVERAL / ALL CYCLES")
    print("=" * 90)

    if REFIT_ALL_CYCLES or GENERALIZATION_SCOPE == "all_cycles":
        refit_cycle_ids = list(range(len(cycles)))
    else:
        if GENERALIZATION_CYCLE_IDS is None:
            refit_cycle_ids = choose_spread_indices(len(cycles), N_GENERALIZATION_CYCLES)
            if chosen not in refit_cycle_ids:
                refit_cycle_ids = sorted(set(refit_cycle_ids + [chosen]))
        else:
            refit_cycle_ids = sorted(set(int(i) for i in GENERALIZATION_CYCLE_IDS if 0 <= int(i) < len(cycles)))

    print("Refit cycle IDs:", refit_cycle_ids)

    all_cycle_results = []
    monitor_rows = []

    for cid in refit_cycle_ids:
        print("\n" + "#" * 90)
        print(f"REFIT FULL PIPELINE ON CYCLE {cid}")
        print("#" * 90)

        res = run_full_cycle_pipeline(
            cycles[cid],
            cycle_idx=cid,
            deg=POLY_DEG,
            run_stage3=RUN_STAGE3,
            verbose=True,
        )
        all_cycle_results.append(res)

        row = summarize_monitorable_parameters(res)
        monitor_rows.append(row)

    monitor_df = pd.DataFrame(monitor_rows)
    print("\nPER-CYCLE MONITOR SUMMARY")
    print(monitor_df)

    ok_monitor_df = monitor_df[monitor_df["ok"] == True].copy()

    if not ok_monitor_df.empty:
        plt.figure(figsize=(12, 4))
        plt.plot(ok_monitor_df["cycle_idx"], ok_monitor_df["stage2_rmse"], "o-", label="Stage 2 RMSE")
        if "stage3b_rmse" in ok_monitor_df.columns:
            plt.plot(ok_monitor_df["cycle_idx"], ok_monitor_df["stage3b_rmse"], "o-", label="Stage 3b RMSE")
        plt.grid(True)
        plt.xlabel("Cycle index")
        plt.ylabel("RMSE [V]")
        plt.title(f"Per-cycle fit quality across cycles (deg={POLY_DEG})")
        plt.legend()
        plt.tight_layout()
        plt.show()

        plt.figure(figsize=(12, 4))
        plt.plot(ok_monitor_df["cycle_idx"], ok_monitor_df["R0_stage2"], "o-", label="R0 Stage 2")
        if "R0_stage3b" in ok_monitor_df.columns:
            plt.plot(ok_monitor_df["cycle_idx"], ok_monitor_df["R0_stage3b"], "o-", label="R0 Stage 3b")
        plt.grid(True)
        plt.xlabel("Cycle index")
        plt.ylabel("Resistance [Ohm]")
        plt.title("Monitorable R0 across cycles")
        plt.legend()
        plt.tight_layout()
        plt.show()

        if "thetaA_1" in ok_monitor_df.columns:
            plt.figure(figsize=(12, 5))
            for col in [c for c in ok_monitor_df.columns if c.startswith("thetaA_")]:
                plt.plot(ok_monitor_df["cycle_idx"], ok_monitor_df[col], "o-", label=col)
            plt.grid(True)
            plt.xlabel("Cycle index")
            plt.ylabel("Estimated value")
            plt.title("Monitorable thetaA parameters across cycles")
            plt.legend(ncol=2)
            plt.tight_layout()
            plt.show()

        if "thetaB_8" in ok_monitor_df.columns:
            plt.figure(figsize=(12, 5))
            for col in [c for c in ok_monitor_df.columns if c.startswith("thetaB_")]:
                plt.plot(ok_monitor_df["cycle_idx"], ok_monitor_df[col], "o-", label=col)
            plt.grid(True)
            plt.xlabel("Cycle index")
            plt.ylabel("Estimated value")
            plt.title("Monitorable thetaB parameters across cycles")
            plt.legend(ncol=2)
            plt.tight_layout()
            plt.show()

        print("\nSTABILITY CHECKS")
        print("Mean Stage 2 RMSE:", float(ok_monitor_df["stage2_rmse"].mean()))
        print("Std  Stage 2 RMSE:", float(ok_monitor_df["stage2_rmse"].std(ddof=0)))
        if "stage3b_rmse" in ok_monitor_df.columns:
            print("Mean Stage 3b RMSE:", float(ok_monitor_df["stage3b_rmse"].mean()))
            print("Std  Stage 3b RMSE:", float(ok_monitor_df["stage3b_rmse"].std(ddof=0)))
        print("Mean R0 Stage 2:", float(ok_monitor_df["R0_stage2"].mean()))
        print("Std  R0 Stage 2:", float(ok_monitor_df["R0_stage2"].std(ddof=0)))


# %% ======================================================
# CELL 16 — Visualize learned nonlinearity
# =========================================================
if not RUN_NONLINEARITY_PLOTS:
    print("\n[INFO] RUN_NONLINEARITY_PLOTS=False, skipping Cell 16.")
else:
    print("\n" + "=" * 90)
    print("VISUALIZE LEARNED NONLINEARITY")
    print("=" * 90)

    print("\nSelected-cycle nonlinearity:")
    selected_result_for_plot = dict(
        ok=True,
        cycle_idx=chosen,
        deg=POLY_DEG,
        X_proxy=X_proxy,
        xp_sig=xp_sig,
        xn_sig=xn_sig,
        ceL_sig=ceL_sig,
        ceR_sig=ceR_sig,
        ln_ce_ratio=ln_ce_ratio_sig,
        proxy_info=dict(xp_rng=xp_rng, xn_rng=xn_rng, ceL_rng=ceL_rng, ceR_rng=ceR_rng),
        stage2=dict(
            thetaZ_hat=thetaZ_hat,
            zhat_from_thetaZ=zhat_from_thetaZ_stage2,
            dxp=dxp_dbg,
            dxn=dxn_dbg,
        ),
    )
    plot_learned_nonlinearity_from_result(
        selected_result_for_plot,
        title_prefix=f"cycle {chosen} |"
    )

    if RUN_GENERALIZATION_TESTS and RUN_PER_CYCLE_REFITS and "all_cycle_results" in globals():
        ok_results = [r for r in all_cycle_results if r.get("ok", False)]

        if NONLINEARITY_PLOT_CYCLE_IDS is None:
            ids_for_nonlin = [r["cycle_idx"] for r in ok_results]
            ids_for_nonlin = choose_spread_indices(len(ids_for_nonlin), min(N_NONLINEARITY_PLOTS, len(ids_for_nonlin)))
            ids_for_nonlin = [ok_results[i]["cycle_idx"] for i in ids_for_nonlin]
        else:
            ids_for_nonlin = [int(i) for i in NONLINEARITY_PLOT_CYCLE_IDS]

        print("\nAdditional nonlinearity plots for cycles:", ids_for_nonlin)

        for cid in ids_for_nonlin:
            matches = [r for r in ok_results if r["cycle_idx"] == cid]
            if len(matches) == 0:
                continue
            plot_learned_nonlinearity_from_result(
                matches[0],
                title_prefix=f"cycle {cid} |"
            )


# %% ======================================================
# CELL 17 — Extract monitorable parameters and tell degradation story
# =========================================================
if not RUN_GENERALIZATION_TESTS or not RUN_PER_CYCLE_REFITS or "monitor_df" not in globals():
    print("\n[INFO] Skipping degradation-story cell because monitor_df is not available.")
else:
    print("\n" + "=" * 90)
    print("EXTRACT MONITORABLE PARAMETERS + TELL THE DEGRADATION STORY")
    print("=" * 90)

    story_df = monitor_df[monitor_df["ok"] == True].copy().sort_values("cycle_idx")
    print(story_df)

    if not story_df.empty:
        cols_to_show = [
            "cycle_idx",
            "stage2_rmse",
            "stage3b_rmse",
            "R0_stage2",
            "R0_stage3b",
            "xp_rng",
            "xn_rng",
            "z_theta_l2",
            "z_theta_maxabs",
        ]
        cols_to_show = [c for c in cols_to_show if c in story_df.columns]
        print("\nCore story table:")
        print(story_df[cols_to_show])

        plt.figure(figsize=(12, 4))
        plt.plot(story_df["cycle_idx"], story_df["R0_stage2"], "o-", label="R0 Stage 2")
        if "R0_stage3b" in story_df.columns:
            plt.plot(story_df["cycle_idx"], story_df["R0_stage3b"], "o-", label="R0 Stage 3b")
        plt.grid(True)
        plt.xlabel("Cycle index")
        plt.ylabel("Ohmic resistance [Ohm]")
        plt.title("Degradation story: resistance trend")
        plt.legend()
        plt.tight_layout()
        plt.show()

        plt.figure(figsize=(12, 4))
        plt.plot(story_df["cycle_idx"], story_df["stage2_rmse"], "o-", label="Stage 2 RMSE")
        if "stage3b_rmse" in story_df.columns:
            plt.plot(story_df["cycle_idx"], story_df["stage3b_rmse"], "o-", label="Stage 3b RMSE")
        plt.grid(True)
        plt.xlabel("Cycle index")
        plt.ylabel("RMSE [V]")
        plt.title("Degradation story: fit quality trend")
        plt.legend()
        plt.tight_layout()
        plt.show()

        plt.figure(figsize=(12, 4))
        plt.plot(story_df["cycle_idx"], story_df["z_theta_l2"], "o-", label="||thetaZ||_2")
        plt.plot(story_df["cycle_idx"], story_df["z_theta_maxabs"], "o-", label="max|thetaZ|")
        plt.grid(True)
        plt.xlabel("Cycle index")
        plt.ylabel("Magnitude")
        plt.title("Degradation story: nonlinearity-complexity trend")
        plt.legend()
        plt.tight_layout()
        plt.show()

        print("\nTEXT STORY")
        r0_corr = safe_corr(story_df["cycle_idx"].to_numpy(), story_df["R0_stage2"].to_numpy())
        rmse_corr = safe_corr(story_df["cycle_idx"].to_numpy(), story_df["stage2_rmse"].to_numpy())
        zcorr = safe_corr(story_df["cycle_idx"].to_numpy(), story_df["z_theta_l2"].to_numpy())

        print(f"- Correlation(cycle index, R0_stage2)   = {r0_corr:.6g}")
        print(f"- Correlation(cycle index, stage2_rmse) = {rmse_corr:.6g}")
        print(f"- Correlation(cycle index, ||thetaZ||2) = {zcorr:.6g}")

        if np.isfinite(r0_corr):
            if r0_corr > 0.3:
                print("- R0 tends to increase with cycle index, which is consistent with resistance growth.")
            elif r0_corr < -0.3:
                print("- R0 tends to decrease with cycle index, which may indicate estimator instability or cycle nonuniformity.")
            else:
                print("- R0 is relatively flat over the tested cycles, suggesting little resistance drift or insufficient sensitivity.")

        if np.isfinite(zcorr):
            if zcorr > 0.3:
                print("- The nonlinearity magnitude tends to grow with cycle index.")
            elif zcorr < -0.3:
                print("- The nonlinearity magnitude tends to shrink with cycle index.")
            else:
                print("- The nonlinearity magnitude is fairly stable over the tested cycles.")

        print("- Final judgment should combine: fit stability, R0 trend, A/B trend, and smoothness of the learned nonlinearity.")


# %% ======================================================
# CELL 18 — Degree comparison helper (17 vs 15 vs 14)
# =========================================================
if not RUN_MULTI_DEGREE_COMPARISON:
    print("\n[INFO] RUN_MULTI_DEGREE_COMPARISON=False, skipping Cell 18.")
else:
    print("\n" + "=" * 90)
    print("MULTI-DEGREE COMPARISON (17 vs 15 vs 14)")
    print("=" * 90)

    if MULTI_DEGREE_SCOPE == "all_cycles":
        degree_cycle_ids = list(range(len(cycles)))
    else:
        if MULTI_DEGREE_CYCLE_IDS is None:
            degree_cycle_ids = choose_spread_indices(len(cycles), N_MULTI_DEGREE_CYCLES)
            if chosen not in degree_cycle_ids:
                degree_cycle_ids = sorted(set(degree_cycle_ids + [chosen]))
        else:
            degree_cycle_ids = sorted(set(int(i) for i in MULTI_DEGREE_CYCLE_IDS if 0 <= int(i) < len(cycles)))

    print("Degree-comparison cycles:", degree_cycle_ids)
    print("Degrees:", MULTI_DEGREE_LIST)

    degree_rows = []

    for deg_test in MULTI_DEGREE_LIST:
        for cid in degree_cycle_ids:
            print("\n" + "-" * 80)
            print(f"DEGREE TEST | cycle={cid}, deg={deg_test}")
            print("-" * 80)

            res = run_full_cycle_pipeline(
                cycles[cid],
                cycle_idx=cid,
                deg=deg_test,
                run_stage3=RUN_STAGE3,
                verbose=False,
            )

            row = dict(cycle_idx=cid, deg=deg_test, ok=res.get("ok", False))
            if res.get("ok", False):
                row["stage2_rmse"] = float(res["stage2"]["fit"]["rmse"])
                row["stage2_r2"]   = float(res["stage2"]["fit"]["r2"])
                row["R0_stage2"]   = float(res["stage2"]["R0_hat"])
                row["z_theta_l2"]  = float(np.linalg.norm(res["stage2"]["thetaZ_hat"]))
                row["z_theta_maxabs"] = float(np.max(np.abs(res["stage2"]["thetaZ_hat"])))

                if res["stage3"]["stage3b"] is not None:
                    row["stage3b_rmse"] = float(res["stage3"]["stage3b"]["fit"]["rmse"])
                    row["stage3b_r2"]   = float(res["stage3"]["stage3b"]["fit"]["r2"])
                    row["R0_stage3b"]   = float(res["stage3"]["stage3b"]["R0_hat"])
                else:
                    row["stage3b_rmse"] = np.nan
                    row["stage3b_r2"]   = np.nan
                    row["R0_stage3b"]   = np.nan
            else:
                row["reason"] = res.get("reason", "unknown failure")

            degree_rows.append(row)

    degree_df = pd.DataFrame(degree_rows)
    print("\nMULTI-DEGREE SUMMARY")
    print(degree_df)

    ok_degree_df = degree_df[degree_df["ok"] == True].copy()

    if not ok_degree_df.empty:
        grouped = ok_degree_df.groupby("deg").agg(
            stage2_rmse_mean=("stage2_rmse", "mean"),
            stage2_rmse_std=("stage2_rmse", "std"),
            stage2_r2_mean=("stage2_r2", "mean"),
            stage3b_rmse_mean=("stage3b_rmse", "mean"),
            stage3b_rmse_std=("stage3b_rmse", "std"),
            stage3b_r2_mean=("stage3b_r2", "mean"),
            z_theta_l2_mean=("z_theta_l2", "mean"),
            z_theta_maxabs_mean=("z_theta_maxabs", "mean"),
        ).reset_index()

        print("\nAGGREGATED DEGREE COMPARISON")
        print(grouped)

        plt.figure(figsize=(10, 4))
        plt.plot(grouped["deg"], grouped["stage2_rmse_mean"], "o-", label="mean Stage 2 RMSE")
        if "stage3b_rmse_mean" in grouped.columns:
            plt.plot(grouped["deg"], grouped["stage3b_rmse_mean"], "o-", label="mean Stage 3b RMSE")
        plt.grid(True)
        plt.xlabel("Polynomial degree")
        plt.ylabel("Mean RMSE [V]")
        plt.title("Degree selection across several cycles")
        plt.legend()
        plt.tight_layout()
        plt.show()

        plt.figure(figsize=(10, 4))
        plt.plot(grouped["deg"], grouped["z_theta_l2_mean"], "o-", label="mean ||thetaZ||2")
        plt.plot(grouped["deg"], grouped["z_theta_maxabs_mean"], "o-", label="mean max|thetaZ|")
        plt.grid(True)
        plt.xlabel("Polynomial degree")
        plt.ylabel("Magnitude")
        plt.title("Parameter growth vs degree")
        plt.legend()
        plt.tight_layout()
        plt.show()

        print("\nDEGREE DECISION GUIDE")
        print("- Keep degree 17 if it stays smooth, stable, and has clearly better mean RMSE without ugly parameter growth.")
        print("- Back off to degree 15 if fit is close but stability is better.")
        print("- Back off to degree 14 if 17/15 look too sensitive across cycles or the nonlinearity becomes rough.")# =========================================================
# real_data_physics_informed_additive_poly_sysid_pipeline.ipynb
# =========================================================
# PURPOSE
#   Real-data notebook for system identification using a
#   physics-consistent additive polynomial voltage surrogate:
#
#       Z_hat(x) = c0
#                  + sum_{k=1..d} a_k * dxp^k
#                  + sum_{k=1..d} b_k * dxn^k
#                  + k_ln * ln(ceR/ceL)        [optional]
#
#   where
#
#       dxp = (xp - xp_ref) / xp_scale
#       dxn = (xn - xn_ref) / xn_scale
#
#   and terminal voltage model:
#
#       V_hat = Z_hat(x) - Ns * I * R0 - v_rc   [if USE_RC=True]
#       V_hat = Z_hat(x) - Ns * I * R0          [if USE_RC=False]
#
# CORE IDEA
#   We want the voltage surrogate to stay close to battery physics.
#   Therefore, we do NOT use explicit time-dependent transient terms
#   such as c_j * exp(-t/tau_j) in Z_hat.
#
#   Instead, Z_hat is built only from state-dependent quantities:
#     - positive-electrode proxy stoichiometry xp
#     - negative-electrode proxy stoichiometry xn
#     - optional electrolyte log feature ln(ceR/ceL)
#
#   This keeps the surrogate more interpretable and more consistent
#   with the physical structure of the battery model.
#
# RELATION TO STATIC LS / STATIC LS+R
#   The static LS and static LS+R checks use the SAME polynomial feature
#   structure as the surrogate:
#
#       Phi(x) = [1, dxp, dxp^2, ..., dxp^d, dxn, dxn^2, ..., dxn^d]
#                [+ ln(ceR/ceL) if enabled]
#
#   Static LS solves:
#
#       Y ≈ Phi * theta_Z
#
#   Static LS+R solves:
#
#       Y ≈ Phi * theta_Z - I * R0
#
#   These static fits are used as transparent reference models to check
#   how expressive the chosen surrogate basis is.
#
#   Important:
#   - Static LS / LS+R and CT Stage 2 share the same functional form.
#   - But CT Stage 2 still learns parameters through the dynamic model,
#     so its fitted coefficients do not have to match the static LS
#     solution exactly.
#
# WORKFLOW
#   Cell 0  : HPC-safe imports and environment setup
#   Cell 1  : User settings
#   Cell 2  : Load raw .mpr file and inspect
#   Cell 3  : Plot/score helpers + manual least-squares helpers
#   Cell 4  : Find discharge cycles and select one
#   Cell 5  : Prepare (t,U,Y): units, discharge-only, resample, trim
#   Cell 6  : Nominal 14-state proxy model for xp/xn/ceL/ceR
#   Cell 7  : JAX helpers and parameterized A(thetaA), B(thetaB)
#   Cell 8  : Physics-consistent additive polynomial surrogate builder
#   Cell 9  : Stage 2 fit: surrogate + R0, with A/B fixed nominal
#   Cell 10 : Degree sweep summary or optional sweep code
#   Cell 11 : Stage 3a fit A/B only, with Stage-2 surrogate frozen
#   Cell 12 : Stage 3b unfreeze everything and refine full model
#   Cell 13 : Reusable cycle pipeline helpers
#   Cell 14 : Cross-cycle transfer/generalization of selected-cycle model
#   Cell 15 : Fit several/all cycles independently and summarize stability
#   Cell 16 : Visualize learned nonlinearity
#   Cell 17 : Extract monitorable parameters and tell degradation story
#   Cell 18 : Degree comparison helper (17 vs 15 vs 14)
#
# NOTES
#   - This notebook is for REAL DATA, so there is no generated-truth system.
#   - "A/B fixed" means fixed to the nominal proxy dynamics from Config.
#   - The explicit time-transient basis has been removed from Z_hat to keep
#     the surrogate closer to battery physics.
#   - If early-time transient mismatch remains, a more physical next step is
#     to add dynamic states (for example RC states), rather than adding
#     explicit time exponentials to Z_hat.
#   - Static LS and LS+R are included both as performance references and
#     as interpretable ways to understand the surrogate basis.
#   - We keep the random-cycle workflow, but we now also test the model on
#     several or all cycles to judge stability and generalization.
# =========================================================


# %% ======================================================
# CELL 0 — Imports + notebook setup (HPC-safe, threaded CPU)
# =========================================================
from __future__ import annotations

import os
import math
import warnings
from dataclasses import dataclass
from typing import Tuple, List, Dict, Any, Optional

# ---- Threading: pick up CPUs from Slurm if present, else default to 8 ----
N = int(os.environ.get("SLURM_CPUS_PER_TASK", "8"))

# IMPORTANT: set BEFORE importing jax / numpy-heavy libs
os.environ["XLA_FLAGS"] = (
    f"--xla_cpu_multi_thread_eigen=true intra_op_parallelism_threads={N}"
)
os.environ["OMP_NUM_THREADS"] = str(N)
os.environ["MKL_NUM_THREADS"] = str(N)
os.environ["OPENBLAS_NUM_THREADS"] = str(N)
os.environ["NUMEXPR_NUM_THREADS"] = str(N)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

import control as ct
from scipy.linalg import block_diag

import jax
import jax.numpy as jnp
import diffrax
from jax_sysid.models import CTModel

# Project modules
import src.file_loader as fl

# JAX settings
jax.config.update("jax_enable_x64", True)
DTYPE = jnp.float64

print("SLURM_CPUS_PER_TASK:", os.environ.get("SLURM_CPUS_PER_TASK"))
print("XLA_FLAGS:", os.environ.get("XLA_FLAGS"))
print("JAX devices:", jax.devices())
print("JAX default dtype:", DTYPE)


# %% ======================================================
# CELL 1 — User settings (EDIT THESE)
# =========================================================
print("CWD:", os.getcwd())

# -----------------------------
# File and column settings
# -----------------------------
MPR_PATH = "12to1-25%CNC-3%GQDs _C01.mpr"

TIME_COL = "time/s"
I_COL    = "I/mA"
V_COL    = "Ewe/V"

# -----------------------------
# Cycle selection
# -----------------------------
# Modes:
#   "index"  -> use CYCLE_INDEX
#   "random" -> choose one cycle randomly
CYCLE_MODE  = "random"     # "index" | "random"
CYCLE_INDEX = 0

# Random behavior:
#   RANDOM_SEED = 42   -> reproducible random choice
#   RANDOM_SEED = None -> different random choice on different runs
RANDOM_SEED = 42

# Optional filtering / cycle construction
USE_MIN_CYCLE_LEN = False
MIN_CYCLE_LEN = 50

# Prepend some points before discharge starts, similar to old src logic
INCLUDE_PREVIOUS_SEGMENT = False
N_PREV_POINTS = 10

# -----------------------------
# Data prep
# -----------------------------
FORCE_UNITS = "A"       # "auto" | "mA" | "A"
V_REF       = "none"    # "none" | "first" | "mean"
RESAMPLE    = True

ENFORCE_DISCHARGE_ONLY = True
RAW_DISCHARGE_SIGN     = "negative"

# Initial state guess (proxy stoich)
XN0 = 0.60
XP0 = 0.60
CE0_DEV = 0.0

# Optional trim after selecting cycle
TMAX = -1.0

# Optional drop of first few samples after cycle extraction
DROP_FIRST_N = 0
# DROP_FIRST_N = 1

# -----------------------------
# Stage-2 polynomial settings
# -----------------------------
POLY_DEG = 17
DEGREE_SWEEP = [14, 15, 17]

USE_LN_FEATURE = False

# Physics-consistent surrogate:
# no explicit time-transient basis in Z_hat
USE_FAST_TRANSIENT_BASIS = False
TRANSIENT_TAU_LIST = []

# -----------------------------
# RC branch settings
# -----------------------------
USE_RC = False
R1_INIT_OHM = 0.05
TAU_INIT_S  = 20.0

# -----------------------------
# Optimization settings
# -----------------------------
ADAM_EPOCHS_STAGE2  = 1200
ADAM_EPOCHS_STAGE3A = 1500
ADAM_EPOCHS_STAGE3B = 2000

USE_LBFGS = True
LBFGS_EPOCHS = 300

ADAM_ETA_STAGE2  = 5e-4
ADAM_ETA_STAGE3A = 5e-4
ADAM_ETA_STAGE3B = 2e-4

# -----------------------------
# Solver settings
# -----------------------------
SOLVER_RTOL = 1e-5
SOLVER_ATOL = 1e-8
DT0_DIV     = 5.0
MAX_STEPS   = 10_000_000

# -----------------------------
# Sanity thresholds
# -----------------------------
MIN_MED_ABS_I_A = 5e-3
MIN_XP_RANGE    = 1e-4
MIN_XN_RANGE    = 1e-4

# -----------------------------
# Stage-3 regularization
# -----------------------------
RHO_TH_STAGE2  = 1e-8
RHO_TH_STAGE3A = 1e-8
RHO_TH_STAGE3B = 1e-8

# -----------------------------
# Optional light parameter clips
# -----------------------------
CLIP_RAW_A = 4.0
CLIP_RAW_B = 2.0
CLIP_RAW_Z = 50.0

# -----------------------------
# Real-cell config assumptions
# -----------------------------
N_SERIES_REAL = 1

# -----------------------------
# Debug / gating
# -----------------------------
RUN_DEGREE_SWEEP = False
RUN_STAGE3 = True

# -----------------------------
# New pipeline / generalization settings
# -----------------------------
RUN_GENERALIZATION_TESTS = True

# How many cycles to use in transfer/generalization tests:
#   "selected_subset" | "all_cycles"
GENERALIZATION_SCOPE = "selected_subset"

# Candidate cycles to test:
#   If None, notebook auto-picks a spread across the available cycles.
GENERALIZATION_CYCLE_IDS = None

# Number of cycles for auto-selected subset when GENERALIZATION_SCOPE="selected_subset"
N_GENERALIZATION_CYCLES = 8

# Whether to fit every tested cycle independently (heavier but useful)
RUN_PER_CYCLE_REFITS = True

# If True and GENERALIZATION_SCOPE="all_cycles", refit all cycles
REFIT_ALL_CYCLES = False

# Plot learned nonlinearity for selected cycle and a few extra cycles
RUN_NONLINEARITY_PLOTS = True
NONLINEARITY_PLOT_CYCLE_IDS = None
N_NONLINEARITY_PLOTS = 4

# Degree-comparison test across several cycles
RUN_MULTI_DEGREE_COMPARISON = True
MULTI_DEGREE_LIST = [17, 15, 14]
MULTI_DEGREE_SCOPE = "selected_subset"   # "selected_subset" | "all_cycles"
MULTI_DEGREE_CYCLE_IDS = None
N_MULTI_DEGREE_CYCLES = 6

print("Notebook target: real_data_physics_informed_additive_poly_sysid_pipeline.ipynb")
print("Main polynomial degree:", POLY_DEG)
print("Degree sweep:", DEGREE_SWEEP)
print("USE_LN_FEATURE:", USE_LN_FEATURE)
print("USE_FAST_TRANSIENT_BASIS:", USE_FAST_TRANSIENT_BASIS)
print("TRANSIENT_TAU_LIST:", TRANSIENT_TAU_LIST)
print("USE_RC:", USE_RC)
print("CYCLE_MODE:", CYCLE_MODE)
print("CYCLE_INDEX:", CYCLE_INDEX)
print("RANDOM_SEED:", RANDOM_SEED)
print("USE_MIN_CYCLE_LEN:", USE_MIN_CYCLE_LEN)
print("MIN_CYCLE_LEN:", MIN_CYCLE_LEN)
print("INCLUDE_PREVIOUS_SEGMENT:", INCLUDE_PREVIOUS_SEGMENT)
print("N_PREV_POINTS:", N_PREV_POINTS)
print("RESAMPLE:", RESAMPLE)
print("TMAX:", TMAX)
print("DROP_FIRST_N:", DROP_FIRST_N)
print("RUN_DEGREE_SWEEP:", RUN_DEGREE_SWEEP)
print("RUN_STAGE3:", RUN_STAGE3)
print("RUN_GENERALIZATION_TESTS:", RUN_GENERALIZATION_TESTS)
print("GENERALIZATION_SCOPE:", GENERALIZATION_SCOPE)
print("RUN_PER_CYCLE_REFITS:", RUN_PER_CYCLE_REFITS)
print("RUN_NONLINEARITY_PLOTS:", RUN_NONLINEARITY_PLOTS)
print("RUN_MULTI_DEGREE_COMPARISON:", RUN_MULTI_DEGREE_COMPARISON)


# %% ======================================================
# CELL 2 — Load + plot raw signals
# =========================================================
mpr_file = fl.load_mpr(MPR_PATH)
df0 = pd.DataFrame(mpr_file.data)

print("Columns:", df0.columns.tolist())
assert TIME_COL in df0.columns, f"Missing {TIME_COL}"
assert I_COL in df0.columns, f"Missing {I_COL}"
assert V_COL in df0.columns, f"Missing {V_COL}"

df = df0.set_index(TIME_COL)[[I_COL, V_COL]].copy().sort_index()
df = df.dropna()

print(df.head())

plt.figure(figsize=(12, 6))
plt.subplot(2, 1, 1)
plt.plot(df.index.values, df[I_COL].values)
plt.grid(True)
plt.title("Raw Current")
plt.ylabel(I_COL)

plt.subplot(2, 1, 2)
plt.plot(df.index.values, df[V_COL].values)
plt.grid(True)
plt.title("Raw Voltage")
plt.ylabel(V_COL)
plt.xlabel("time [s]")
plt.tight_layout()
plt.show()


# %% ======================================================
# CELL 3 — Helpers: plots + scoring + manual LS
# =========================================================
def plot_voltage(t, y, yhat=None, title="Voltage"):
    t = np.asarray(t).reshape(-1)
    y = np.asarray(y).reshape(-1, 1)

    plt.figure(figsize=(10, 4))
    plt.plot(t, y[:, 0], label="Measured")
    if yhat is not None:
        yhat = np.asarray(yhat).reshape(-1, 1)
        plt.plot(t, yhat[:, 0], "--", label="Pred")
    plt.grid(True)
    plt.legend()
    plt.xlabel("t [s]")
    plt.ylabel("V [V]")
    plt.title(title)
    ax = plt.gca()
    ax.yaxis.set_major_formatter(mticker.ScalarFormatter(useOffset=False))
    ax.ticklabel_format(axis="y", style="plain", useOffset=False)
    plt.tight_layout()
    plt.show()


def vec_reshape(y):
    y = np.asarray(y)
    if y.ndim == 1:
        y = y.reshape(-1, 1)
    return y


def compute_scores(Y, Yhat, fit="R2"):
    Y = vec_reshape(Y)
    Yhat = vec_reshape(Yhat)
    y = Y[:, 0]
    yh = Yhat[:, 0]
    fit = fit.lower()

    if fit == "r2":
        denom = np.sum((y - np.mean(y)) ** 2) + 1e-12
        return 100.0 * (1.0 - np.sum((yh - y) ** 2) / denom)

    if fit == "rmse":
        return float(np.sqrt(np.mean((yh - y) ** 2)))

    if fit == "bfr":
        denom = np.sum((y - np.mean(y)) ** 2) + 1e-12
        return 100.0 * (1.0 - np.linalg.norm(yh - y) / np.sqrt(denom))

    raise ValueError("fit must be one of: R2 | BFR | RMSE")


def summarize_err(y_true: np.ndarray, y_pred: np.ndarray, name=""):
    y_true = vec_reshape(y_true)
    y_pred = vec_reshape(y_pred)
    err = y_pred[:, 0] - y_true[:, 0]

    rmse = float(np.sqrt(np.mean(err ** 2)))
    mae  = float(np.mean(np.abs(err)))
    p95  = float(np.percentile(np.abs(err), 95))
    p99  = float(np.percentile(np.abs(err), 99))
    mx   = float(np.max(np.abs(err)))

    print(f"\n{name} error summary:")
    print(f"  RMSE  = {rmse:.6g} V")
    print(f"  MAE   = {mae:.6g} V")
    print(f"  P95   = {p95:.6g} V")
    print(f"  P99   = {p99:.6g} V")
    print(f"  MAX   = {mx:.6g} V")
    return dict(rmse=rmse, mae=mae, p95=p95, p99=p99, max_abs=mx)


def report_fit(name, Y_true, Y_pred):
    Y_true = vec_reshape(Y_true)
    Y_pred = vec_reshape(Y_pred)
    err = Y_pred - Y_true
    max_abs = float(np.max(np.abs(err)))
    rmse = float(np.sqrt(np.mean(err ** 2)))
    r2 = float(compute_scores(Y_true, Y_pred, fit="R2"))
    bfr = float(compute_scores(Y_true, Y_pred, fit="BFR"))
    print(f"{name}: max|err|={max_abs:.6g}, rmse={rmse:.6g}, R2%={r2:.3f}, BFR%={bfr:.3f}")
    return dict(max_abs=max_abs, rmse=rmse, r2=r2, bfr=bfr)


def plot_current_and_voltage(t, u, y, title="Current and voltage"):
    t = np.asarray(t).reshape(-1)
    u = np.asarray(u).reshape(-1)
    y = np.asarray(y).reshape(-1)

    plt.figure(figsize=(11, 6))

    plt.subplot(2, 1, 1)
    plt.plot(t, u)
    plt.grid(True)
    plt.ylabel("I [A]")
    plt.title(title)

    plt.subplot(2, 1, 2)
    plt.plot(t, y)
    plt.grid(True)
    plt.ylabel("V [V]")
    plt.xlabel("t [s]")

    plt.tight_layout()
    plt.show()


def report_signal_span(name, x):
    x = np.asarray(x).reshape(-1)
    print(
        f"{name}: min={float(np.min(x)):.6g}, "
        f"max={float(np.max(x)):.6g}, "
        f"span={float(np.max(x)-np.min(x)):.6g}, "
        f"std={float(np.std(x)):.6g}"
    )


def simple_line_fit_rmse(t, y):
    t = np.asarray(t).reshape(-1)
    y = np.asarray(y).reshape(-1)
    p = np.polyfit(t, y, 1)
    ylin = np.polyval(p, t)
    rmse = float(np.sqrt(np.mean((y - ylin) ** 2)))
    return rmse, p


def plot_residuals(t, y, yhat, title="Residuals"):
    t = np.asarray(t).reshape(-1)
    y = np.asarray(y).reshape(-1)
    yhat = np.asarray(yhat).reshape(-1)
    err = yhat - y

    plt.figure(figsize=(10, 4))
    plt.plot(t, err)
    plt.grid(True)
    plt.xlabel("t [s]")
    plt.ylabel("Pred - Meas [V]")
    plt.title(title)
    plt.tight_layout()
    plt.show()


def plot_voltage_with_baselines(t, y, curves: dict, title="Voltage comparison"):
    t = np.asarray(t).reshape(-1)
    y = np.asarray(y).reshape(-1)

    plt.figure(figsize=(10, 4))
    plt.plot(t, y, label="Measured", linewidth=2)

    for name, vals in curves.items():
        vals = np.asarray(vals).reshape(-1)
        plt.plot(t, vals, "--", label=name)

    plt.grid(True)
    plt.xlabel("t [s]")
    plt.ylabel("V [V]")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()


def solve_ls_normal_eq(X: np.ndarray, y: np.ndarray, ridge: float = 0.0):
    X = np.asarray(X, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64).reshape(-1)

    XtX = X.T @ X
    Xty = X.T @ y

    if ridge > 0.0:
        XtX = XtX + ridge * np.eye(X.shape[1], dtype=np.float64)

    beta = np.linalg.solve(XtX, Xty)
    return beta


def solve_ls_qr(X: np.ndarray, y: np.ndarray):
    X = np.asarray(X, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64).reshape(-1)

    Q, R = np.linalg.qr(X, mode="reduced")
    beta = np.linalg.solve(R, Q.T @ y)
    return beta


def compare_ls_solvers(X: np.ndarray, y: np.ndarray, ridge: float = 1e-12, label="LS compare"):
    X = np.asarray(X, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64).reshape(-1)

    beta_ne = solve_ls_normal_eq(X, y, ridge=ridge)
    beta_qr = solve_ls_qr(X, y)
    beta_np, *_ = np.linalg.lstsq(X, y, rcond=None)

    diff_ne_np = float(np.linalg.norm(beta_ne - beta_np))
    diff_qr_np = float(np.linalg.norm(beta_qr - beta_np))
    diff_ne_qr = float(np.linalg.norm(beta_ne - beta_qr))

    print(f"\n[{label}] coefficient comparison")
    print(f"  ||beta_normal_eq - beta_lstsq|| = {diff_ne_np:.6e}")
    print(f"  ||beta_qr        - beta_lstsq|| = {diff_qr_np:.6e}")
    print(f"  ||beta_normal_eq - beta_qr   || = {diff_ne_qr:.6e}")

    return {
        "beta_normal_eq": beta_ne,
        "beta_qr": beta_qr,
        "beta_lstsq": beta_np,
        "diff_ne_np": diff_ne_np,
        "diff_qr_np": diff_qr_np,
        "diff_ne_qr": diff_ne_qr,
    }


def choose_spread_indices(n_total: int, n_pick: int) -> List[int]:
    if n_total <= 0:
        return []
    n_pick = max(1, min(n_pick, n_total))
    if n_pick == 1:
        return [0]
    vals = np.linspace(0, n_total - 1, n_pick)
    out = sorted(set(int(round(v)) for v in vals))
    while len(out) < n_pick:
        for k in range(n_total):
            if k not in out:
                out.append(k)
                if len(out) == n_pick:
                    break
    return sorted(out[:n_pick])


def monotonicity_fraction(x: np.ndarray) -> float:
    x = np.asarray(x).reshape(-1)
    if len(x) < 2:
        return np.nan
    dx = np.diff(x)
    return float(np.mean(dx >= 0.0))


def safe_corr(x: np.ndarray, y: np.ndarray) -> float:
    x = np.asarray(x).reshape(-1)
    y = np.asarray(y).reshape(-1)
    if len(x) < 2 or np.std(x) < 1e-15 or np.std(y) < 1e-15:
        return np.nan
    return float(np.corrcoef(x, y)[0, 1])


# %% ======================================================
# CELL 4 — Cycle finding + selection by index or random
# =========================================================
def get_previous_segment_by_iloc(df: pd.DataFrame, start_pos: int, n_prev_points: int = 10) -> pd.DataFrame:
    lo = max(0, start_pos - int(n_prev_points))
    return df.iloc[lo:start_pos].copy()


def find_discharging_cycles_with_meta(
    df: pd.DataFrame,
    i_col: str,
    tol_I: float = 1e-9,
    min_len: int | None = None,
    include_previous_segment: bool = False,
    n_prev_points: int = 10,
):
    I = df[i_col].to_numpy(dtype=float).reshape(-1)
    mask = I < -tol_I

    cycles = []
    cycle_meta = []

    start = None
    for k in range(len(mask)):
        if mask[k] and start is None:
            start = k

        end_now = (start is not None) and ((not mask[k]) or (k == len(mask) - 1))
        if end_now:
            end = k if (not mask[k]) else (k + 1)
            seg_len_core = end - start

            keep_seg = True
            if min_len is not None:
                keep_seg = seg_len_core >= int(min_len)

            if keep_seg:
                core_seg = df.iloc[start:end].copy()

                if include_previous_segment:
                    prev_seg = get_previous_segment_by_iloc(df, start, n_prev_points=n_prev_points)
                    seg = pd.concat([prev_seg, core_seg], axis=0)
                else:
                    seg = core_seg

                cycles.append(seg)

                cycle_meta.append(
                    dict(
                        start_row=int(start),
                        end_row=int(end - 1),
                        n_points_total=int(len(seg)),
                        n_points_core=int(seg_len_core),
                        t_start=float(seg.index[0]),
                        t_end=float(seg.index[-1]),
                        duration=float(seg.index[-1] - seg.index[0]),
                        i_mean=float(np.mean(core_seg[i_col].to_numpy(dtype=float))),
                        i_min=float(np.min(core_seg[i_col].to_numpy(dtype=float))),
                        i_max=float(np.max(core_seg[i_col].to_numpy(dtype=float))),
                    )
                )
            start = None

    return cycles, cycle_meta


min_len_for_search = int(MIN_CYCLE_LEN) if USE_MIN_CYCLE_LEN else None

cycles, cycle_meta = find_discharging_cycles_with_meta(
    df,
    i_col=I_COL,
    tol_I=1e-9,
    min_len=min_len_for_search,
    include_previous_segment=INCLUDE_PREVIOUS_SEGMENT,
    n_prev_points=N_PREV_POINTS,
)

if min_len_for_search is None:
    print(f"Found {len(cycles)} discharge segments with no minimum-length filter")
else:
    print(f"Found {len(cycles)} discharge segments with min_len={min_len_for_search}")

if len(cycles) == 0:
    raise RuntimeError("No discharge segments found. Check sign, current column, or cycle filters.")

cycle_lengths_total = np.array([m["n_points_total"] for m in cycle_meta], dtype=int)
cycle_lengths_core = np.array([m["n_points_core"] for m in cycle_meta], dtype=int)
cycle_durations = np.array([m["duration"] for m in cycle_meta], dtype=float)

print("[CYCLE SUMMARY]")
print("  total length min/median/max :", int(np.min(cycle_lengths_total)), int(np.median(cycle_lengths_total)), int(np.max(cycle_lengths_total)))
print("  core  length min/median/max :", int(np.min(cycle_lengths_core)), int(np.median(cycle_lengths_core)), int(np.max(cycle_lengths_core)))
print("  duration min/median/max [s] :", float(np.min(cycle_durations)), float(np.median(cycle_durations)), float(np.max(cycle_durations)))

mode = str(CYCLE_MODE).strip().lower()

if mode == "index":
    chosen = int(CYCLE_INDEX)
    chosen = max(0, min(chosen, len(cycles) - 1))
    selection_note = f"index mode -> using cycle {chosen}"

elif mode == "random":
    if RANDOM_SEED is None:
        chosen = int(np.random.randint(0, len(cycles)))
        selection_note = (
            f"random mode -> RANDOM_SEED=None, selected cycle {chosen} "
            f"(this may change from run to run)"
        )
    else:
        rng = np.random.default_rng(int(RANDOM_SEED))
        chosen = int(rng.integers(low=0, high=len(cycles)))
        selection_note = (
            f"random mode -> seed={RANDOM_SEED}, selected cycle {chosen} "
            f"(reuse with CYCLE_MODE='index', CYCLE_INDEX={chosen})"
        )
else:
    raise ValueError("CYCLE_MODE must be 'index' or 'random'.")

cycle_df = cycles[chosen].copy()
meta = cycle_meta[chosen]

print(selection_note)
print(
    f"Selected cycle index: {chosen} | "
    f"n_points_total={meta['n_points_total']} | "
    f"n_points_core={meta['n_points_core']} | "
    f"duration={meta['duration']:.6g} s | "
    f"raw time range=[{meta['t_start']:.6g}, {meta['t_end']:.6g}]"
)
print(
    f"Current summary on selected core cycle: "
    f"mean={meta['i_mean']:.6g}, min={meta['i_min']:.6g}, max={meta['i_max']:.6g}"
)

plt.figure(figsize=(12, 4))
plt.plot(cycle_df.index.values, cycle_df[I_COL].values)
plt.grid(True)
plt.title(f"Selected discharge cycle (index={chosen}, mode={mode})")
plt.ylabel(I_COL)
plt.xlabel("time [s]")
plt.tight_layout()
plt.show()


# %% ======================================================
# CELL 5 — Prepare (t,U,Y): units, discharge-only, resample, trim
# =========================================================
def _sanity_report_current_units(I_raw: np.ndarray):
    I_as_A  = I_raw
    I_as_mA = I_raw * 1e-3
    print("[UNIT CHECK] If column is actually A:")
    print("  range [A]:", float(np.min(I_as_A)), float(np.max(I_as_A)),
          " median|I|:", float(np.median(np.abs(I_as_A))))
    print("[UNIT CHECK] If column is actually mA:")
    print("  range [A]:", float(np.min(I_as_mA)), float(np.max(I_as_mA)),
          " median|I|:", float(np.median(np.abs(I_as_mA))))


def _guess_I_in_amps(I_raw: np.ndarray) -> Tuple[np.ndarray, str]:
    med = float(np.nanmedian(np.abs(I_raw)))
    if med >= 1e-3:
        return I_raw, f"auto units: treating as A (median|I|={med:.6g})"
    if med >= 1.0:
        return I_raw * 1e-3, f"auto units: mA->A (median|I|={med:.6g} mA)"
    return I_raw, f"auto units: ambiguous, using A (median|I|={med:.6g})"


def resample_uniform(t: np.ndarray, u: np.ndarray, y: np.ndarray, dt: float):
    t0, t1 = float(t[0]), float(t[-1])
    tg = np.arange(t0, t1 + dt, dt, dtype=np.float64)
    u1 = np.interp(tg, t, u[:, 0]).reshape(-1, 1)
    y1 = np.interp(tg, t, y[:, 0]).reshape(-1, 1)
    return tg, u1, y1


t = cycle_df.index.to_numpy(dtype=np.float64).reshape(-1)
I_raw = cycle_df[I_COL].to_numpy(dtype=np.float64).reshape(-1)
V_raw = cycle_df[V_COL].to_numpy(dtype=np.float64).reshape(-1)

_sanity_report_current_units(I_raw)

if str(FORCE_UNITS).lower() == "auto":
    I_A, note = _guess_I_in_amps(I_raw)
    print(note)
elif str(FORCE_UNITS).lower() == "ma":
    I_A = I_raw * 1e-3
    print("FORCE_UNITS=mA => converting to A")
elif str(FORCE_UNITS).lower() == "a":
    I_A = I_raw
    print("FORCE_UNITS=A => using as A")
else:
    raise ValueError("FORCE_UNITS must be 'auto'|'mA'|'A'")

# make time strictly increasing
keep = np.ones_like(t, dtype=bool)
keep[1:] = t[1:] > t[:-1]
t, I_A, V_raw = t[keep], I_A[keep], V_raw[keep]

# rebase time
t = t - t[0]

if ENFORCE_DISCHARGE_ONLY:
    tol = 1e-12
    if RAW_DISCHARGE_SIGN == "negative":
        mask = I_A < -tol
    elif RAW_DISCHARGE_SIGN == "positive":
        mask = I_A > +tol
    else:
        raise ValueError("RAW_DISCHARGE_SIGN must be 'negative' or 'positive'")

    before = len(t)
    t, I_A, V_raw = t[mask], I_A[mask], V_raw[mask]
    if len(t) < 5:
        raise RuntimeError("After discharge-only filter, not enough points.")
    t = t - t[0]
    print(f"Discharge-only kept {len(t)}/{before} points")

if DROP_FIRST_N > 0:
    if len(t) <= DROP_FIRST_N + 3:
        raise RuntimeError("DROP_FIRST_N is too large for this selected cycle.")
    t = t[DROP_FIRST_N:]
    I_A = I_A[DROP_FIRST_N:]
    V_raw = V_raw[DROP_FIRST_N:]
    t = t - t[0]
    print(f"Dropped first {DROP_FIRST_N} samples after cycle extraction")

if V_REF == "none":
    V = V_raw
elif V_REF == "first":
    V = V_raw - float(V_raw[0])
elif V_REF == "mean":
    V = V_raw - float(np.mean(V_raw))
else:
    raise ValueError("V_REF must be none|first|mean")

t_np = t.reshape(-1)
U_np = I_A.reshape(-1, 1)
Y_np = V.reshape(-1, 1)

print("Prepared shapes (before resample/trim):", t_np.shape, U_np.shape, Y_np.shape)
print("I range [A]:", float(U_np.min()), float(U_np.max()),
      " median|I| [A]:", float(np.median(np.abs(U_np[:, 0]))))
print("V range [V]:", float(Y_np.min()), float(Y_np.max()))

plot_voltage(t_np, Y_np, title="Prepared (pre-resample/trim)")
plot_current_and_voltage(t_np, U_np[:, 0], Y_np[:, 0], title="Prepared signals (pre-resample/trim)")

med_abs_I = float(np.median(np.abs(U_np[:, 0])))
if med_abs_I < float(MIN_MED_ABS_I_A):
    print("\n[WARN] Current magnitude is very small -> dynamics may look stiff/flat.")
    print("       Double-check FORCE_UNITS and chosen cycle.\n")

if RESAMPLE and len(t_np) > 2:
    dt_med = float(np.median(np.diff(t_np)))
    if not (np.isfinite(dt_med) and dt_med > 0):
        raise RuntimeError("Bad dt_med; cannot resample.")
    t_np, U_np, Y_np = resample_uniform(t_np, U_np, Y_np, dt=dt_med)
    print("Resampled: dt_med =", dt_med, "len =", len(t_np))
else:
    print("RESAMPLE=False -> keeping original time grid")

if TMAX > 0:
    m = t_np <= float(TMAX)
    t_np, U_np, Y_np = t_np[m], U_np[m], Y_np[m]
    print("Trimmed to TMAX:", TMAX, "len =", len(t_np))

print("Prepared shapes (final):", t_np.shape, U_np.shape, Y_np.shape)
print("Final I range [A]:", float(U_np.min()), float(U_np.max()),
      " median|I| [A]:", float(np.median(np.abs(U_np[:, 0]))))
print("Final V range [V]:", float(Y_np.min()), float(Y_np.max()))

plot_voltage(t_np, Y_np, title="Prepared (final)")
plot_current_and_voltage(t_np, U_np[:, 0], Y_np[:, 0], title="Prepared signals (final)")

print("\n[DEBUG] Final prepared signal spans")
report_signal_span("time", t_np)
report_signal_span("current [A]", U_np[:, 0])
report_signal_span("voltage [V]", Y_np[:, 0])

line_rmse, line_coeff = simple_line_fit_rmse(t_np, Y_np[:, 0])
print(f"[DEBUG] Voltage vs. straight-line RMSE = {line_rmse:.6g} V")
print(f"[DEBUG] Linear trend coeffs = slope {line_coeff[0]:.6g}, intercept {line_coeff[1]:.6g}")


# %% ======================================================
# CELL 6 — Nominal 14-state proxy model + proxy signals
# =========================================================
IDX = {
    "cn": slice(0, 4),
    "cp": slice(4, 8),
    "ce": slice(8, 14),
    "cn_surf": 3,
    "cp_surf": 7,
    "ce_left": 8,
    "ce_right": 13,
}


@dataclass
class Config:
    R: float = 8.314462618
    F: float = 96485.33212
    T: float = 298.15
    T_ref: float = 298.15

    L1: float = 25e-6
    L2: float = 20e-6
    L3: float = 25e-6
    Rn: float = 5e-6
    Rp: float = 5e-6
    A: float = 1.0

    Dn: float = 1e-14
    Dp: float = 1e-14
    De: float = 7.23e-10
    eps: float = 0.30

    kappa_n_eff: float = 1.0
    kappa_s_eff: float = 1.0
    kappa_p_eff: float = 1.0

    a_s_n: float = 1.0e6
    a_s_p: float = 1.0e6
    k_n0: float = 2.0e-11
    k_p0: float = 2.0e-11
    use_arrhenius: bool = False
    Ea_n: float = 0.0
    Ea_p: float = 0.0

    lam_n: float = 0.0
    lam_p: float = 0.0

    csn_max: float = 3.1e4
    csp_max: float = 3.1e4

    ce0: float = 1000.0
    t_plus: float = 0.38
    k_f: float = 1.0
    R_ohm: float = 0.0
    use_dynamic_film: bool = False
    Rf: float = 0.0
    L_sei: float = 0.0
    kappa_sei: float = 1.0

    ce_is_deviation: bool = True
    discharge_positive: bool = False
    ln_orientation: str = "right_over_left"
    eta_mode: str = "diff"

    I_dyn: float = 2.0
    I_for_voltage: float = 2.0

    theta_guard: float = 1e-3
    I0_floor_p: float = 1e-2
    I0_floor_n: float = 1e-2
    bv_scale: float = 0.7
    N_series: int = N_SERIES_REAL


def build_An(cfg: Config) -> np.ndarray:
    s = cfg.Dn / (cfg.Rn ** 2)
    A = np.zeros((4, 4))
    A[0, 0], A[0, 1] = -24 * s, 24 * s
    A[1, 0], A[1, 1], A[1, 2] = 16 * s, -40 * s, 24 * s
    A[2, 1], A[2, 2], A[2, 3] = 16 * s, -40 * s, 24 * s
    A[3, 2], A[3, 3] = 16 * s, -16 * s
    return A


def build_Bn(cfg: Config) -> np.ndarray:
    sign = -1.0 if cfg.discharge_positive else +1.0
    b = np.zeros((4, 1))
    b[-1, 0] = sign * (6.0 / cfg.Rn) * (1.0 / (cfg.F * cfg.a_s_n * cfg.A * cfg.L1))
    return b


def build_Ap(cfg: Config) -> np.ndarray:
    s = cfg.Dp / (cfg.Rp ** 2)
    A = np.zeros((4, 4))
    A[0, 0], A[0, 1] = -24 * s, 24 * s
    A[1, 0], A[1, 1], A[1, 2] = 16 * s, -40 * s, 24 * s
    A[2, 1], A[2, 2], A[2, 3] = 16 * s, -40 * s, 24 * s
    A[3, 2], A[3, 3] = 16 * s, -16 * s
    return A


def build_Bp(cfg: Config) -> np.ndarray:
    sign = +1.0 if cfg.discharge_positive else -1.0
    b = np.zeros((4, 1))
    b[-1, 0] = sign * (6.0 / cfg.Rp) * (1.0 / (cfg.F * cfg.a_s_p * cfg.A * cfg.L3))
    return b


def build_Ae(cfg: Config) -> np.ndarray:
    K = cfg.De / cfg.eps
    Ae = np.zeros((6, 6))

    w_in = lambda L: K * 4.0 / (L ** 2)
    w_intf = lambda La, Lb: K * 16.0 / ((La + Lb) ** 2)

    w11 = w_in(cfg.L1)
    w12 = w_intf(cfg.L1, cfg.L2)
    w23 = w_in(cfg.L2)
    w34 = w_intf(cfg.L2, cfg.L3)
    w45 = w_in(cfg.L3)

    Ae[0, 0] = -(w11)
    Ae[0, 1] = +(w11)
    Ae[1, 0] = +(w11)
    Ae[1, 1] = -(w11 + w12)
    Ae[1, 2] = +(w12)
    Ae[2, 1] = +(w12)
    Ae[2, 2] = -(w12 + w23)
    Ae[2, 3] = +(w23)
    Ae[3, 2] = +(w23)
    Ae[3, 3] = -(w23 + w34)
    Ae[3, 4] = +(w34)
    Ae[4, 3] = +(w34)
    Ae[4, 4] = -(w34 + w45)
    Ae[4, 5] = +(w45)
    Ae[5, 4] = +(w45)
    Ae[5, 5] = -(w45)

    return Ae


def build_Be(cfg: Config) -> np.ndarray:
    b = np.zeros((6, 1))
    sign_left  = -1.0 if cfg.discharge_positive else +1.0
    sign_right = +1.0 if cfg.discharge_positive else -1.0
    s1 = sign_left  * (1.0 - cfg.t_plus) / (cfg.F * cfg.A * cfg.L1 * cfg.eps)
    s3 = sign_right * (1.0 - cfg.t_plus) / (cfg.F * cfg.A * cfg.L3 * cfg.eps)
    b[0, 0] = s1
    b[1, 0] = s1
    b[4, 0] = s3
    b[5, 0] = s3
    return b


def assemble_system(cfg: Config):
    An, Ap, Ae = build_An(cfg), build_Ap(cfg), build_Ae(cfg)
    Bn, Bp, Be = build_Bn(cfg), build_Bp(cfg), build_Be(cfg)
    Aglob = block_diag(An, Ap, Ae)
    Bglob = np.vstack([Bn, Bp, Be])
    S = ct.ss(Aglob, Bglob, np.eye(14), np.zeros((14, 1)))
    return S, Aglob, Bglob


def make_x0(cfg: Config, theta_n0=0.6, theta_p0=0.6, ce0=0.0):
    x0 = np.zeros(14, dtype=np.float64)
    x0[IDX["cn"]] = float(theta_n0) * cfg.csn_max
    x0[IDX["cp"]] = float(theta_p0) * cfg.csp_max
    x0[IDX["ce"]] = float(ce0)
    return x0


CFG = Config()
CFG.discharge_positive = False

Sx, A_nom_np, B_nom_np = assemble_system(CFG)
x0_nom = make_x0(CFG, theta_n0=XN0, theta_p0=XP0, ce0=CE0_DEV)

resp = ct.forced_response(Sx, T=t_np, U=U_np[:, 0], X0=x0_nom)
X_proxy = np.asarray(resp.states).T

xp_sig = (X_proxy[:, IDX["cp_surf"]] / CFG.csp_max).astype(np.float64)
xn_sig = (X_proxy[:, IDX["cn_surf"]] / CFG.csn_max).astype(np.float64)

ceL_raw_sig = X_proxy[:, IDX["ce_left"]].astype(np.float64)
ceR_raw_sig = X_proxy[:, IDX["ce_right"]].astype(np.float64)

if CFG.ce_is_deviation:
    ceL_sig = CFG.ce0 + ceL_raw_sig
    ceR_sig = CFG.ce0 + ceR_raw_sig
else:
    ceL_sig = ceL_raw_sig
    ceR_sig = ceR_raw_sig

xp_rng = float(xp_sig.max() - xp_sig.min())
xn_rng = float(xn_sig.max() - xn_sig.min())
ceL_rng = float(ceL_sig.max() - ceL_sig.min())
ceR_rng = float(ceR_sig.max() - ceR_sig.min())

print("[PROXY] xp range:", float(xp_sig.min()), float(xp_sig.max()), " span:", xp_rng)
print("[PROXY] xn range:", float(xn_sig.min()), float(xn_sig.max()), " span:", xn_rng)
print("[PROXY] ceL min/max:", float(np.min(ceL_sig)), float(np.max(ceL_sig)), " span:", ceL_rng)
print("[PROXY] ceR min/max:", float(np.min(ceR_sig)), float(np.max(ceR_sig)), " span:", ceR_rng)

if xp_rng < MIN_XP_RANGE or xn_rng < MIN_XN_RANGE:
    print("\n[WARN] Proxies barely move -> model may look stiff.")
    print("       This usually means wrong current units, wrong cycle, or too little excitation.\n")

plt.figure(figsize=(12, 6))
plt.subplot(3, 1, 1)
plt.plot(t_np, xp_sig)
plt.grid(True)
plt.ylabel("xp proxy")

plt.subplot(3, 1, 2)
plt.plot(t_np, xn_sig)
plt.grid(True)
plt.ylabel("xn proxy")

plt.subplot(3, 1, 3)
plt.plot(t_np, ceL_sig, label="ceL")
plt.plot(t_np, ceR_sig, label="ceR")
plt.grid(True)
plt.ylabel("ce proxy")
plt.xlabel("time [s]")
plt.legend()
plt.tight_layout()
plt.show()

print("\n[DEBUG] Proxy signal spans")
report_signal_span("xp proxy", xp_sig)
report_signal_span("xn proxy", xn_sig)
report_signal_span("ceL proxy", ceL_sig)
report_signal_span("ceR proxy", ceR_sig)

if xp_rng < 1e-7 and xn_rng < 1e-7:
    print("[HARD WARNING] Stage 2 cannot learn meaningful state-dependent surrogate from this run.")
    print("               Fix excitation/units first before trusting any optimizer result.")

ln_ce_ratio_sig = np.log(np.maximum(ceR_sig / np.maximum(ceL_sig, 1e-12), 1e-12))

print("\n[DEBUG] Feature leverage check")
report_signal_span("xp proxy", xp_sig)
report_signal_span("xn proxy", xn_sig)
report_signal_span("ln(ceR/ceL)", ln_ce_ratio_sig)

plt.figure(figsize=(12, 4))
plt.plot(t_np, ln_ce_ratio_sig)
plt.grid(True)
plt.xlabel("t [s]")
plt.ylabel("ln(ceR/ceL)")
plt.title("Electrolyte log feature on proxy trajectory")
plt.tight_layout()
plt.show()

if np.max(np.abs(ln_ce_ratio_sig)) < 1e-5:
    print("[INFO] ln(ceR/ceL) feature is effectively inactive on this dataset.")


# %% ======================================================
# CELL 7 — JAX helpers + A(thetaA), B(thetaB) parameterization
# =========================================================
def softplus(x):
    return jnp.log1p(jnp.exp(-jnp.abs(x))) + jnp.maximum(x, 0.0)


def pos(x, floor=1e-12):
    return softplus(x) + floor


def softplus_inv(y):
    y = float(max(y, 1e-12))
    if y > 20:
        return y
    return np.log(np.expm1(y))


def raw_from_pos(val, floor=1e-12):
    val = float(max(val, floor + 1e-12))
    return softplus_inv(val - floor)


def thetaA_nom_from_cfg(cfg: Config) -> np.ndarray:
    th1 = cfg.Dn / (cfg.Rn ** 2)
    th2 = cfg.Dp / (cfg.Rp ** 2)
    K = cfg.De / cfg.eps
    th3 = K / (cfg.L1 ** 2)
    th4 = K / ((cfg.L1 + cfg.L2) ** 2)
    th5 = K / (cfg.L2 ** 2)
    th6 = K / ((cfg.L2 + cfg.L3) ** 2)
    th7 = K / (cfg.L3 ** 2)
    return np.array([th1, th2, th3, th4, th5, th6, th7], dtype=np.float64)


def thetaB_nom_from_cfg(cfg: Config) -> np.ndarray:
    sign_n = -1.0 if cfg.discharge_positive else +1.0
    sign_p = +1.0 if cfg.discharge_positive else -1.0

    th8 = sign_n * (1.0 / cfg.Rn) * (1.0 / (cfg.F * cfg.a_s_n * cfg.A * cfg.L1))
    th9 = sign_p * (1.0 / cfg.Rp) * (1.0 / (cfg.F * cfg.a_s_p * cfg.A * cfg.L3))

    sign_left  = -1.0 if cfg.discharge_positive else +1.0
    sign_right = +1.0 if cfg.discharge_positive else -1.0
    th10 = sign_left  * (1.0 - cfg.t_plus) / (cfg.F * cfg.A * cfg.L1 * cfg.eps)
    th11 = sign_right * (1.0 - cfg.t_plus) / (cfg.F * cfg.A * cfg.L3 * cfg.eps)

    return np.array([th8, th9, th10, th11], dtype=np.float64)


thetaA_nom_np = thetaA_nom_from_cfg(CFG)
thetaB_nom_np = thetaB_nom_from_cfg(CFG)

thetaA_nom = jnp.array(thetaA_nom_np, dtype=DTYPE)
thetaB_nom = jnp.array(thetaB_nom_np, dtype=DTYPE)


@jax.jit
def build_A_from_thetaA(thetaA: jnp.ndarray) -> jnp.ndarray:
    th1, th2, th3, th4, th5, th6, th7 = thetaA

    An = jnp.array([
        [-24 * th1,  24 * th1,    0.0,    0.0],
        [ 16 * th1, -40 * th1, 24 * th1,    0.0],
        [   0.0,    16 * th1, -40 * th1, 24 * th1],
        [   0.0,      0.0,    16 * th1, -16 * th1],
    ], dtype=DTYPE)

    Ap = jnp.array([
        [-24 * th2,  24 * th2,    0.0,    0.0],
        [ 16 * th2, -40 * th2, 24 * th2,    0.0],
        [   0.0,    16 * th2, -40 * th2, 24 * th2],
        [   0.0,      0.0,    16 * th2, -16 * th2],
    ], dtype=DTYPE)

    Ae = jnp.array([
        [-4 * th3,                     4 * th3,                    0.0,                    0.0,                    0.0,      0.0],
        [ 4 * th3, -(4 * th3 + 16 * th4),                 16 * th4,                    0.0,                    0.0,      0.0],
        [ 0.0,                    16 * th4, -(16 * th4 + 4 * th5),                  4 * th5,                    0.0,      0.0],
        [ 0.0,                      0.0,                  4 * th5, -(4 * th5 + 16 * th6),                 16 * th6,     0.0],
        [ 0.0,                      0.0,                    0.0,                  16 * th6, -(16 * th6 + 4 * th7),  4 * th7],
        [ 0.0,                      0.0,                    0.0,                    0.0,                  4 * th7, -4 * th7],
    ], dtype=DTYPE)

    A = jnp.zeros((14, 14), dtype=DTYPE)
    A = A.at[0:4, 0:4].set(An)
    A = A.at[4:8, 4:8].set(Ap)
    A = A.at[8:14, 8:14].set(Ae)
    return A


@jax.jit
def build_B_from_thetaB(thetaB: jnp.ndarray) -> jnp.ndarray:
    th8, th9, th10, th11 = thetaB
    B = jnp.zeros((14, 1), dtype=DTYPE)
    B = B.at[3, 0].set(6.0 * th8)
    B = B.at[7, 0].set(6.0 * th9)
    B = B.at[8, 0].set(th10)
    B = B.at[9, 0].set(th10)
    B = B.at[12, 0].set(th11)
    B = B.at[13, 0].set(th11)
    return B


# %% ======================================================
# CELL 8 — Additive polynomial surrogate builder
# =========================================================
def make_additive_poly_surrogate_fns(
    cfg: Config,
    deg: int,
    xp_ref: float,
    xn_ref: float,
    xp_scale: float,
    xn_scale: float,
    use_ln_feature: bool = False,
):
    xp_powers = jnp.arange(1, deg + 1, dtype=DTYPE)
    xn_powers = jnp.arange(1, deg + 1, dtype=DTYPE)

    xp_ref_j = DTYPE(float(xp_ref))
    xn_ref_j = DTYPE(float(xn_ref))
    xp_scale_j = DTYPE(float(max(xp_scale, 1e-12)))
    xn_scale_j = DTYPE(float(max(xn_scale, 1e-12)))

    @jax.jit
    def additive_features_from_x(x14: jnp.ndarray) -> jnp.ndarray:
        xp = jnp.clip(x14[IDX["cp_surf"]] / DTYPE(cfg.csp_max), 1e-9, 1.0 - 1e-9)
        xn = jnp.clip(x14[IDX["cn_surf"]] / DTYPE(cfg.csn_max), 1e-9, 1.0 - 1e-9)

        dxp = (xp - xp_ref_j) / xp_scale_j
        dxn = (xn - xn_ref_j) / xn_scale_j

        xp_feats = dxp ** xp_powers
        xn_feats = dxn ** xn_powers

        return jnp.concatenate(
            [jnp.ones((1,), dtype=DTYPE), xp_feats, xn_feats],
            axis=0
        )

    @jax.jit
    def ln_ce_ratio_from_x(x14: jnp.ndarray) -> jnp.ndarray:
        ceL_raw = x14[IDX["ce_left"]]
        ceR_raw = x14[IDX["ce_right"]]

        ceL = (DTYPE(cfg.ce0) + ceL_raw) if cfg.ce_is_deviation else ceL_raw
        ceR = (DTYPE(cfg.ce0) + ceR_raw) if cfg.ce_is_deviation else ceR_raw

        ceL = jnp.maximum(ceL, 1e-12)
        ceR = jnp.maximum(ceR, 1e-12)

        ln_arg = (ceR / ceL) if (cfg.ln_orientation == "right_over_left") else (ceL / ceR)
        return jnp.log(jnp.maximum(ln_arg, 1e-12))

    n_base = 1 + deg + deg
    n_thetaZ = n_base + (1 if use_ln_feature else 0)

    def unpack_thetaZ(thetaZ: jnp.ndarray):
        thetaZ = thetaZ.reshape(-1)
        theta_base = thetaZ[:n_base]

        if use_ln_feature:
            k_ln = thetaZ[n_base]
            return theta_base, k_ln
        return theta_base, None

    @jax.jit
    def zhat_from_thetaZ(x14: jnp.ndarray, thetaZ: jnp.ndarray) -> jnp.ndarray:
        thetaZ = jnp.clip(thetaZ.reshape(-1), -DTYPE(CLIP_RAW_Z), DTYPE(CLIP_RAW_Z))
        phi = additive_features_from_x(x14)
        theta_base, k_ln = unpack_thetaZ(thetaZ)

        z = jnp.dot(phi, theta_base)

        if use_ln_feature:
            lnratio = ln_ce_ratio_from_x(x14)
            z = z + k_ln * lnratio

        return z

    def make_thetaZ0_from_voltage(Y_np: np.ndarray) -> np.ndarray:
        thetaZ0 = np.zeros(n_thetaZ, dtype=np.float64)
        thetaZ0[0] = float(np.mean(Y_np[:, 0]))
        return thetaZ0

    meta = dict(
        deg=deg,
        n_thetaZ=n_thetaZ,
        use_ln_feature=use_ln_feature,
        xp_ref=xp_ref,
        xn_ref=xn_ref,
        xp_scale=xp_scale,
        xn_scale=xn_scale,
        structure="c0 + sum a_k*dxp^k + sum b_k*dxn^k [+ k_ln ln(ceR/ceL)]",
    )
    return make_thetaZ0_from_voltage, zhat_from_thetaZ, meta


# %% ======================================================
# CELL 9 — Stage 2: fit surrogate + R0, A/B fixed nominal
# =========================================================
print("\n==============================")
print("Stage 2: Fit surrogate + R0 with A and B fixed to nominal proxy dynamics")
print("==============================")

if float(np.median(np.abs(U_np[:, 0]))) < MIN_MED_ABS_I_A:
    raise RuntimeError(
        "Median |I| is too small for useful identification. "
        "Likely wrong current units. Try FORCE_UNITS='A'."
    )

if (xp_rng < 1e-7) and (xn_rng < 1e-7):
    raise RuntimeError(
        "xp/xn proxy motion is essentially zero. "
        "Do not fit Stage 2 yet. Fix current units or choose a better cycle."
    )

xp_ref = float(np.mean(xp_sig))
xn_ref = float(np.mean(xn_sig))

xp_scale = float(max(np.std(xp_sig), 1e-6))
xn_scale = float(max(np.std(xn_sig), 1e-6))

print("\n[DEBUG] Center-scale settings")
print("xp_ref   =", xp_ref)
print("xn_ref   =", xn_ref)
print("xp_scale =", xp_scale)
print("xn_scale =", xn_scale)

make_thetaZ0, zhat_from_thetaZ_stage2, metaZ = make_additive_poly_surrogate_fns(
    CFG,
    POLY_DEG,
    xp_ref=xp_ref,
    xn_ref=xn_ref,
    xp_scale=xp_scale,
    xn_scale=xn_scale,
    use_ln_feature=USE_LN_FEATURE,
)

dxp_dbg = (xp_sig - xp_ref) / xp_scale
dxn_dbg = (xn_sig - xn_ref) / xn_scale

Phi_cols = [np.ones_like(dxp_dbg)]
for k in range(1, POLY_DEG + 1):
    Phi_cols.append(dxp_dbg ** k)
for k in range(1, POLY_DEG + 1):
    Phi_cols.append(dxn_dbg ** k)

if USE_LN_FEATURE:
    Phi_cols.append(np.log(np.maximum(ceR_sig / np.maximum(ceL_sig, 1e-12), 1e-12)))

Phi = np.column_stack(Phi_cols)

print("\n[DEBUG] Stage-2 feature matrix diagnostics")
print("Phi shape:", Phi.shape)
col_std = np.std(Phi, axis=0)
for i, s in enumerate(col_std):
    print(f"  feature[{i}] std = {s:.6e}")

try:
    cond_phi = np.linalg.cond(Phi)
    print("Feature matrix cond(Phi):", cond_phi)
except Exception as e:
    print("Could not compute cond(Phi):", e)

thetaZ0 = make_thetaZ0(Y_np)
rawR0_0 = raw_from_pos(0.05, floor=1e-12)

params_stage2_init = [
    thetaZ0.astype(np.float64),
    np.array([rawR0_0], dtype=np.float64),
]
nx_stage2 = 14
x0_stage2 = X_proxy[0].astype(np.float64)

A_nom = jnp.array(A_nom_np, dtype=DTYPE)
B_nom = jnp.array(B_nom_np, dtype=DTYPE)

@jax.jit
def state_fcn_stage2(x, u, t, params):
    I = u[0]
    return A_nom @ x + (B_nom[:, 0] * I)

@jax.jit
def output_fcn_stage2(x, u, t, params):
    thetaZ, rawR0 = params
    I = u[0]
    Z = zhat_from_thetaZ_stage2(x, thetaZ)
    R0 = pos(rawR0[0], 1e-12)
    Vhat = Z - DTYPE(CFG.N_series) * I * R0
    return jnp.array([Vhat], dtype=DTYPE)

V_mean_baseline = np.full_like(Y_np[:, 0], np.mean(Y_np[:, 0]))
report_fit("Baseline mean-voltage", Y_np, V_mean_baseline.reshape(-1, 1))

coef_lin = np.polyfit(t_np, Y_np[:, 0], 1)
V_line_baseline = np.polyval(coef_lin, t_np)
report_fit("Baseline linear-in-time", Y_np, V_line_baseline.reshape(-1, 1))

cmp_ls = compare_ls_solvers(Phi, Y_np[:, 0], ridge=1e-12, label="Static LS")
coef_ls_manual = cmp_ls["beta_qr"]
coef_ls_np = cmp_ls["beta_lstsq"]

Y_ls_manual = Phi @ coef_ls_manual
Y_ls_np = Phi @ coef_ls_np

print("\n[DEBUG] Static LS on surrogate features (manual QR)")
report_fit("Static LS feature fit (manual QR)", Y_np, Y_ls_manual.reshape(-1, 1))

print("\n[DEBUG] Static LS on surrogate features (np.linalg.lstsq)")
report_fit("Static LS feature fit (np.linalg.lstsq)", Y_np, Y_ls_np.reshape(-1, 1))

Phi_plus_R = np.column_stack([Phi, -U_np[:, 0]])

cmp_lsr = compare_ls_solvers(Phi_plus_R, Y_np[:, 0], ridge=1e-12, label="Static LS+R")
coef_ls_r_manual = cmp_lsr["beta_qr"]
coef_ls_r_np = cmp_lsr["beta_lstsq"]

Y_ls_r_manual = Phi_plus_R @ coef_ls_r_manual
Y_ls_r_np = Phi_plus_R @ coef_ls_r_np

print("\n[DEBUG] Static LS+R (manual QR)")
report_fit("Static LS+R fit (manual QR)", Y_np, Y_ls_r_manual.reshape(-1, 1))

print("\n[DEBUG] Static LS+R (np.linalg.lstsq)")
report_fit("Static LS+R fit (np.linalg.lstsq)", Y_np, Y_ls_r_np.reshape(-1, 1))

R0_ls_manual = float(coef_ls_r_manual[-1])
R0_ls_np = float(coef_ls_r_np[-1])

print("\n[DEBUG] LS+R inferred ohmic coefficient")
print(f"  R0_manual_QR   = {R0_ls_manual:.12g}")
print(f"  R0_np_lstsq    = {R0_ls_np:.12g}")
print(f"  |difference|   = {abs(R0_ls_manual - R0_ls_np):.6e}")

plot_voltage_with_baselines(
    t_np,
    Y_np[:, 0],
    {
        "Linear baseline": V_line_baseline,
        "Static LS (manual)": Y_ls_manual,
        "Static LS+R (manual)": Y_ls_r_manual,
    },
    title="Stage-2 baseline and manual LS comparisons (physics-only)"
)

thetaZ_min = -10.0 * np.ones_like(thetaZ0, dtype=np.float64)
thetaZ_max =  10.0 * np.ones_like(thetaZ0, dtype=np.float64)

thetaZ_min[0] = float(np.min(Y_np[:, 0]) - 0.5)
thetaZ_max[0] = float(np.max(Y_np[:, 0]) + 0.5)

rawR0_min = np.array([raw_from_pos(1e-4, floor=1e-12)], dtype=np.float64)
rawR0_max = np.array([raw_from_pos(5.0,  floor=1e-12)], dtype=np.float64)

model_stage2 = CTModel(nx_stage2, 1, 1, state_fcn=state_fcn_stage2, output_fcn=output_fcn_stage2)
model_stage2.init(params=params_stage2_init, x0=x0_stage2)

try:
    model_stage2.loss(rho_x0=0.0, rho_th=RHO_TH_STAGE2, train_x0=False, xsat=1e9)
except TypeError:
    model_stage2.loss(rho_x0=0.0, rho_th=RHO_TH_STAGE2)

lbfgs_epochs = int(LBFGS_EPOCHS) if USE_LBFGS else 0

model_stage2.optimization(
    adam_epochs=int(ADAM_EPOCHS_STAGE2),
    lbfgs_epochs=lbfgs_epochs,
    adam_eta=float(ADAM_ETA_STAGE2),
    params_min=[thetaZ_min, rawR0_min],
    params_max=[thetaZ_max, rawR0_max],
)

dt = float(t_np[1] - t_np[0]) if len(t_np) > 1 else 1.0
model_stage2.integration_options(
    ode_solver=diffrax.Tsit5(),
    dt0=dt / float(DT0_DIV),
    max_steps=int(MAX_STEPS),
    stepsize_controller=diffrax.PIDController(
        rtol=float(SOLVER_RTOL),
        atol=float(SOLVER_ATOL)
    ),
)

Y0_stage2, _ = model_stage2.predict(model_stage2.x0, U_np, t_np)
report_fit("Stage 2 pre-fit", Y_np, Y0_stage2)

model_stage2.fit(Y_np, U_np, t_np)

Yhat_stage2, Xhat_stage2 = model_stage2.predict(model_stage2.x0, U_np, t_np)
report_fit("Stage 2 post-fit", Y_np, Yhat_stage2)
summarize_err(Y_np, Yhat_stage2, name=f"Stage 2 (train, deg={POLY_DEG})")

plot_voltage(t_np, Y_np, Yhat_stage2, title=f"Stage 2: physics-only surrogate fit (deg={POLY_DEG})")
plot_voltage_with_baselines(
    t_np,
    Y_np[:, 0],
    {
        "Static LS (manual)": Y_ls_manual,
        "Static LS+R (manual)": Y_ls_r_manual,
        "CT Stage 2": Yhat_stage2[:, 0],
    },
    title="Measured vs manual LS vs CT Stage 2"
)
plot_residuals(t_np, Y_np[:, 0], Yhat_stage2[:, 0], title="Stage 2 residuals")

thetaZ_hat = np.asarray(model_stage2.params[0]).reshape(-1)
rawR0_hat  = float(np.asarray(model_stage2.params[1]).reshape(-1)[0])
R0_hat     = float(np.asarray(pos(jnp.array(rawR0_hat, dtype=DTYPE), 1e-12)))

print("\nStage 2 learned:")
print("  surrogate structure:", metaZ["structure"])
print(f"  POLY_DEG={metaZ['deg']}, USE_LN_FEATURE={metaZ['use_ln_feature']}")
print("  thetaZ_hat shape:", thetaZ_hat.shape, " (n_thetaZ =", metaZ["n_thetaZ"], ")")
print("  R0_hat:", R0_hat)

print("\n[DEBUG] Stage 2 diagnostics")
report_signal_span("Measured V", Y_np[:, 0])
report_signal_span("Predicted V", Yhat_stage2[:, 0])
report_signal_span("Residual", Yhat_stage2[:, 0] - Y_np[:, 0])

corr_v = float(np.corrcoef(Y_np[:, 0], Yhat_stage2[:, 0])[0, 1]) if len(Y_np) > 2 else np.nan
print("Voltage correlation(meas,pred):", corr_v)

pred_span = float(np.max(Yhat_stage2[:, 0]) - np.min(Yhat_stage2[:, 0]))
meas_span = float(np.max(Y_np[:, 0]) - np.min(Y_np[:, 0]))

if pred_span < 0.05 * meas_span:
    print("\n[HARD WARNING] Stage 2 prediction span is too small relative to measured span.")
    print("               This fit is effectively flat and should not be used to start Stage 3.")


# %% ======================================================
# CELL 10 — Degree sweep summary or optional sweep code
# =========================================================
def run_stage2_for_degree_real(
    deg: int,
    use_ln_feature: bool = False,
    adam_epochs: int = 600,
):
    xp_ref_d = float(np.mean(xp_sig))
    xn_ref_d = float(np.mean(xn_sig))
    xp_scale_d = float(max(np.std(xp_sig), 1e-6))
    xn_scale_d = float(max(np.std(xn_sig), 1e-6))

    make_thetaZ0_d, zhat_from_thetaZ_d, meta = make_additive_poly_surrogate_fns(
        CFG,
        deg,
        xp_ref=xp_ref_d,
        xn_ref=xn_ref_d,
        xp_scale=xp_scale_d,
        xn_scale=xn_scale_d,
        use_ln_feature=use_ln_feature,
    )

    dxp_d = (xp_sig - xp_ref_d) / xp_scale_d
    dxn_d = (xn_sig - xn_ref_d) / xn_scale_d

    Phi_cols_d = [np.ones_like(dxp_d)]
    for k in range(1, deg + 1):
        Phi_cols_d.append(dxp_d ** k)
    for k in range(1, deg + 1):
        Phi_cols_d.append(dxn_d ** k)

    if use_ln_feature:
        Phi_cols_d.append(np.log(np.maximum(ceR_sig / np.maximum(ceL_sig, 1e-12), 1e-12)))

    Phi_d = np.column_stack(Phi_cols_d)

    beta_ls_d = solve_ls_qr(Phi_d, Y_np[:, 0])
    Y_ls_d = Phi_d @ beta_ls_d

    Phi_plus_R_d = np.column_stack([Phi_d, -U_np[:, 0]])
    beta_lsr_d = solve_ls_qr(Phi_plus_R_d, Y_np[:, 0])
    Y_ls_r_d = Phi_plus_R_d @ beta_lsr_d

    thetaZ0_d = make_thetaZ0_d(Y_np)
    rawR0_0_d = raw_from_pos(0.05, floor=1e-12)

    params0_d = [
        thetaZ0_d.astype(np.float64),
        np.array([rawR0_0_d], dtype=np.float64),
    ]

    x0_d = X_proxy[0].astype(np.float64)

    @jax.jit
    def state_fcn_d(x, u, t, params):
        I = u[0]
        return A_nom @ x + (B_nom[:, 0] * I)

    @jax.jit
    def output_fcn_d(x, u, t, params):
        thetaZ, rawR0 = params
        I = u[0]
        Z = zhat_from_thetaZ_d(x, thetaZ)
        R0 = pos(rawR0[0], 1e-12)
        Vhat = Z - DTYPE(CFG.N_series) * I * R0
        return jnp.array([Vhat], dtype=DTYPE)

    thetaZ_min_d = -10.0 * np.ones_like(thetaZ0_d, dtype=np.float64)
    thetaZ_max_d =  10.0 * np.ones_like(thetaZ0_d, dtype=np.float64)
    thetaZ_min_d[0] = float(np.min(Y_np[:, 0]) - 0.5)
    thetaZ_max_d[0] = float(np.max(Y_np[:, 0]) + 0.5)

    rawR0_min_d = np.array([raw_from_pos(1e-4, floor=1e-12)], dtype=np.float64)
    rawR0_max_d = np.array([raw_from_pos(5.0,  floor=1e-12)], dtype=np.float64)

    m = CTModel(14, 1, 1, state_fcn=state_fcn_d, output_fcn=output_fcn_d)
    m.init(params=params0_d, x0=x0_d)

    try:
        m.loss(rho_x0=0.0, rho_th=RHO_TH_STAGE2, train_x0=False, xsat=1e9)
    except TypeError:
        m.loss(rho_x0=0.0, rho_th=RHO_TH_STAGE2)

    lbfgs_epochs = int(LBFGS_EPOCHS) if USE_LBFGS else 0

    m.optimization(
        adam_epochs=int(adam_epochs),
        lbfgs_epochs=lbfgs_epochs,
        adam_eta=float(ADAM_ETA_STAGE2),
        params_min=[thetaZ_min_d, rawR0_min_d],
        params_max=[thetaZ_max_d, rawR0_max_d],
    )

    m.integration_options(
        ode_solver=diffrax.Tsit5(),
        dt0=dt / float(DT0_DIV),
        max_steps=int(MAX_STEPS),
        stepsize_controller=diffrax.PIDController(
            rtol=float(SOLVER_RTOL),
            atol=float(SOLVER_ATOL)
        ),
    )

    m.fit(Y_np, U_np, t_np)
    Yhat_d, _ = m.predict(m.x0, U_np, t_np)

    ct_metrics = summarize_err(Y_np, Yhat_d, name=f"CT Stage 2 train (deg={deg})")
    ls_metrics = summarize_err(Y_np, Y_ls_d.reshape(-1, 1), name=f"Static LS train (deg={deg})")
    lsr_metrics = summarize_err(Y_np, Y_ls_r_d.reshape(-1, 1), name=f"Static LS+R train (deg={deg})")

    return dict(
        deg=deg,
        n_thetaZ=meta["n_thetaZ"],
        ct=ct_metrics,
        ls=ls_metrics,
        lsr=lsr_metrics,
        structure=meta["structure"],
    )


if RUN_DEGREE_SWEEP:
    print("\n" + "=" * 90)
    print("RUNNING DEGREE SWEEP")
    print("=" * 90)

    sweep = []
    for d in DEGREE_SWEEP:
        print("\n" + "#" * 80)
        print(f"SWEEP DEGREE = {d}")
        print("#" * 80)
        sweep.append(
            run_stage2_for_degree_real(
                deg=d,
                use_ln_feature=USE_LN_FEATURE,
                adam_epochs=600,
            )
        )

    print("\n" + "=" * 90)
    print("SWEEP SUMMARY (real-data Stage 2, physics-only)")
    print("=" * 90)
    for r in sweep:
        print(
            f"deg={r['deg']:2d} | "
            f"CT RMSE={r['ct']['rmse']:.3e} | "
            f"LS RMSE={r['ls']['rmse']:.3e} | "
            f"LS+R RMSE={r['lsr']['rmse']:.3e} | "
            f"CT MAX={r['ct']['max_abs']:.3e}"
        )
else:
    print("\n[INFO] RUN_DEGREE_SWEEP=False, skipping Cell 10.")
    print("[INFO] Based on your previous sweep:")
    print("       - practical sweet-spot range: degree 14 to 17")
    print("       - chosen operating degree now:", POLY_DEG)
    print("       - now we will judge it on several/all cycles, not only one cycle")


# %% ======================================================
# CELL 11 — Stage 3a: fit A,B only, surrogate frozen from Stage 2
# =========================================================
if not RUN_STAGE3:
    print("\n[INFO] RUN_STAGE3=False, skipping Stage 3a.")
else:
    print("\n==============================")
    print("Stage 3a: Fit A,B with Stage-2 surrogate frozen")
    print("==============================")

    thetaA_nom_init = thetaA_nom_np.copy()
    thetaB_nom_init = thetaB_nom_np.copy()

    rawA_init = np.log(np.maximum(thetaA_nom_init, 1e-12))
    rawB_init = thetaB_nom_init.copy()
    rawAB_init = np.concatenate([rawA_init, rawB_init]).astype(np.float64)

    nx_stage3a = 14
    x0_stage3a = X_proxy[0].astype(np.float64)

    @jax.jit
    def unpack_ab(raw_ab: jnp.ndarray):
        raw_ab = raw_ab.reshape(-1)
        rawA = jnp.clip(raw_ab[0:7], -DTYPE(CLIP_RAW_A), DTYPE(CLIP_RAW_A))
        rawB = jnp.clip(raw_ab[7:11], -DTYPE(CLIP_RAW_B), DTYPE(CLIP_RAW_B))
        thetaA = jnp.exp(rawA)
        thetaB = rawB
        return thetaA, thetaB

    @jax.jit
    def state_fcn_stage3a(x, u, t, params):
        I = u[0]
        (raw_ab,) = params
        thetaA, thetaB = unpack_ab(raw_ab)
        A = build_A_from_thetaA(thetaA)
        B = build_B_from_thetaB(thetaB)
        return A @ x + (B[:, 0] * I)

    @jax.jit
    def output_fcn_stage3a(x, u, t, params):
        I = u[0]
        Z = zhat_from_thetaZ_stage2(x, jnp.array(thetaZ_hat, dtype=DTYPE))
        R0 = pos(jnp.array(rawR0_hat, dtype=DTYPE), 1e-12)
        Vhat = Z - DTYPE(CFG.N_series) * I * R0
        return jnp.array([Vhat], dtype=DTYPE)

    model_stage3a = CTModel(14, 1, 1, state_fcn=state_fcn_stage3a, output_fcn=output_fcn_stage3a)
    model_stage3a.init(params=[rawAB_init], x0=x0_stage3a)

    try:
        model_stage3a.loss(rho_x0=0.0, rho_th=RHO_TH_STAGE3A, train_x0=False, xsat=1e9)
    except TypeError:
        model_stage3a.loss(rho_x0=0.0, rho_th=RHO_TH_STAGE3A)

    model_stage3a.optimization(
        adam_epochs=int(ADAM_EPOCHS_STAGE3A),
        lbfgs_epochs=0,
        adam_eta=float(ADAM_ETA_STAGE3A),
    )

    model_stage3a.integration_options(
        ode_solver=diffrax.Tsit5(),
        dt0=dt / float(DT0_DIV),
        max_steps=int(MAX_STEPS),
        stepsize_controller=diffrax.PIDController(rtol=float(SOLVER_RTOL), atol=float(SOLVER_ATOL)),
    )

    Y0_stage3a, _ = model_stage3a.predict(model_stage3a.x0, U_np, t_np)
    report_fit("Stage 3a pre-fit", Y_np, Y0_stage3a)

    model_stage3a.fit(Y_np, U_np, t_np)

    Yhat_stage3a, _ = model_stage3a.predict(model_stage3a.x0, U_np, t_np)
    report_fit("Stage 3a post-fit", Y_np, Yhat_stage3a)
    summarize_err(Y_np, Yhat_stage3a, name=f"Stage 3a (train, deg={POLY_DEG})")
    plot_voltage(t_np, Y_np, Yhat_stage3a, title=f"Stage 3a fit (deg={POLY_DEG})")

    rawAB_hat = np.asarray(model_stage3a.params[0]).reshape(-1)
    rawA_hat_stage3a = rawAB_hat[0:7].copy()
    rawB_hat_stage3a = rawAB_hat[7:11].copy()


# %% ======================================================
# CELL 12 — Stage 3b: unfreeze everything and refine full model
# =========================================================
if not RUN_STAGE3:
    print("\n[INFO] RUN_STAGE3=False, skipping Stage 3b.")
else:
    print("\n==============================")
    print("Stage 3b: Unfreeze everything and refine FULL model")
    print("==============================")

    N_THETAZ = int(thetaZ_hat.shape[0])

    raw_init_full = np.concatenate([
        rawA_hat_stage3a.astype(np.float64),
        rawB_hat_stage3a.astype(np.float64),
        thetaZ_hat.astype(np.float64),
        np.array([rawR0_hat], dtype=np.float64),
    ]).astype(np.float64)

    def unpack_full(raw_theta: jnp.ndarray):
        raw_theta = raw_theta.reshape(-1)
        rawA   = jnp.clip(raw_theta[0:7], -DTYPE(CLIP_RAW_A), DTYPE(CLIP_RAW_A))
        rawB   = jnp.clip(raw_theta[7:11], -DTYPE(CLIP_RAW_B), DTYPE(CLIP_RAW_B))
        thetaZ = jnp.clip(raw_theta[11:11+N_THETAZ], -DTYPE(CLIP_RAW_Z), DTYPE(CLIP_RAW_Z))
        rawR0  = raw_theta[11+N_THETAZ]
        thetaA = jnp.exp(rawA)
        thetaB = rawB
        return thetaA, thetaB, thetaZ, rawR0

    @jax.jit
    def state_fcn_stage3b(x, u, t, params):
        I = u[0]
        (raw_theta,) = params
        thetaA, thetaB, _, _ = unpack_full(raw_theta)
        A = build_A_from_thetaA(thetaA)
        B = build_B_from_thetaB(thetaB)
        return A @ x + (B[:, 0] * I)

    @jax.jit
    def output_fcn_stage3b(x, u, t, params):
        I = u[0]
        (raw_theta,) = params
        _, _, thetaZ, rawR0 = unpack_full(raw_theta)
        Z = zhat_from_thetaZ_stage2(x, thetaZ)
        R0 = pos(rawR0, 1e-12)
        Vhat = Z - DTYPE(CFG.N_series) * I * R0
        return jnp.array([Vhat], dtype=DTYPE)

    model_stage3b = CTModel(14, 1, 1, state_fcn=state_fcn_stage3b, output_fcn=output_fcn_stage3b)
    model_stage3b.init(params=[raw_init_full], x0=X_proxy[0].astype(np.float64))

    try:
        model_stage3b.loss(rho_x0=0.0, rho_th=RHO_TH_STAGE3B, train_x0=False, xsat=1e9)
    except TypeError:
        model_stage3b.loss(rho_x0=0.0, rho_th=RHO_TH_STAGE3B)

    model_stage3b.optimization(
        adam_epochs=int(ADAM_EPOCHS_STAGE3B),
        lbfgs_epochs=0,
        adam_eta=float(ADAM_ETA_STAGE3B),
    )

    model_stage3b.integration_options(
        ode_solver=diffrax.Tsit5(),
        dt0=dt / float(DT0_DIV),
        max_steps=int(MAX_STEPS),
        stepsize_controller=diffrax.PIDController(rtol=float(SOLVER_RTOL), atol=float(SOLVER_ATOL)),
    )

    Y0_stage3b, _ = model_stage3b.predict(model_stage3b.x0, U_np, t_np)
    report_fit("Stage 3b pre-fit", Y_np, Y0_stage3b)

    model_stage3b.fit(Y_np, U_np, t_np)

    Yhat_stage3b, _ = model_stage3b.predict(model_stage3b.x0, U_np, t_np)
    report_fit("Stage 3b post-fit", Y_np, Yhat_stage3b)
    summarize_err(Y_np, Yhat_stage3b, name=f"Stage 3b (train, deg={POLY_DEG})")
    plot_voltage(t_np, Y_np, Yhat_stage3b, title=f"Stage 3b fit (deg={POLY_DEG})")
    plot_residuals(t_np, Y_np[:, 0], Yhat_stage3b[:, 0], title="Stage 3b residuals")

    raw_full_hat = np.asarray(model_stage3b.params[0]).reshape(-1)
    thetaA_hat_stage3b = np.exp(np.clip(raw_full_hat[0:7], -CLIP_RAW_A, CLIP_RAW_A))
    thetaB_hat_stage3b = raw_full_hat[7:11].copy()
    thetaZ_hat_stage3b = raw_full_hat[11:11+N_THETAZ].copy()
    rawR0_hat_stage3b = float(raw_full_hat[11+N_THETAZ])
    R0_hat_stage3b = float(np.asarray(pos(jnp.array(rawR0_hat_stage3b, dtype=DTYPE), 1e-12)))


# %% ======================================================
# CELL 13 — Reusable cycle pipeline helpers
# =========================================================
def prepare_cycle_data(
    cycle_df_local: pd.DataFrame,
    i_col: str = I_COL,
    v_col: str = V_COL,
    force_units: str = FORCE_UNITS,
    v_ref: str = V_REF,
    resample: bool = RESAMPLE,
    enforce_discharge_only: bool = ENFORCE_DISCHARGE_ONLY,
    raw_discharge_sign: str = RAW_DISCHARGE_SIGN,
    tmax: float = TMAX,
    drop_first_n: int = DROP_FIRST_N,
):
    t = cycle_df_local.index.to_numpy(dtype=np.float64).reshape(-1)
    I_raw = cycle_df_local[i_col].to_numpy(dtype=np.float64).reshape(-1)
    V_raw = cycle_df_local[v_col].to_numpy(dtype=np.float64).reshape(-1)

    if str(force_units).lower() == "auto":
        I_A, _ = _guess_I_in_amps(I_raw)
    elif str(force_units).lower() == "ma":
        I_A = I_raw * 1e-3
    elif str(force_units).lower() == "a":
        I_A = I_raw
    else:
        raise ValueError("FORCE_UNITS must be 'auto'|'mA'|'A'")

    keep = np.ones_like(t, dtype=bool)
    keep[1:] = t[1:] > t[:-1]
    t, I_A, V_raw = t[keep], I_A[keep], V_raw[keep]

    t = t - t[0]

    if enforce_discharge_only:
        tol = 1e-12
        if raw_discharge_sign == "negative":
            mask = I_A < -tol
        elif raw_discharge_sign == "positive":
            mask = I_A > +tol
        else:
            raise ValueError("RAW_DISCHARGE_SIGN must be 'negative' or 'positive'")
        t, I_A, V_raw = t[mask], I_A[mask], V_raw[mask]
        if len(t) < 5:
            raise RuntimeError("After discharge-only filter, not enough points.")
        t = t - t[0]

    if drop_first_n > 0:
        if len(t) <= drop_first_n + 3:
            raise RuntimeError("DROP_FIRST_N is too large for this selected cycle.")
        t = t[drop_first_n:]
        I_A = I_A[drop_first_n:]
        V_raw = V_raw[drop_first_n:]
        t = t - t[0]

    if v_ref == "none":
        V = V_raw
    elif v_ref == "first":
        V = V_raw - float(V_raw[0])
    elif v_ref == "mean":
        V = V_raw - float(np.mean(V_raw))
    else:
        raise ValueError("V_REF must be none|first|mean")

    t_np_local = t.reshape(-1)
    U_np_local = I_A.reshape(-1, 1)
    Y_np_local = V.reshape(-1, 1)

    if resample and len(t_np_local) > 2:
        dt_med = float(np.median(np.diff(t_np_local)))
        if not (np.isfinite(dt_med) and dt_med > 0):
            raise RuntimeError("Bad dt_med; cannot resample.")
        t_np_local, U_np_local, Y_np_local = resample_uniform(t_np_local, U_np_local, Y_np_local, dt=dt_med)

    if tmax > 0:
        m = t_np_local <= float(tmax)
        t_np_local, U_np_local, Y_np_local = t_np_local[m], U_np_local[m], Y_np_local[m]

    return t_np_local, U_np_local, Y_np_local


def build_proxy_signals(
    t_np_local: np.ndarray,
    U_np_local: np.ndarray,
    cfg: Config,
    xn0: float = XN0,
    xp0: float = XP0,
    ce0_dev: float = CE0_DEV,
):
    Sx_local, A_nom_local, B_nom_local = assemble_system(cfg)
    x0_nom_local = make_x0(cfg, theta_n0=xn0, theta_p0=xp0, ce0=ce0_dev)

    resp_local = ct.forced_response(Sx_local, T=t_np_local, U=U_np_local[:, 0], X0=x0_nom_local)
    X_proxy_local = np.asarray(resp_local.states).T

    xp_sig_local = (X_proxy_local[:, IDX["cp_surf"]] / cfg.csp_max).astype(np.float64)
    xn_sig_local = (X_proxy_local[:, IDX["cn_surf"]] / cfg.csn_max).astype(np.float64)

    ceL_raw_local = X_proxy_local[:, IDX["ce_left"]].astype(np.float64)
    ceR_raw_local = X_proxy_local[:, IDX["ce_right"]].astype(np.float64)

    if cfg.ce_is_deviation:
        ceL_sig_local = cfg.ce0 + ceL_raw_local
        ceR_sig_local = cfg.ce0 + ceR_raw_local
    else:
        ceL_sig_local = ceL_raw_local
        ceR_sig_local = ceR_raw_local

    ln_ce_ratio_local = np.log(np.maximum(ceR_sig_local / np.maximum(ceL_sig_local, 1e-12), 1e-12))

    info = dict(
        xp_rng=float(xp_sig_local.max() - xp_sig_local.min()),
        xn_rng=float(xn_sig_local.max() - xn_sig_local.min()),
        ceL_rng=float(ceL_sig_local.max() - ceL_sig_local.min()),
        ceR_rng=float(ceR_sig_local.max() - ceR_sig_local.min()),
    )

    return X_proxy_local, A_nom_local, B_nom_local, xp_sig_local, xn_sig_local, ceL_sig_local, ceR_sig_local, ln_ce_ratio_local, info


def build_phi_from_proxy(
    xp_sig_local: np.ndarray,
    xn_sig_local: np.ndarray,
    ceL_sig_local: np.ndarray,
    ceR_sig_local: np.ndarray,
    deg: int,
    xp_ref_local: float,
    xn_ref_local: float,
    xp_scale_local: float,
    xn_scale_local: float,
    use_ln_feature: bool,
):
    dxp_local = (xp_sig_local - xp_ref_local) / xp_scale_local
    dxn_local = (xn_sig_local - xn_ref_local) / xn_scale_local

    cols = [np.ones_like(dxp_local)]
    for k in range(1, deg + 1):
        cols.append(dxp_local ** k)
    for k in range(1, deg + 1):
        cols.append(dxn_local ** k)

    if use_ln_feature:
        cols.append(np.log(np.maximum(ceR_sig_local / np.maximum(ceL_sig_local, 1e-12), 1e-12)))

    Phi_local = np.column_stack(cols)
    return Phi_local, dxp_local, dxn_local


def fit_stage2_for_cycle(
    t_np_local: np.ndarray,
    U_np_local: np.ndarray,
    Y_np_local: np.ndarray,
    X_proxy_local: np.ndarray,
    A_nom_local: np.ndarray,
    B_nom_local: np.ndarray,
    xp_sig_local: np.ndarray,
    xn_sig_local: np.ndarray,
    ceL_sig_local: np.ndarray,
    ceR_sig_local: np.ndarray,
    cfg: Config,
    deg: int = POLY_DEG,
    use_ln_feature: bool = USE_LN_FEATURE,
):
    xp_ref_local = float(np.mean(xp_sig_local))
    xn_ref_local = float(np.mean(xn_sig_local))
    xp_scale_local = float(max(np.std(xp_sig_local), 1e-6))
    xn_scale_local = float(max(np.std(xn_sig_local), 1e-6))

    make_thetaZ0_local, zhat_from_thetaZ_local, meta_local = make_additive_poly_surrogate_fns(
        cfg,
        deg,
        xp_ref=xp_ref_local,
        xn_ref=xn_ref_local,
        xp_scale=xp_scale_local,
        xn_scale=xn_scale_local,
        use_ln_feature=use_ln_feature,
    )

    Phi_local, dxp_local, dxn_local = build_phi_from_proxy(
        xp_sig_local, xn_sig_local, ceL_sig_local, ceR_sig_local,
        deg=deg,
        xp_ref_local=xp_ref_local,
        xn_ref_local=xn_ref_local,
        xp_scale_local=xp_scale_local,
        xn_scale_local=xn_scale_local,
        use_ln_feature=use_ln_feature,
    )

    thetaZ0_local = make_thetaZ0_local(Y_np_local)
    rawR0_0_local = raw_from_pos(0.05, floor=1e-12)

    params_stage2_init_local = [
        thetaZ0_local.astype(np.float64),
        np.array([rawR0_0_local], dtype=np.float64),
    ]

    A_nom_j = jnp.array(A_nom_local, dtype=DTYPE)
    B_nom_j = jnp.array(B_nom_local, dtype=DTYPE)

    @jax.jit
    def state_fcn_local(x, u, t, params):
        I = u[0]
        return A_nom_j @ x + (B_nom_j[:, 0] * I)

    @jax.jit
    def output_fcn_local(x, u, t, params):
        thetaZ, rawR0 = params
        I = u[0]
        Z = zhat_from_thetaZ_local(x, thetaZ)
        R0 = pos(rawR0[0], 1e-12)
        Vhat = Z - DTYPE(cfg.N_series) * I * R0
        return jnp.array([Vhat], dtype=DTYPE)

    thetaZ_min_local = -10.0 * np.ones_like(thetaZ0_local, dtype=np.float64)
    thetaZ_max_local =  10.0 * np.ones_like(thetaZ0_local, dtype=np.float64)
    thetaZ_min_local[0] = float(np.min(Y_np_local[:, 0]) - 0.5)
    thetaZ_max_local[0] = float(np.max(Y_np_local[:, 0]) + 0.5)

    rawR0_min_local = np.array([raw_from_pos(1e-4, floor=1e-12)], dtype=np.float64)
    rawR0_max_local = np.array([raw_from_pos(5.0,  floor=1e-12)], dtype=np.float64)

    model_local = CTModel(14, 1, 1, state_fcn=state_fcn_local, output_fcn=output_fcn_local)
    model_local.init(params=params_stage2_init_local, x0=X_proxy_local[0].astype(np.float64))

    try:
        model_local.loss(rho_x0=0.0, rho_th=RHO_TH_STAGE2, train_x0=False, xsat=1e9)
    except TypeError:
        model_local.loss(rho_x0=0.0, rho_th=RHO_TH_STAGE2)

    lbfgs_epochs_local = int(LBFGS_EPOCHS) if USE_LBFGS else 0

    model_local.optimization(
        adam_epochs=int(ADAM_EPOCHS_STAGE2),
        lbfgs_epochs=lbfgs_epochs_local,
        adam_eta=float(ADAM_ETA_STAGE2),
        params_min=[thetaZ_min_local, rawR0_min_local],
        params_max=[thetaZ_max_local, rawR0_max_local],
    )

    dt_local = float(t_np_local[1] - t_np_local[0]) if len(t_np_local) > 1 else 1.0
    model_local.integration_options(
        ode_solver=diffrax.Tsit5(),
        dt0=dt_local / float(DT0_DIV),
        max_steps=int(MAX_STEPS),
        stepsize_controller=diffrax.PIDController(
            rtol=float(SOLVER_RTOL),
            atol=float(SOLVER_ATOL)
        ),
    )

    model_local.fit(Y_np_local, U_np_local, t_np_local)
    Yhat_local, Xhat_local = model_local.predict(model_local.x0, U_np_local, t_np_local)

    thetaZ_hat_local = np.asarray(model_local.params[0]).reshape(-1)
    rawR0_hat_local  = float(np.asarray(model_local.params[1]).reshape(-1)[0])
    R0_hat_local     = float(np.asarray(pos(jnp.array(rawR0_hat_local, dtype=DTYPE), 1e-12)))

    ls_beta_local = solve_ls_qr(Phi_local, Y_np_local[:, 0])
    Y_ls_local = Phi_local @ ls_beta_local

    Phi_plus_R_local = np.column_stack([Phi_local, -U_np_local[:, 0]])
    lsr_beta_local = solve_ls_qr(Phi_plus_R_local, Y_np_local[:, 0])
    Y_lsr_local = Phi_plus_R_local @ lsr_beta_local

    return dict(
        model=model_local,
        Yhat=Yhat_local,
        Xhat=Xhat_local,
        thetaZ_hat=thetaZ_hat_local,
        rawR0_hat=rawR0_hat_local,
        R0_hat=R0_hat_local,
        Y_ls=Y_ls_local.reshape(-1, 1),
        Y_lsr=Y_lsr_local.reshape(-1, 1),
        metaZ=meta_local,
        xp_ref=xp_ref_local,
        xn_ref=xn_ref_local,
        xp_scale=xp_scale_local,
        xn_scale=xn_scale_local,
        Phi=Phi_local,
        dxp=dxp_local,
        dxn=dxn_local,
        zhat_from_thetaZ=zhat_from_thetaZ_local,
        fit=report_fit("Stage 2 reusable fit", Y_np_local, Yhat_local),
    )


def fit_stage3_for_cycle(
    t_np_local: np.ndarray,
    U_np_local: np.ndarray,
    Y_np_local: np.ndarray,
    X_proxy_local: np.ndarray,
    stage2_out: dict,
    cfg: Config,
    run_stage3: bool = RUN_STAGE3,
):
    out = dict(stage3a=None, stage3b=None)

    if not run_stage3:
        return out

    thetaZ_hat_local = stage2_out["thetaZ_hat"]
    rawR0_hat_local = stage2_out["rawR0_hat"]
    zhat_from_thetaZ_local = stage2_out["zhat_from_thetaZ"]

    thetaA_nom_init = thetaA_nom_np.copy()
    thetaB_nom_init = thetaB_nom_np.copy()

    rawA_init = np.log(np.maximum(thetaA_nom_init, 1e-12))
    rawB_init = thetaB_nom_init.copy()
    rawAB_init = np.concatenate([rawA_init, rawB_init]).astype(np.float64)

    @jax.jit
    def unpack_ab(raw_ab: jnp.ndarray):
        raw_ab = raw_ab.reshape(-1)
        rawA = jnp.clip(raw_ab[0:7], -DTYPE(CLIP_RAW_A), DTYPE(CLIP_RAW_A))
        rawB = jnp.clip(raw_ab[7:11], -DTYPE(CLIP_RAW_B), DTYPE(CLIP_RAW_B))
        thetaA = jnp.exp(rawA)
        thetaB = rawB
        return thetaA, thetaB

    @jax.jit
    def state_fcn_stage3a_local(x, u, t, params):
        I = u[0]
        (raw_ab,) = params
        thetaA, thetaB = unpack_ab(raw_ab)
        A = build_A_from_thetaA(thetaA)
        B = build_B_from_thetaB(thetaB)
        return A @ x + (B[:, 0] * I)

    @jax.jit
    def output_fcn_stage3a_local(x, u, t, params):
        I = u[0]
        Z = zhat_from_thetaZ_local(x, jnp.array(thetaZ_hat_local, dtype=DTYPE))
        R0 = pos(jnp.array(rawR0_hat_local, dtype=DTYPE), 1e-12)
        Vhat = Z - DTYPE(cfg.N_series) * I * R0
        return jnp.array([Vhat], dtype=DTYPE)

    model_stage3a_local = CTModel(14, 1, 1, state_fcn=state_fcn_stage3a_local, output_fcn=output_fcn_stage3a_local)
    model_stage3a_local.init(params=[rawAB_init], x0=X_proxy_local[0].astype(np.float64))

    try:
        model_stage3a_local.loss(rho_x0=0.0, rho_th=RHO_TH_STAGE3A, train_x0=False, xsat=1e9)
    except TypeError:
        model_stage3a_local.loss(rho_x0=0.0, rho_th=RHO_TH_STAGE3A)

    model_stage3a_local.optimization(
        adam_epochs=int(ADAM_EPOCHS_STAGE3A),
        lbfgs_epochs=0,
        adam_eta=float(ADAM_ETA_STAGE3A),
    )

    dt_local = float(t_np_local[1] - t_np_local[0]) if len(t_np_local) > 1 else 1.0
    model_stage3a_local.integration_options(
        ode_solver=diffrax.Tsit5(),
        dt0=dt_local / float(DT0_DIV),
        max_steps=int(MAX_STEPS),
        stepsize_controller=diffrax.PIDController(rtol=float(SOLVER_RTOL), atol=float(SOLVER_ATOL)),
    )

    model_stage3a_local.fit(Y_np_local, U_np_local, t_np_local)
    Yhat_stage3a_local, _ = model_stage3a_local.predict(model_stage3a_local.x0, U_np_local, t_np_local)

    rawAB_hat_local = np.asarray(model_stage3a_local.params[0]).reshape(-1)
    rawA_hat_local = rawAB_hat_local[0:7].copy()
    rawB_hat_local = rawAB_hat_local[7:11].copy()

    out["stage3a"] = dict(
        model=model_stage3a_local,
        Yhat=Yhat_stage3a_local,
        rawAB_hat=rawAB_hat_local,
        rawA_hat=rawA_hat_local,
        rawB_hat=rawB_hat_local,
        fit=report_fit("Stage 3a reusable fit", Y_np_local, Yhat_stage3a_local),
    )

    N_THETAZ_LOCAL = int(thetaZ_hat_local.shape[0])

    raw_init_full_local = np.concatenate([
        rawA_hat_local.astype(np.float64),
        rawB_hat_local.astype(np.float64),
        thetaZ_hat_local.astype(np.float64),
        np.array([rawR0_hat_local], dtype=np.float64),
    ]).astype(np.float64)

    def unpack_full(raw_theta: jnp.ndarray):
        raw_theta = raw_theta.reshape(-1)
        rawA   = jnp.clip(raw_theta[0:7], -DTYPE(CLIP_RAW_A), DTYPE(CLIP_RAW_A))
        rawB   = jnp.clip(raw_theta[7:11], -DTYPE(CLIP_RAW_B), DTYPE(CLIP_RAW_B))
        thetaZ = jnp.clip(raw_theta[11:11+N_THETAZ_LOCAL], -DTYPE(CLIP_RAW_Z), DTYPE(CLIP_RAW_Z))
        rawR0  = raw_theta[11+N_THETAZ_LOCAL]
        thetaA = jnp.exp(rawA)
        thetaB = rawB
        return thetaA, thetaB, thetaZ, rawR0

    @jax.jit
    def state_fcn_stage3b_local(x, u, t, params):
        I = u[0]
        (raw_theta,) = params
        thetaA, thetaB, _, _ = unpack_full(raw_theta)
        A = build_A_from_thetaA(thetaA)
        B = build_B_from_thetaB(thetaB)
        return A @ x + (B[:, 0] * I)

    @jax.jit
    def output_fcn_stage3b_local(x, u, t, params):
        I = u[0]
        (raw_theta,) = params
        _, _, thetaZ, rawR0 = unpack_full(raw_theta)
        Z = zhat_from_thetaZ_local(x, thetaZ)
        R0 = pos(rawR0, 1e-12)
        Vhat = Z - DTYPE(cfg.N_series) * I * R0
        return jnp.array([Vhat], dtype=DTYPE)

    model_stage3b_local = CTModel(14, 1, 1, state_fcn=state_fcn_stage3b_local, output_fcn=output_fcn_stage3b_local)
    model_stage3b_local.init(params=[raw_init_full_local], x0=X_proxy_local[0].astype(np.float64))

    try:
        model_stage3b_local.loss(rho_x0=0.0, rho_th=RHO_TH_STAGE3B, train_x0=False, xsat=1e9)
    except TypeError:
        model_stage3b_local.loss(rho_x0=0.0, rho_th=RHO_TH_STAGE3B)

    model_stage3b_local.optimization(
        adam_epochs=int(ADAM_EPOCHS_STAGE3B),
        lbfgs_epochs=0,
        adam_eta=float(ADAM_ETA_STAGE3B),
    )

    model_stage3b_local.integration_options(
        ode_solver=diffrax.Tsit5(),
        dt0=dt_local / float(DT0_DIV),
        max_steps=int(MAX_STEPS),
        stepsize_controller=diffrax.PIDController(rtol=float(SOLVER_RTOL), atol=float(SOLVER_ATOL)),
    )

    model_stage3b_local.fit(Y_np_local, U_np_local, t_np_local)
    Yhat_stage3b_local, _ = model_stage3b_local.predict(model_stage3b_local.x0, U_np_local, t_np_local)

    raw_full_hat_local = np.asarray(model_stage3b_local.params[0]).reshape(-1)
    thetaA_hat_local = np.exp(np.clip(raw_full_hat_local[0:7], -CLIP_RAW_A, CLIP_RAW_A))
    thetaB_hat_local = raw_full_hat_local[7:11].copy()
    thetaZ_hat_stage3b_local = raw_full_hat_local[11:11+N_THETAZ_LOCAL].copy()
    rawR0_hat_stage3b_local = float(raw_full_hat_local[11+N_THETAZ_LOCAL])
    R0_hat_stage3b_local = float(np.asarray(pos(jnp.array(rawR0_hat_stage3b_local, dtype=DTYPE), 1e-12)))

    out["stage3b"] = dict(
        model=model_stage3b_local,
        Yhat=Yhat_stage3b_local,
        raw_full_hat=raw_full_hat_local,
        thetaA_hat=thetaA_hat_local,
        thetaB_hat=thetaB_hat_local,
        thetaZ_hat=thetaZ_hat_stage3b_local,
        rawR0_hat=rawR0_hat_stage3b_local,
        R0_hat=R0_hat_stage3b_local,
        fit=report_fit("Stage 3b reusable fit", Y_np_local, Yhat_stage3b_local),
    )

    return out


def run_full_cycle_pipeline(
    cycle_df_local: pd.DataFrame,
    cycle_idx: int,
    deg: int = POLY_DEG,
    run_stage3: bool = RUN_STAGE3,
    verbose: bool = False,
):
    try:
        t_local, U_local, Y_local = prepare_cycle_data(cycle_df_local)

        med_abs_I_local = float(np.median(np.abs(U_local[:, 0])))
        if med_abs_I_local < float(MIN_MED_ABS_I_A):
            return dict(
                ok=False,
                cycle_idx=cycle_idx,
                reason=f"median current too small: {med_abs_I_local:.6g} A"
            )

        proxy = build_proxy_signals(
            t_local, U_local, cfg=CFG,
            xn0=XN0, xp0=XP0, ce0_dev=CE0_DEV
        )
        (
            X_proxy_local, A_nom_local, B_nom_local,
            xp_sig_local, xn_sig_local,
            ceL_sig_local, ceR_sig_local,
            ln_ce_ratio_local, proxy_info_local
        ) = proxy

        if (proxy_info_local["xp_rng"] < 1e-7) and (proxy_info_local["xn_rng"] < 1e-7):
            return dict(
                ok=False,
                cycle_idx=cycle_idx,
                reason="xp/xn proxy motion essentially zero"
            )

        stage2_out = fit_stage2_for_cycle(
            t_local, U_local, Y_local,
            X_proxy_local, A_nom_local, B_nom_local,
            xp_sig_local, xn_sig_local, ceL_sig_local, ceR_sig_local,
            cfg=CFG, deg=deg, use_ln_feature=USE_LN_FEATURE,
        )

        stage3_out = fit_stage3_for_cycle(
            t_local, U_local, Y_local,
            X_proxy_local,
            stage2_out,
            cfg=CFG,
            run_stage3=run_stage3,
        )

        res = dict(
            ok=True,
            cycle_idx=cycle_idx,
            deg=deg,
            t=t_local,
            U=U_local,
            Y=Y_local,
            X_proxy=X_proxy_local,
            xp_sig=xp_sig_local,
            xn_sig=xn_sig_local,
            ceL_sig=ceL_sig_local,
            ceR_sig=ceR_sig_local,
            ln_ce_ratio=ln_ce_ratio_local,
            proxy_info=proxy_info_local,
            stage2=stage2_out,
            stage3=stage3_out,
        )

        if verbose:
            print(f"[cycle {cycle_idx}] completed successfully")

        return res

    except Exception as e:
        return dict(
            ok=False,
            cycle_idx=cycle_idx,
            deg=deg,
            reason=str(e),
        )


def evaluate_stage2_transfer(
    target_cycle_df: pd.DataFrame,
    fitted_stage2: dict,
    source_deg: int,
    source_cycle_idx: int,
):
    t_local, U_local, Y_local = prepare_cycle_data(target_cycle_df)

    proxy = build_proxy_signals(
        t_local, U_local, cfg=CFG,
        xn0=XN0, xp0=XP0, ce0_dev=CE0_DEV
    )
    (
        X_proxy_local, A_nom_local, B_nom_local,
        xp_sig_local, xn_sig_local,
        ceL_sig_local, ceR_sig_local,
        ln_ce_ratio_local, proxy_info_local
    ) = proxy

    xp_ref_local = fitted_stage2["xp_ref"]
    xn_ref_local = fitted_stage2["xn_ref"]
    xp_scale_local = fitted_stage2["xp_scale"]
    xn_scale_local = fitted_stage2["xn_scale"]

    _, zhat_from_thetaZ_local, _ = make_additive_poly_surrogate_fns(
        CFG,
        source_deg,
        xp_ref=xp_ref_local,
        xn_ref=xn_ref_local,
        xp_scale=xp_scale_local,
        xn_scale=xn_scale_local,
        use_ln_feature=USE_LN_FEATURE,
    )

    thetaZ_hat_local = fitted_stage2["thetaZ_hat"]
    rawR0_hat_local = fitted_stage2["rawR0_hat"]

    A_nom_j = jnp.array(A_nom_local, dtype=DTYPE)
    B_nom_j = jnp.array(B_nom_local, dtype=DTYPE)

    @jax.jit
    def state_fcn_transfer(x, u, t, params):
        I = u[0]
        return A_nom_j @ x + (B_nom_j[:, 0] * I)

    @jax.jit
    def output_fcn_transfer(x, u, t, params):
        I = u[0]
        Z = zhat_from_thetaZ_local(x, jnp.array(thetaZ_hat_local, dtype=DTYPE))
        R0 = pos(jnp.array(rawR0_hat_local, dtype=DTYPE), 1e-12)
        Vhat = Z - DTYPE(CFG.N_series) * I * R0
        return jnp.array([Vhat], dtype=DTYPE)

    m = CTModel(14, 1, 1, state_fcn=state_fcn_transfer, output_fcn=output_fcn_transfer)
    m.init(params=[np.array([0.0])], x0=X_proxy_local[0].astype(np.float64))  # dummy params
    dt_local = float(t_local[1] - t_local[0]) if len(t_local) > 1 else 1.0
    m.integration_options(
        ode_solver=diffrax.Tsit5(),
        dt0=dt_local / float(DT0_DIV),
        max_steps=int(MAX_STEPS),
        stepsize_controller=diffrax.PIDController(
            rtol=float(SOLVER_RTOL),
            atol=float(SOLVER_ATOL)
        ),
    )

    Yhat_local, _ = m.predict(m.x0, U_local, t_local)
    fit_local = report_fit(
        f"Stage 2 transfer | source cycle={source_cycle_idx} -> target cycle",
        Y_local, Yhat_local
    )

    return dict(
        t=t_local,
        U=U_local,
        Y=Y_local,
        Yhat=Yhat_local,
        fit=fit_local,
        proxy_info=proxy_info_local,
    )


def evaluate_stage3b_transfer(
    target_cycle_df: pd.DataFrame,
    fitted_stage3b: dict,
    fitted_stage2_meta: dict,
    source_deg: int,
    source_cycle_idx: int,
):
    t_local, U_local, Y_local = prepare_cycle_data(target_cycle_df)

    proxy = build_proxy_signals(
        t_local, U_local, cfg=CFG,
        xn0=XN0, xp0=XP0, ce0_dev=CE0_DEV
    )
    (
        X_proxy_local, _, _,
        _, _, _, _, _, proxy_info_local
    ) = proxy

    xp_ref_local = fitted_stage2_meta["xp_ref"]
    xn_ref_local = fitted_stage2_meta["xn_ref"]
    xp_scale_local = fitted_stage2_meta["xp_scale"]
    xn_scale_local = fitted_stage2_meta["xn_scale"]

    _, zhat_from_thetaZ_local, _ = make_additive_poly_surrogate_fns(
        CFG,
        source_deg,
        xp_ref=xp_ref_local,
        xn_ref=xn_ref_local,
        xp_scale=xp_scale_local,
        xn_scale=xn_scale_local,
        use_ln_feature=USE_LN_FEATURE,
    )

    thetaA_hat_local = fitted_stage3b["thetaA_hat"]
    thetaB_hat_local = fitted_stage3b["thetaB_hat"]
    thetaZ_hat_local = fitted_stage3b["thetaZ_hat"]
    rawR0_hat_local = fitted_stage3b["rawR0_hat"]

    @jax.jit
    def state_fcn_transfer(x, u, t, params):
        I = u[0]
        A = build_A_from_thetaA(jnp.array(thetaA_hat_local, dtype=DTYPE))
        B = build_B_from_thetaB(jnp.array(thetaB_hat_local, dtype=DTYPE))
        return A @ x + (B[:, 0] * I)

    @jax.jit
    def output_fcn_transfer(x, u, t, params):
        I = u[0]
        Z = zhat_from_thetaZ_local(x, jnp.array(thetaZ_hat_local, dtype=DTYPE))
        R0 = pos(jnp.array(rawR0_hat_local, dtype=DTYPE), 1e-12)
        Vhat = Z - DTYPE(CFG.N_series) * I * R0
        return jnp.array([Vhat], dtype=DTYPE)

    m = CTModel(14, 1, 1, state_fcn=state_fcn_transfer, output_fcn=output_fcn_transfer)
    m.init(params=[np.array([0.0])], x0=X_proxy_local[0].astype(np.float64))
    dt_local = float(t_local[1] - t_local[0]) if len(t_local) > 1 else 1.0
    m.integration_options(
        ode_solver=diffrax.Tsit5(),
        dt0=dt_local / float(DT0_DIV),
        max_steps=int(MAX_STEPS),
        stepsize_controller=diffrax.PIDController(
            rtol=float(SOLVER_RTOL),
            atol=float(SOLVER_ATOL)
        ),
    )

    Yhat_local, _ = m.predict(m.x0, U_local, t_local)
    fit_local = report_fit(
        f"Stage 3b transfer | source cycle={source_cycle_idx} -> target cycle",
        Y_local, Yhat_local
    )

    return dict(
        t=t_local,
        U=U_local,
        Y=Y_local,
        Yhat=Yhat_local,
        fit=fit_local,
        proxy_info=proxy_info_local,
    )


def summarize_monitorable_parameters(result: dict):
    if not result.get("ok", False):
        return dict(ok=False, cycle_idx=result.get("cycle_idx", -1))

    s2 = result["stage2"]
    s3 = result["stage3"]

    out = dict(
        ok=True,
        cycle_idx=result["cycle_idx"],
        deg=result["deg"],
        stage2_rmse=float(s2["fit"]["rmse"]),
        stage2_r2=float(s2["fit"]["r2"]),
        stage2_bfr=float(s2["fit"]["bfr"]),
        R0_stage2=float(s2["R0_hat"]),
        xp_rng=float(result["proxy_info"]["xp_rng"]),
        xn_rng=float(result["proxy_info"]["xn_rng"]),
        ceL_rng=float(result["proxy_info"]["ceL_rng"]),
        ceR_rng=float(result["proxy_info"]["ceR_rng"]),
        ln_ratio_rng=float(np.max(result["ln_ce_ratio"]) - np.min(result["ln_ce_ratio"])),
        z_theta_l2=float(np.linalg.norm(s2["thetaZ_hat"])),
        z_theta_maxabs=float(np.max(np.abs(s2["thetaZ_hat"]))),
    )

    if s3["stage3a"] is not None:
        out["stage3a_rmse"] = float(s3["stage3a"]["fit"]["rmse"])
        out["stage3a_r2"]   = float(s3["stage3a"]["fit"]["r2"])
    else:
        out["stage3a_rmse"] = np.nan
        out["stage3a_r2"]   = np.nan

    if s3["stage3b"] is not None:
        thA = np.asarray(s3["stage3b"]["thetaA_hat"]).reshape(-1)
        thB = np.asarray(s3["stage3b"]["thetaB_hat"]).reshape(-1)

        out["stage3b_rmse"] = float(s3["stage3b"]["fit"]["rmse"])
        out["stage3b_r2"]   = float(s3["stage3b"]["fit"]["r2"])
        out["R0_stage3b"]   = float(s3["stage3b"]["R0_hat"])

        for i, val in enumerate(thA, start=1):
            out[f"thetaA_{i}"] = float(val)
        for i, val in enumerate(thB, start=8):
            out[f"thetaB_{i}"] = float(val)

        out["thetaA_norm"] = float(np.linalg.norm(thA))
        out["thetaB_norm"] = float(np.linalg.norm(thB))
    else:
        out["stage3b_rmse"] = np.nan
        out["stage3b_r2"]   = np.nan
        out["R0_stage3b"]   = np.nan

    return out


def plot_learned_nonlinearity_from_result(result: dict, title_prefix=""):
    if not result.get("ok", False):
        print("Cannot plot nonlinearity for failed result.")
        return

    s2 = result["stage2"]
    xp = np.asarray(result["xp_sig"]).reshape(-1)
    xn = np.asarray(result["xn_sig"]).reshape(-1)
    X_proxy_local = np.asarray(result["X_proxy"])
    thetaZ_local = np.asarray(s2["thetaZ_hat"]).reshape(-1)
    zhat_local = s2["zhat_from_thetaZ"]

    Zvals = []
    for k in range(X_proxy_local.shape[0]):
        z = zhat_local(jnp.array(X_proxy_local[k], dtype=DTYPE), jnp.array(thetaZ_local, dtype=DTYPE))
        Zvals.append(float(np.asarray(z)))
    Zvals = np.asarray(Zvals)

    dxp = np.asarray(s2["dxp"]).reshape(-1)
    dxn = np.asarray(s2["dxn"]).reshape(-1)

    plt.figure(figsize=(12, 4))
    plt.subplot(1, 3, 1)
    plt.plot(xp, Zvals, "o-")
    plt.grid(True)
    plt.xlabel("xp")
    plt.ylabel("Z_hat")
    plt.title(f"{title_prefix} Z_hat vs xp")

    plt.subplot(1, 3, 2)
    plt.plot(xn, Zvals, "o-")
    plt.grid(True)
    plt.xlabel("xn")
    plt.ylabel("Z_hat")
    plt.title(f"{title_prefix} Z_hat vs xn")

    plt.subplot(1, 3, 3)
    plt.scatter(dxp, dxn, c=Zvals)
    plt.grid(True)
    plt.xlabel("dxp")
    plt.ylabel("dxn")
    plt.title(f"{title_prefix} operating path colored by Z_hat")
    plt.tight_layout()
    plt.show()

    print(f"{title_prefix} nonlinearity summaries")
    print("  monotonicity fraction vs time on Z_hat:", monotonicity_fraction(Zvals))
    print("  corr(Z_hat, xp):", safe_corr(Zvals, xp))
    print("  corr(Z_hat, xn):", safe_corr(Zvals, xn))


# %% ======================================================
# CELL 14 — Cross-cycle transfer/generalization of selected-cycle model
# =========================================================
if not RUN_GENERALIZATION_TESTS:
    print("\n[INFO] RUN_GENERALIZATION_TESTS=False, skipping Cell 14.")
else:
    print("\n" + "=" * 90)
    print("CROSS-CYCLE TRANSFER / GENERALIZATION TESTS")
    print("=" * 90)

    if GENERALIZATION_SCOPE == "all_cycles":
        test_cycle_ids = list(range(len(cycles)))
    else:
        if GENERALIZATION_CYCLE_IDS is None:
            test_cycle_ids = choose_spread_indices(len(cycles), N_GENERALIZATION_CYCLES)
            if chosen not in test_cycle_ids:
                test_cycle_ids = sorted(set(test_cycle_ids + [chosen]))
        else:
            test_cycle_ids = sorted(set(int(i) for i in GENERALIZATION_CYCLE_IDS if 0 <= int(i) < len(cycles)))

    print("Transfer/generalization test cycle IDs:", test_cycle_ids)
    print("Selected/fitted source cycle:", chosen)

    transfer_rows = []

    for cid in test_cycle_ids:
        cyc_df = cycles[cid]

        print("\n" + "-" * 80)
        print(f"TRANSFER TEST ON TARGET CYCLE {cid}")
        print("-" * 80)

        row = dict(target_cycle=cid, source_cycle=chosen, deg=POLY_DEG)

        try:
            s2_transfer = evaluate_stage2_transfer(
                cyc_df,
                fitted_stage2=dict(
                    thetaZ_hat=thetaZ_hat,
                    rawR0_hat=rawR0_hat,
                    xp_ref=xp_ref,
                    xn_ref=xn_ref,
                    xp_scale=xp_scale,
                    xn_scale=xn_scale,
                ),
                source_deg=POLY_DEG,
                source_cycle_idx=chosen,
            )
            row["stage2_transfer_rmse"] = float(s2_transfer["fit"]["rmse"])
            row["stage2_transfer_r2"] = float(s2_transfer["fit"]["r2"])
            row["stage2_transfer_bfr"] = float(s2_transfer["fit"]["bfr"])
        except Exception as e:
            row["stage2_transfer_rmse"] = np.nan
            row["stage2_transfer_r2"] = np.nan
            row["stage2_transfer_bfr"] = np.nan
            row["stage2_transfer_error"] = str(e)

        if RUN_STAGE3:
            try:
                s3_transfer = evaluate_stage3b_transfer(
                    cyc_df,
                    fitted_stage3b=dict(
                        thetaA_hat=thetaA_hat_stage3b,
                        thetaB_hat=thetaB_hat_stage3b,
                        thetaZ_hat=thetaZ_hat_stage3b,
                        rawR0_hat=rawR0_hat_stage3b,
                        R0_hat=R0_hat_stage3b,
                    ),
                    fitted_stage2_meta=dict(
                        xp_ref=xp_ref,
                        xn_ref=xn_ref,
                        xp_scale=xp_scale,
                        xn_scale=xn_scale,
                    ),
                    source_deg=POLY_DEG,
                    source_cycle_idx=chosen,
                )
                row["stage3b_transfer_rmse"] = float(s3_transfer["fit"]["rmse"])
                row["stage3b_transfer_r2"] = float(s3_transfer["fit"]["r2"])
                row["stage3b_transfer_bfr"] = float(s3_transfer["fit"]["bfr"])
            except Exception as e:
                row["stage3b_transfer_rmse"] = np.nan
                row["stage3b_transfer_r2"] = np.nan
                row["stage3b_transfer_bfr"] = np.nan
                row["stage3b_transfer_error"] = str(e)

        transfer_rows.append(row)

    transfer_df = pd.DataFrame(transfer_rows)
    print("\nTRANSFER SUMMARY")
    print(transfer_df)

    if not transfer_df.empty:
        plt.figure(figsize=(12, 4))
        plt.plot(transfer_df["target_cycle"], transfer_df["stage2_transfer_rmse"], "o-", label="Stage 2 transfer RMSE")
        if "stage3b_transfer_rmse" in transfer_df.columns:
            plt.plot(transfer_df["target_cycle"], transfer_df["stage3b_transfer_rmse"], "o-", label="Stage 3b transfer RMSE")
        plt.grid(True)
        plt.xlabel("Target cycle index")
        plt.ylabel("RMSE [V]")
        plt.title("Cross-cycle transfer RMSE from selected-cycle fitted model")
        plt.legend()
        plt.tight_layout()
        plt.show()

        plt.figure(figsize=(12, 4))
        plt.plot(transfer_df["target_cycle"], transfer_df["stage2_transfer_r2"], "o-", label="Stage 2 transfer R2%")
        if "stage3b_transfer_r2" in transfer_df.columns:
            plt.plot(transfer_df["target_cycle"], transfer_df["stage3b_transfer_r2"], "o-", label="Stage 3b transfer R2%")
        plt.grid(True)
        plt.xlabel("Target cycle index")
        plt.ylabel("R2 [%]")
        plt.title("Cross-cycle transfer R2 from selected-cycle fitted model")
        plt.legend()
        plt.tight_layout()
        plt.show()


# %% ======================================================
# CELL 15 — Fit several/all cycles independently and summarize stability
# =========================================================
if not RUN_GENERALIZATION_TESTS or not RUN_PER_CYCLE_REFITS:
    print("\n[INFO] Skipping per-cycle refits in Cell 15.")
else:
    print("\n" + "=" * 90)
    print("PER-CYCLE REFITS ACROSS SEVERAL / ALL CYCLES")
    print("=" * 90)

    if REFIT_ALL_CYCLES or GENERALIZATION_SCOPE == "all_cycles":
        refit_cycle_ids = list(range(len(cycles)))
    else:
        if GENERALIZATION_CYCLE_IDS is None:
            refit_cycle_ids = choose_spread_indices(len(cycles), N_GENERALIZATION_CYCLES)
            if chosen not in refit_cycle_ids:
                refit_cycle_ids = sorted(set(refit_cycle_ids + [chosen]))
        else:
            refit_cycle_ids = sorted(set(int(i) for i in GENERALIZATION_CYCLE_IDS if 0 <= int(i) < len(cycles)))

    print("Refit cycle IDs:", refit_cycle_ids)

    all_cycle_results = []
    monitor_rows = []

    for cid in refit_cycle_ids:
        print("\n" + "#" * 90)
        print(f"REFIT FULL PIPELINE ON CYCLE {cid}")
        print("#" * 90)

        res = run_full_cycle_pipeline(
            cycles[cid],
            cycle_idx=cid,
            deg=POLY_DEG,
            run_stage3=RUN_STAGE3,
            verbose=True,
        )
        all_cycle_results.append(res)

        row = summarize_monitorable_parameters(res)
        monitor_rows.append(row)

    monitor_df = pd.DataFrame(monitor_rows)
    print("\nPER-CYCLE MONITOR SUMMARY")
    print(monitor_df)

    ok_monitor_df = monitor_df[monitor_df["ok"] == True].copy()

    if not ok_monitor_df.empty:
        plt.figure(figsize=(12, 4))
        plt.plot(ok_monitor_df["cycle_idx"], ok_monitor_df["stage2_rmse"], "o-", label="Stage 2 RMSE")
        if "stage3b_rmse" in ok_monitor_df.columns:
            plt.plot(ok_monitor_df["cycle_idx"], ok_monitor_df["stage3b_rmse"], "o-", label="Stage 3b RMSE")
        plt.grid(True)
        plt.xlabel("Cycle index")
        plt.ylabel("RMSE [V]")
        plt.title(f"Per-cycle fit quality across cycles (deg={POLY_DEG})")
        plt.legend()
        plt.tight_layout()
        plt.show()

        plt.figure(figsize=(12, 4))
        plt.plot(ok_monitor_df["cycle_idx"], ok_monitor_df["R0_stage2"], "o-", label="R0 Stage 2")
        if "R0_stage3b" in ok_monitor_df.columns:
            plt.plot(ok_monitor_df["cycle_idx"], ok_monitor_df["R0_stage3b"], "o-", label="R0 Stage 3b")
        plt.grid(True)
        plt.xlabel("Cycle index")
        plt.ylabel("Resistance [Ohm]")
        plt.title("Monitorable R0 across cycles")
        plt.legend()
        plt.tight_layout()
        plt.show()

        if "thetaA_1" in ok_monitor_df.columns:
            plt.figure(figsize=(12, 5))
            for col in [c for c in ok_monitor_df.columns if c.startswith("thetaA_")]:
                plt.plot(ok_monitor_df["cycle_idx"], ok_monitor_df[col], "o-", label=col)
            plt.grid(True)
            plt.xlabel("Cycle index")
            plt.ylabel("Estimated value")
            plt.title("Monitorable thetaA parameters across cycles")
            plt.legend(ncol=2)
            plt.tight_layout()
            plt.show()

        if "thetaB_8" in ok_monitor_df.columns:
            plt.figure(figsize=(12, 5))
            for col in [c for c in ok_monitor_df.columns if c.startswith("thetaB_")]:
                plt.plot(ok_monitor_df["cycle_idx"], ok_monitor_df[col], "o-", label=col)
            plt.grid(True)
            plt.xlabel("Cycle index")
            plt.ylabel("Estimated value")
            plt.title("Monitorable thetaB parameters across cycles")
            plt.legend(ncol=2)
            plt.tight_layout()
            plt.show()

        print("\nSTABILITY CHECKS")
        print("Mean Stage 2 RMSE:", float(ok_monitor_df["stage2_rmse"].mean()))
        print("Std  Stage 2 RMSE:", float(ok_monitor_df["stage2_rmse"].std(ddof=0)))
        if "stage3b_rmse" in ok_monitor_df.columns:
            print("Mean Stage 3b RMSE:", float(ok_monitor_df["stage3b_rmse"].mean()))
            print("Std  Stage 3b RMSE:", float(ok_monitor_df["stage3b_rmse"].std(ddof=0)))
        print("Mean R0 Stage 2:", float(ok_monitor_df["R0_stage2"].mean()))
        print("Std  R0 Stage 2:", float(ok_monitor_df["R0_stage2"].std(ddof=0)))


# %% ======================================================
# CELL 16 — Visualize learned nonlinearity
# =========================================================
if not RUN_NONLINEARITY_PLOTS:
    print("\n[INFO] RUN_NONLINEARITY_PLOTS=False, skipping Cell 16.")
else:
    print("\n" + "=" * 90)
    print("VISUALIZE LEARNED NONLINEARITY")
    print("=" * 90)

    print("\nSelected-cycle nonlinearity:")
    selected_result_for_plot = dict(
        ok=True,
        cycle_idx=chosen,
        deg=POLY_DEG,
        X_proxy=X_proxy,
        xp_sig=xp_sig,
        xn_sig=xn_sig,
        ceL_sig=ceL_sig,
        ceR_sig=ceR_sig,
        ln_ce_ratio=ln_ce_ratio_sig,
        proxy_info=dict(xp_rng=xp_rng, xn_rng=xn_rng, ceL_rng=ceL_rng, ceR_rng=ceR_rng),
        stage2=dict(
            thetaZ_hat=thetaZ_hat,
            zhat_from_thetaZ=zhat_from_thetaZ_stage2,
            dxp=dxp_dbg,
            dxn=dxn_dbg,
        ),
    )
    plot_learned_nonlinearity_from_result(
        selected_result_for_plot,
        title_prefix=f"cycle {chosen} |"
    )

    if RUN_GENERALIZATION_TESTS and RUN_PER_CYCLE_REFITS and "all_cycle_results" in globals():
        ok_results = [r for r in all_cycle_results if r.get("ok", False)]

        if NONLINEARITY_PLOT_CYCLE_IDS is None:
            ids_for_nonlin = [r["cycle_idx"] for r in ok_results]
            ids_for_nonlin = choose_spread_indices(len(ids_for_nonlin), min(N_NONLINEARITY_PLOTS, len(ids_for_nonlin)))
            ids_for_nonlin = [ok_results[i]["cycle_idx"] for i in ids_for_nonlin]
        else:
            ids_for_nonlin = [int(i) for i in NONLINEARITY_PLOT_CYCLE_IDS]

        print("\nAdditional nonlinearity plots for cycles:", ids_for_nonlin)

        for cid in ids_for_nonlin:
            matches = [r for r in ok_results if r["cycle_idx"] == cid]
            if len(matches) == 0:
                continue
            plot_learned_nonlinearity_from_result(
                matches[0],
                title_prefix=f"cycle {cid} |"
            )


# %% ======================================================
# CELL 17 — Extract monitorable parameters and tell degradation story
# =========================================================
if not RUN_GENERALIZATION_TESTS or not RUN_PER_CYCLE_REFITS or "monitor_df" not in globals():
    print("\n[INFO] Skipping degradation-story cell because monitor_df is not available.")
else:
    print("\n" + "=" * 90)
    print("EXTRACT MONITORABLE PARAMETERS + TELL THE DEGRADATION STORY")
    print("=" * 90)

    story_df = monitor_df[monitor_df["ok"] == True].copy().sort_values("cycle_idx")
    print(story_df)

    if not story_df.empty:
        cols_to_show = [
            "cycle_idx",
            "stage2_rmse",
            "stage3b_rmse",
            "R0_stage2",
            "R0_stage3b",
            "xp_rng",
            "xn_rng",
            "z_theta_l2",
            "z_theta_maxabs",
        ]
        cols_to_show = [c for c in cols_to_show if c in story_df.columns]
        print("\nCore story table:")
        print(story_df[cols_to_show])

        plt.figure(figsize=(12, 4))
        plt.plot(story_df["cycle_idx"], story_df["R0_stage2"], "o-", label="R0 Stage 2")
        if "R0_stage3b" in story_df.columns:
            plt.plot(story_df["cycle_idx"], story_df["R0_stage3b"], "o-", label="R0 Stage 3b")
        plt.grid(True)
        plt.xlabel("Cycle index")
        plt.ylabel("Ohmic resistance [Ohm]")
        plt.title("Degradation story: resistance trend")
        plt.legend()
        plt.tight_layout()
        plt.show()

        plt.figure(figsize=(12, 4))
        plt.plot(story_df["cycle_idx"], story_df["stage2_rmse"], "o-", label="Stage 2 RMSE")
        if "stage3b_rmse" in story_df.columns:
            plt.plot(story_df["cycle_idx"], story_df["stage3b_rmse"], "o-", label="Stage 3b RMSE")
        plt.grid(True)
        plt.xlabel("Cycle index")
        plt.ylabel("RMSE [V]")
        plt.title("Degradation story: fit quality trend")
        plt.legend()
        plt.tight_layout()
        plt.show()

        plt.figure(figsize=(12, 4))
        plt.plot(story_df["cycle_idx"], story_df["z_theta_l2"], "o-", label="||thetaZ||_2")
        plt.plot(story_df["cycle_idx"], story_df["z_theta_maxabs"], "o-", label="max|thetaZ|")
        plt.grid(True)
        plt.xlabel("Cycle index")
        plt.ylabel("Magnitude")
        plt.title("Degradation story: nonlinearity-complexity trend")
        plt.legend()
        plt.tight_layout()
        plt.show()

        print("\nTEXT STORY")
        r0_corr = safe_corr(story_df["cycle_idx"].to_numpy(), story_df["R0_stage2"].to_numpy())
        rmse_corr = safe_corr(story_df["cycle_idx"].to_numpy(), story_df["stage2_rmse"].to_numpy())
        zcorr = safe_corr(story_df["cycle_idx"].to_numpy(), story_df["z_theta_l2"].to_numpy())

        print(f"- Correlation(cycle index, R0_stage2)   = {r0_corr:.6g}")
        print(f"- Correlation(cycle index, stage2_rmse) = {rmse_corr:.6g}")
        print(f"- Correlation(cycle index, ||thetaZ||2) = {zcorr:.6g}")

        if np.isfinite(r0_corr):
            if r0_corr > 0.3:
                print("- R0 tends to increase with cycle index, which is consistent with resistance growth.")
            elif r0_corr < -0.3:
                print("- R0 tends to decrease with cycle index, which may indicate estimator instability or cycle nonuniformity.")
            else:
                print("- R0 is relatively flat over the tested cycles, suggesting little resistance drift or insufficient sensitivity.")

        if np.isfinite(zcorr):
            if zcorr > 0.3:
                print("- The nonlinearity magnitude tends to grow with cycle index.")
            elif zcorr < -0.3:
                print("- The nonlinearity magnitude tends to shrink with cycle index.")
            else:
                print("- The nonlinearity magnitude is fairly stable over the tested cycles.")

        print("- Final judgment should combine: fit stability, R0 trend, A/B trend, and smoothness of the learned nonlinearity.")


# %% ======================================================
# CELL 18 — Degree comparison helper (17 vs 15 vs 14)
# =========================================================
if not RUN_MULTI_DEGREE_COMPARISON:
    print("\n[INFO] RUN_MULTI_DEGREE_COMPARISON=False, skipping Cell 18.")
else:
    print("\n" + "=" * 90)
    print("MULTI-DEGREE COMPARISON (17 vs 15 vs 14)")
    print("=" * 90)

    if MULTI_DEGREE_SCOPE == "all_cycles":
        degree_cycle_ids = list(range(len(cycles)))
    else:
        if MULTI_DEGREE_CYCLE_IDS is None:
            degree_cycle_ids = choose_spread_indices(len(cycles), N_MULTI_DEGREE_CYCLES)
            if chosen not in degree_cycle_ids:
                degree_cycle_ids = sorted(set(degree_cycle_ids + [chosen]))
        else:
            degree_cycle_ids = sorted(set(int(i) for i in MULTI_DEGREE_CYCLE_IDS if 0 <= int(i) < len(cycles)))

    print("Degree-comparison cycles:", degree_cycle_ids)
    print("Degrees:", MULTI_DEGREE_LIST)

    degree_rows = []

    for deg_test in MULTI_DEGREE_LIST:
        for cid in degree_cycle_ids:
            print("\n" + "-" * 80)
            print(f"DEGREE TEST | cycle={cid}, deg={deg_test}")
            print("-" * 80)

            res = run_full_cycle_pipeline(
                cycles[cid],
                cycle_idx=cid,
                deg=deg_test,
                run_stage3=RUN_STAGE3,
                verbose=False,
            )

            row = dict(cycle_idx=cid, deg=deg_test, ok=res.get("ok", False))
            if res.get("ok", False):
                row["stage2_rmse"] = float(res["stage2"]["fit"]["rmse"])
                row["stage2_r2"]   = float(res["stage2"]["fit"]["r2"])
                row["R0_stage2"]   = float(res["stage2"]["R0_hat"])
                row["z_theta_l2"]  = float(np.linalg.norm(res["stage2"]["thetaZ_hat"]))
                row["z_theta_maxabs"] = float(np.max(np.abs(res["stage2"]["thetaZ_hat"])))

                if res["stage3"]["stage3b"] is not None:
                    row["stage3b_rmse"] = float(res["stage3"]["stage3b"]["fit"]["rmse"])
                    row["stage3b_r2"]   = float(res["stage3"]["stage3b"]["fit"]["r2"])
                    row["R0_stage3b"]   = float(res["stage3"]["stage3b"]["R0_hat"])
                else:
                    row["stage3b_rmse"] = np.nan
                    row["stage3b_r2"]   = np.nan
                    row["R0_stage3b"]   = np.nan
            else:
                row["reason"] = res.get("reason", "unknown failure")

            degree_rows.append(row)

    degree_df = pd.DataFrame(degree_rows)
    print("\nMULTI-DEGREE SUMMARY")
    print(degree_df)

    ok_degree_df = degree_df[degree_df["ok"] == True].copy()

    if not ok_degree_df.empty:
        grouped = ok_degree_df.groupby("deg").agg(
            stage2_rmse_mean=("stage2_rmse", "mean"),
            stage2_rmse_std=("stage2_rmse", "std"),
            stage2_r2_mean=("stage2_r2", "mean"),
            stage3b_rmse_mean=("stage3b_rmse", "mean"),
            stage3b_rmse_std=("stage3b_rmse", "std"),
            stage3b_r2_mean=("stage3b_r2", "mean"),
            z_theta_l2_mean=("z_theta_l2", "mean"),
            z_theta_maxabs_mean=("z_theta_maxabs", "mean"),
        ).reset_index()

        print("\nAGGREGATED DEGREE COMPARISON")
        print(grouped)

        plt.figure(figsize=(10, 4))
        plt.plot(grouped["deg"], grouped["stage2_rmse_mean"], "o-", label="mean Stage 2 RMSE")
        if "stage3b_rmse_mean" in grouped.columns:
            plt.plot(grouped["deg"], grouped["stage3b_rmse_mean"], "o-", label="mean Stage 3b RMSE")
        plt.grid(True)
        plt.xlabel("Polynomial degree")
        plt.ylabel("Mean RMSE [V]")
        plt.title("Degree selection across several cycles")
        plt.legend()
        plt.tight_layout()
        plt.show()

        plt.figure(figsize=(10, 4))
        plt.plot(grouped["deg"], grouped["z_theta_l2_mean"], "o-", label="mean ||thetaZ||2")
        plt.plot(grouped["deg"], grouped["z_theta_maxabs_mean"], "o-", label="mean max|thetaZ|")
        plt.grid(True)
        plt.xlabel("Polynomial degree")
        plt.ylabel("Magnitude")
        plt.title("Parameter growth vs degree")
        plt.legend()
        plt.tight_layout()
        plt.show()

        print("\nDEGREE DECISION GUIDE")
        print("- Keep degree 17 if it stays smooth, stable, and has clearly better mean RMSE without ugly parameter growth.")
        print("- Back off to degree 15 if fit is close but stability is better.")
        print("- Back off to degree 14 if 17/15 look too sensitive across cycles or the nonlinearity becomes rough.")